<a href="https://colab.research.google.com/github/CodeHunterOfficial/ArabovMKDeep/blob/main/NLP-2026/Lecture_4/%D0%9B%D0%B5%D0%BA%D1%86%D0%B8%D1%8F_4_2_LoRA_%D0%B8_QLoRA_%D0%BF%D1%80%D0%B0%D0%BA%D1%82%D0%B8%D1%87%D0%B5%D1%81%D0%BA%D0%B0%D1%8F_%D0%BF%D0%B0%D1%80%D0%B0%D0%BC%D0%B5%D1%82%D1%80%D0%B8%D1%87%D0%B5%D1%81%D0%BA%D0%B8_%D1%8D%D1%84%D1%84%D0%B5%D0%BA%D1%82%D0%B8%D0%B2%D0%BD%D0%B0%D1%8F_%D0%BD%D0%B0%D1%81%D1%82%D1%80%D0%BE%D0%B9%D0%BA%D0%B0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Лекция 4.2. LoRA и QLoRA: практическая параметрически-эффективная настройка

## Тема 1. Математические основы LoRA

**Цель:** глубоко, но доступно разобрать математику LoRA — от расчёта памяти до тонкостей инициализации. Мы используем оригинальную статью (Hu et al., 2021) как основной источник и дополняем её числовыми примерами.

---

### 1.1. Проблема полной настройки: почему это дорого?

Полная тонкая настройка (Full Fine-Tuning) обновляет все параметры модели. Для этого необходимо хранить в видеопамяти GPU следующие компоненты:

#### Расчёт памяти для модели с $P$ параметрами в bf16 (2 байта)

| Компонент | Формула | Для 7B ($P=7 \cdot 10^9$) | Почему |
| :--- | :--- | :--- | :--- |
| **Веса модели** | $P \times 2$ байт | 14 ГБ | Нужны для forward/backward |
| **Градиенты** | $P \times 2$ байт | 14 ГБ | Для каждого веса нужно хранить градиент |
| **Состояния Adam** | $2 \times P \times 4$ байт | 56 ГБ | Adam хранит два fp32 момента: среднее и квадрат градиента |
| **Активации** | Зависят от batch size | ~10–20 ГБ | Промежуточные значения (можно уменьшить через checkpointing) |
| **Итого** | $P \times (2+2+8) +$ активации $\approx 12P$ байт | **~84–100 ГБ** | Для 7B нужно минимум 80 ГБ VRAM |

#### Почему Adam требует 8 байт на параметр?

Adam хранит два момента для каждого параметра:
- $m_t = \beta_1 m_{t-1} + (1-\beta_1)g_t$ — экспоненциально взвешенное среднее градиентов
- $v_t = \beta_2 v_{t-1} + (1-\beta_2)g_t^2$ — экспоненциально взвешенное среднее квадратов градиентов

Оба момента хранятся в fp32 (4 байта) для численной стабильности $\rightarrow 2 \times 4 = 8$ байт на параметр.

#### Что это означает на практике — данные из статьи LoRA:

В оригинальной статье авторы пишут:

> *"On GPT-3 175B, we reduce the VRAM consumption during training from 1.2TB to 350GB. With $r = 4$ and only the query and value projection matrices being adapted, the checkpoint size is reduced by roughly $10,000\times$ (from 350GB to 35MB)."*

| Модель | Полная настройка | LoRA ($r=4$) | Экономия |
| :--- | :--- | :--- | :--- |
| **GPT-3 175B** | 1.2 TB VRAM | 350 GB VRAM | **×3.4** |
| **GPT-3 175B** | 350 GB чекпоинт | 35 MB чекпоинт | **×10 000** |

**Почему это дорого:**

- **Оборудование:** для 7B нужен A100 80GB (и то с аккумуляцией). Для 13B — 2–4 GPU. Для 70B — кластер из 8+ GPU. Для 175B — десятки GPU.
- **Время и стоимость:** обучение 7B на 100K примеров — 10–20 часов на A100. Аренда A100 стоит $2–4/час → **$30–80 за эксперимент**.
- **Эксперименты:** подбор гиперпараметров требует множества прогонов → затраты растут экспоненциально.

**Именно поэтому нужен LoRA:** он снижает требования к памяти и делает дообучение доступным на обычных GPU.

---

### 1.2. Гипотеза низкой внутренней размерности

**Откуда берётся идея LoRA?**

В 2018–2020 годах исследователи обнаружили важный эмпирический факт:

> **Предобученные модели живут на низкой внутренней размерности.** Их можно эффективно адаптировать к новым задачам, изменяя лишь небольшое подмножество «направлений» в пространстве весов.

В статье LoRA авторы ссылаются на работы:
- Li et al. (2018a) — измерение внутренней размерности ландшафта потерь
- Aghajanyan et al. (2020) — объяснение эффективности тонкой настройки через низкую внутреннюю размерность

**Гипотеза LoRA:**

> **Изменения весов $\Delta W = W - W_0$ при адаптации к новой задаче имеют низкий «внутренний ранг» (intrinsic rank).**

Формально: пусть $W_0 \in \mathbb{R}^{d \times k}$ — веса предобученной модели. После адаптации мы получаем $W = W_0 + \Delta W$. Авторы LoRA показали, что матрица $\Delta W$ может быть хорошо аппроксимирована произведением двух матриц малого размера:

$$
\Delta W \approx B \cdot A, \quad B \in \mathbb{R}^{d \times r}, \; A \in \mathbb{R}^{r \times k}, \; r \ll \min(d,k).
$$

**Почему это так?** Предобученная модель уже содержит богатые представления языка, фактов и рассуждений. Адаптация к конкретной задаче требует лишь **небольших корректировок** — она затрагивает ограниченное число «направлений» в пространстве весов. Это похоже на то, как опытный музыкант, разучивая новое произведение, корректирует лишь несколько движений, а не переучивается игре с нуля.

#### Глубинное объяснение: почему $\Delta W$ имеет низкий ранг с точки зрения оптимизации

> Градиенты на предобученных весах имеют структуру, которая не затрагивает все сингулярные направления равномерно. В процессе предобучения модель «научилась» использовать все доступные сингулярные компоненты, но **не все из них одинаково важны для конкретной задачи**. Некоторые сингулярные направления являются общими (универсальными), другие — шумными или редко используемыми. При адаптации градиенты преимущественно «толкают» веса в направлениях, которые уже были частично активированы во время предобучения, но для новой задачи требуют уточнения. LoRA фактически обучается только в наиболее чувствительных направлениях, игнорируя шумные компоненты, что и приводит к низкому эффективному рангу обновления.

#### Эмпирическое подтверждение из статьи (Таблица 6):

| Ранг $r$ | Trainable Params | WikiSQL Accuracy |
| :--- | :--- | :--- |
| **1** | 4.7M | 73.4% |
| **2** | 9.4M | 73.3% |
| **4** | 18.8M | 73.7% |
| **8** | 37.7M | 73.8% |
| **64** | 301.9M | 73.5% |

На GPT-3 175B, даже при $r = 1$, LoRA достигает качества, близкого к полной настройке (73.8% при $r=8$). Это прямое доказательство: **ранг адаптации действительно очень низкий!**

Авторы также исследовали перекрытие подпространств между $A_{r=8}$ и $A_{r=64}$:

> *"Directions corresponding to the top singular vector overlap significantly between $A_{r=8}$ and $A_{r=64}$, while others do not... providing an explanation of why $r = 1$ performs quite well."*

---
## 1.3. Формула LoRA: $W' = W_0 + B \cdot A$

Основная идея LoRA — заморозить предобученные веса $W_0$ и вместо их обновления вводить **обучаемые матрицы малого ранга** $A$ и $B$:

$$
W' = W_0 + \Delta W = W_0 + \frac{\alpha}{r} B \cdot A,
$$

где:
- $W_0 \in \mathbb{R}^{d \times k}$ — замороженные веса (не обучаются).
- $A \in \mathbb{R}^{r \times k}$ — обучаемая матрица.
- $B \in \mathbb{R}^{d \times r}$ — обучаемая матрица.
- $r \ll \min(d,k)$ — ранг адаптации (обычно $r = 16$).
- $\alpha$ — масштабирующий коэффициент (обычно $\alpha = 2 \times r$).

**Как выглядит проход вперёд?**

Для входного вектора $x \in \mathbb{R}^{k}$:

$$
h = W_0 x + \Delta W x = W_0 x + \frac{\alpha}{r} B A x.
$$

То есть к выходу основного слоя добавляется адаптивный член. Обратите внимание на порядок умножения: $B A x$ — сначала матрица $A$ проецирует входной вектор $x$ в низкоразмерное пространство размерности $r$, а затем матрица $B$ проецирует обратно в исходное пространство размерности $d$.

---

### Инициализация: почему $B$ нулевая, а $A$ случайная?

| Матрица | Инициализация | Почему |
| :--- | :--- | :--- |
| **A** | Случайная (Гаусс $\mathcal{N}(0, \sigma^2)$) | Даёт разные направления для градиентов, предотвращает симметрию |
| **B** | Нулевая | В начале $B \cdot A = 0$, значит $W' = W_0$. Модель стартует с предобученных весов |

**Почему это важно?** Если бы $B$ тоже была случайной, то адаптер вносил бы случайные изменения на старте, что могло бы привести к нестабильности и скачкам loss. А так — обучение начинается с уже хорошо работающих весов.

В статье авторы пишут:

> *"We use a random Gaussian initialization for A and zero for B, so $\Delta W = BA$ is zero at the beginning of training."*

---

### Что происходит с градиентами?

При обучении мы обновляем только $A$ и $B$. Градиенты вычисляются через цепное правило. Для матрицы $B$:

$$
\frac{\partial \mathcal{L}}{\partial B} = \frac{\partial \mathcal{L}}{\partial (B A)} A^T.
$$

Для матрицы $A$:

$$
\frac{\partial \mathcal{L}}{\partial A} = B^T \frac{\partial \mathcal{L}}{\partial (B A)}.
$$

Здесь $\frac{\partial \mathcal{L}}{\partial (B A)}$ — это градиент по выходу адаптера, который приходит от вышележащих слоёв модели. Он вычисляется во время обратного распространения и затем "размножается" на $A$ и $B$ через их производные.

Так как $B$ начинается с нуля, $\partial \mathcal{L}/\partial A = 0$ на первых шагах. Это значит, что $A$ не обновляется, пока $B$ не станет ненулевым. На практике градиенты всё равно идут, потому что $B$ обновляется первым (через $\partial \mathcal{L}/\partial B$) и начинает «протаскивать» градиент к $A$.

Этот механизм обеспечивает **естественную регуляризацию**: в начале обучения $A$ не меняется, пока $B$ не накопит достаточно информации, что предотвращает хаотичные обновления на ранних этапах.

---

### Масштабирование $\alpha / r$

На практике выход адаптера умножается на коэффициент $\alpha / r$:

$$
W' = W_0 + \frac{\alpha}{r} B \cdot A,
$$

где $\alpha$ — константа (`lora_alpha`), обычно равная $2 \times r$.

**Зачем это нужно?**

При смене ранга $r$ мы хотим сохранить примерно одинаковый масштаб обновлений. Если $r$ увеличивается вдвое, количество параметров удваивается. Без масштабирования градиенты могли бы изменить свою величину. Коэффициент $\alpha/r$ компенсирует это: чем больше $r$, тем меньше масштаб, и наоборот.

В статье авторы пишут:

> *"We then scale $\Delta W x$ by $\frac{\alpha}{r}$, where $\alpha$ is a constant in $r$. When optimizing with Adam, tuning $\alpha$ is roughly the same as tuning the learning rate if we scale the initialization appropriately."*

**Практическое правило:** $\alpha = 2 \times r$ (например, $r=16 \rightarrow \alpha=32$). Это даёт стабильные градиенты и предсказуемое поведение при изменении ранга.


---

### 1.4. Что такое ранг $r$ и как его выбрать?

**Ранг $r$** — размерность подпространства, в котором мы ищем изменения весов. Чем больше $r$, тем больше «свободы» у адаптера, но и тем больше параметров нужно обучать.

#### Какой ранг выбрать? — данные из статьи

В статье авторы исследовали влияние ранга на качество:

**Таблица 6 (GPT-3 175B, адаптация $W_q, W_v$):**

| Ранг $r$ | Trainable Params | WikiSQL | MNLI |
| :--- | :--- | :--- | :--- |
| **1** | 4.7M | 73.4% | 91.3% |
| **2** | 9.4M | 73.3% | 91.4% |
| **4** | 18.8M | 73.7% | 91.3% |
| **8** | 37.7M | 73.8% | 91.6% |
| **64** | 301.9M | 73.5% | 91.4% |

**Вывод:** качество практически не растёт после $r=4$–$8$, а количество параметров при $r=64$ в 64 раза больше, чем при $r=1$.

| Ранг $r$ | Параметров на слой ($d=4096$) | Рекомендация |
| :--- | :--- | :--- |
| **1** | $2 \cdot 4096 \cdot 1 = 8\,192$ | Минимальный тест (иногда работает!) |
| **4** | $2 \cdot 4096 \cdot 4 = 32\,768$ | Простые задачи |
| **8** | $2 \cdot 4096 \cdot 8 = 65\,536$ | Базовый уровень |
| **16** | $2 \cdot 4096 \cdot 16 = 131\,072$ | **Стандарт по умолчанию** |
| **32** | $2 \cdot 4096 \cdot 32 = 262\,144$ | Сложные задачи |
| **64** | $2 \cdot 4096 \cdot 64 = 524\,288$ | Очень сложные задачи |

**Почему убывающая отдача после 64?**

Из Таблицы 6 видно, что при $r=64$ качество не улучшается, а параметров становится на порядок больше. Исследование подпространств (Рисунок 3 в статье) показывает, что направления, соответствующие топ-1 сингулярному вектору, перекрываются между $r=1$ и $r=64$, а остальные содержат в основном шум.

> *"This suggests that the top singular-vector directions of $A_{r=8}$ and $A_{r=64}$ are the most useful, while other directions potentially contain mostly random noises accumulated during training."*

**Рекомендация:** начинайте с $r=16$. Если качество не устраивает — попробуйте $r=32$ или $r=64$. Больше 64 — редко когда нужно.

---

### 1.5. Расчёт количества параметров LoRA

Для одного слоя размером $d \times d$ (квадратная матрица, как в трансформерах):

$$
\text{Params}_{\text{LoRA}} = 2 \cdot d \cdot r.
$$

**Почему $2 \cdot d \cdot r$?** — потому что две матрицы: $A$ размером $r \times d$ и $B$ размером $d \times r$.

#### Сравнение с полным слоем ($d^2$):

| Размер слоя $d$ | Полный слой $d^2$ | LoRA ($r=16$) | Экономия |
| :--- | :--- | :--- | :--- |
| **1024** | 1 048 576 | 32 768 | **×32** |
| **4096** | 16 777 216 | 131 072 | **×128** |
| **8192** | 67 108 864 | 262 144 | **×256** |

#### Для всей модели GPT-3 175B (из статьи):

В статье авторы пишут:

> *"With $r = 4$ and only the query and value projection matrices being adapted, the checkpoint size is reduced by roughly $10,000\times$ (from 350GB to 35MB)."*

**Расчёт параметров:**

- Полная модель: **175 млрд** параметров.
- LoRA ($r=4$, адаптируем $W_q$ и $W_v$, 96 слоёв):
  - На слой: $2 \cdot d \cdot r = 2 \cdot 12\,288 \cdot 4 = 98\,304$ параметров
  - На 2 проекции: $2 \times 98\,304 = 196\,608$ параметров на слой
  - На 96 слоёв: $96 \times 196\,608 \approx 18.9$ млн параметров
- Сравнение: $18.9$ млн vs $175$ млрд $\rightarrow$ **в 9 260 раз меньше!**

#### Для 7B модели:

Если адаптировать только $W_q$ и $W_v$ (2 проекции на слой, 32 слоя, $d=4096$, $r=16$):

$$
\text{Params}_{\text{LoRA}} = 2 \cdot 2 \cdot 32 \cdot 4096 \cdot 16 = 8\,388\,608 \approx 8.4\text{ млн}.
$$

Это всего **0.12%** от 7B модели!

---

### 1.6. Ключевые свойства LoRA (из статьи)

#### 1. Нулевая задержка инференса

> *"Our simple linear design allows us to merge the trainable matrices with the frozen weights when deployed, introducing no inference latency compared to a fully fine-tuned model."*

После обучения мы вычисляем $W = W_0 + BA$ и используем её как обычную матрицу — никаких дополнительных слоёв, никакой задержки.

#### 2. Быстрое переключение задач

> *"When we need to switch to another downstream task, we can recover $W_0$ by subtracting $BA$ and then adding a different $B'A'$, a quick operation with very little memory overhead."*

```python
# Храним одну базовую модель + множество адаптеров
base_model = load_model("gpt-3-175b")  # 350 GB

# Задача 1: техподдержка (4.7M параметров)
adapter_1 = load_adapter("support_task")

# Задача 2: юридические консультации (4.7M параметров)
adapter_2 = load_adapter("legal_task")

# Переключение — быстрая операция!
adapter_1.unload()
adapter_2.load()
```

#### 3. Экономия памяти до 3 раз

> *"Compared to GPT-3 175B fine-tuned with Adam, LoRA can reduce the number of trainable parameters by 10,000 times and the GPU memory requirement by 3 times."*

#### 4. Качество лучше или на уровне полной настройки

Из Таблицы 4 (GPT-3 175B):

| Метод | Trainable Params | MNLI Accuracy |
| :--- | :--- | :--- |
| Full Fine-Tune | 175B | 89.5% |
| **LoRA** ($r=8$) | **4.7M** | **91.7%** |

LoRA **превосходит** полную настройку, используя в 37 000 раз меньше параметров!

---

### 1.7. Полный числовой пример: от матриц до обучения

#### Исходные данные

Пусть у нас есть слой трансформера с размерностью $d = 4$ (для простоты). Предобученная матрица:

$$
W_0 = \begin{pmatrix}
1 & 2 & 3 & 4 \\
5 & 6 & 7 & 8 \\
9 & 10 & 11 & 12 \\
13 & 14 & 15 & 16
\end{pmatrix}
$$

Мы хотим адаптировать этот слой под новую задачу с помощью LoRA. Выбираем ранг $r = 2$.

#### Шаг 1: Инициализация

Инициализируем матрицы $A$ и $B$:

**A (случайная, из Гаусса):**

$$
A = \begin{pmatrix}
0.2 & -0.1 & 0.3 & 0.0 \\
0.1 & 0.4 & -0.2 & 0.5
\end{pmatrix}
$$

**B (нулевая):**

$$
B = \begin{pmatrix}
0 & 0 \\
0 & 0 \\
0 & 0 \\
0 & 0
\end{pmatrix}
$$

На старте $\Delta W = B \cdot A = 0$, значит $W' = W_0$. Адаптер не вносит изменений.

#### Шаг 2: Обучение (один шаг)

Предположим, после первого батча градиенты обновили $A$ и $B$. Теперь они выглядят так:

**A (после обновления):**

$$
A = \begin{pmatrix}
0.25 & -0.15 & 0.35 & 0.05 \\
0.15 & 0.45 & -0.25 & 0.55
\end{pmatrix}
$$

**B (после обновления):**

$$
B = \begin{pmatrix}
0.2 & 0.1 \\
0.3 & 0.2 \\
0.1 & 0.4 \\
0.5 & 0.3
\end{pmatrix}
$$

#### Шаг 3: Вычисление $\Delta W = B \cdot A$

$$
\Delta W = B \cdot A =
\begin{pmatrix}
0.2 & 0.1 \\
0.3 & 0.2 \\
0.1 & 0.4 \\
0.5 & 0.3
\end{pmatrix}
\cdot
\begin{pmatrix}
0.25 & -0.15 & 0.35 & 0.05 \\
0.15 & 0.45 & -0.25 & 0.55
\end{pmatrix}
$$

Выполним умножение (элемент $(i,j)$ = сумма произведений):

$$
\Delta W =
\begin{pmatrix}
0.2 \cdot 0.25 + 0.1 \cdot 0.15 & 0.2 \cdot (-0.15) + 0.1 \cdot 0.45 & 0.2 \cdot 0.35 + 0.1 \cdot (-0.25) & 0.2 \cdot 0.05 + 0.1 \cdot 0.55 \\
0.3 \cdot 0.25 + 0.2 \cdot 0.15 & 0.3 \cdot (-0.15) + 0.2 \cdot 0.45 & 0.3 \cdot 0.35 + 0.2 \cdot (-0.25) & 0.3 \cdot 0.05 + 0.2 \cdot 0.55 \\
0.1 \cdot 0.25 + 0.4 \cdot 0.15 & 0.1 \cdot (-0.15) + 0.4 \cdot 0.45 & 0.1 \cdot 0.35 + 0.4 \cdot (-0.25) & 0.1 \cdot 0.05 + 0.4 \cdot 0.55 \\
0.5 \cdot 0.25 + 0.3 \cdot 0.15 & 0.5 \cdot (-0.15) + 0.3 \cdot 0.45 & 0.5 \cdot 0.35 + 0.3 \cdot (-0.25) & 0.5 \cdot 0.05 + 0.3 \cdot 0.55
\end{pmatrix}
$$

$$
\Delta W =
\begin{pmatrix}
0.05 + 0.015 & -0.03 + 0.045 & 0.07 - 0.025 & 0.01 + 0.055 \\
0.075 + 0.03 & -0.045 + 0.09 & 0.105 - 0.05 & 0.015 + 0.11 \\
0.025 + 0.06 & -0.015 + 0.18 & 0.035 - 0.10 & 0.005 + 0.22 \\
0.125 + 0.045 & -0.075 + 0.135 & 0.175 - 0.075 & 0.025 + 0.165
\end{pmatrix}
$$

$$
\Delta W =
\begin{pmatrix}
0.065 & 0.015 & 0.045 & 0.065 \\
0.105 & 0.045 & 0.055 & 0.125 \\
0.085 & 0.165 & -0.065 & 0.225 \\
0.170 & 0.060 & 0.100 & 0.190
\end{pmatrix}
$$

#### Шаг 4: Применение масштабирования $\alpha / r$

Пусть $\alpha = 4$ (по правилу $\alpha = 2 \times r = 2 \times 2 = 4$), тогда $\alpha/r = 4/2 = 2$.

$$
\Delta W_{\text{scaled}} = \frac{\alpha}{r} \cdot \Delta W = 2 \cdot \Delta W =
\begin{pmatrix}
0.130 & 0.030 & 0.090 & 0.130 \\
0.210 & 0.090 & 0.110 & 0.250 \\
0.170 & 0.330 & -0.130 & 0.450 \\
0.340 & 0.120 & 0.200 & 0.380
\end{pmatrix}
$$

#### Шаг 5: Обновлённая матрица весов

$$
W' = W_0 + \Delta W_{\text{scaled}} =
\begin{pmatrix}
1.130 & 2.030 & 3.090 & 4.130 \\
5.210 & 6.090 & 7.110 & 8.250 \\
9.170 & 10.330 & 10.870 & 12.450 \\
13.340 & 14.120 & 15.200 & 16.380
\end{pmatrix}
$$

#### Шаг 6: Сравнение количества параметров

| Параметр | Количество |
| :--- | :--- |
| **Полный слой** ($d^2 = 4 \times 4$) | **16** параметров |
| **LoRA** ($2 \cdot d \cdot r = 2 \cdot 4 \cdot 2$) | **16** параметров ($A$: 8, $B$: 8) |
| **Обучается** | только 16 параметров LoRA |
| **Заморожено** | 16 параметров $W_0$ |

Для реальных моделей ($d=4096, r=16$):

| Параметр | Количество |
| :--- | :--- |
| **Полный слой** ($d^2$) | **16.7M** параметров |
| **LoRA** ($2 \cdot d \cdot r$) | **131K** параметров |
| **Экономия** | **×128** |

---

### Итоговые выводы по теме 1

1. **Полная настройка** требует **~84 ГБ VRAM** для 7B и стоит $30–80/эксперимент.
2. **LoRA** основан на гипотезе о низкой размерности изменений весов ($\Delta W$ имеет низкий ранг).
3. Формула: $W' = W_0 + \frac{\alpha}{r} B A$, где $B=0$ в начале, $A$ случайна.
4. Масштабирование $\alpha/r$ — для стабильности градиентов при смене ранга.
5. **Ранг $r$** — начинайте с 16, увеличивайте до 32–64 при необходимости.
6. LoRA добавляет всего **0.12%** параметров, но даёт качество близкое к Full FT.
7. **Свойства:** нулевая задержка, быстрое переключение задач, экономия до 10 000 раз.

**Помните слова из статьи:**
> *"We show that a very low rank (i.e., $r$ can be one or two) suffices even when the full rank ($d$) is as high as 12,288."*

Именно это делает LoRA революционным методом. Теперь, когда вы понимаете математику, переходим к практике — как настроить LoRA в коде.

# Тема 2. Конфигурация LoRA – ключевые гиперпараметры

**Цель:** дать практическую шпаргалку по выбору гиперпараметров LoRA на основе оригинальной статьи (Hu et al., 2021) и опыта сообщества. Мы разберём каждый параметр, приведём числовые ориентиры и объясним, почему они работают. Все примеры привязаны к реальному коду для Qwen2.5-1.5B-Instruct.

---

## Введение: почему LoRA — не «чёрный ящик»

LoRA даёт нам точный контроль над адаптацией модели через небольшой набор гиперпараметров. Правильный выбор этих параметров — ключ к успеху. В этой шпаргалке мы разберём каждый гиперпараметр: **почему** мы выбираем именно так, **как** это влияет на обучение и **какие значения** работают на практике. Все рекомендации подкреплены данными из оригинальной статьи (Hu et al., 2021) и опытом сообщества.

**Важно:** в коде мы используем названия из библиотеки `transformers` (например, `q_proj`, `v_proj`), а в оригинальной статье они обозначаются как `W_q`, `W_v`. Это одно и то же — просто разные нотации.

---

## 2.1. Целевые слои (`target_modules`): что адаптировать?

В трансформере есть несколько типов матриц весов, к которым можно применить LoRA. Выбор слоёв определяет, **какая часть модели будет адаптироваться** под новую задачу.

### Роль проекций в трансформере

| Проекция | Код (transformers) | Статья (Hu et al.) | Роль | Приоритет |
| :--- | :--- | :--- | :--- | :--- |
| **Query** | `q_proj` | `W_q` | Преобразует запросы в self-attention | ⭐⭐⭐⭐⭐ |
| **Value** | `v_proj` | `W_v` | Преобразует значения в self-attention | ⭐⭐⭐⭐⭐ |
| **Key** | `k_proj` | `W_k` | Преобразует ключи в self-attention | ⭐⭐⭐ |
| **Output** | `o_proj` | `W_o` | Смешивает головы внимания | ⭐⭐⭐⭐ |
| **MLP Gate** | `gate_proj` | `W_gate` | Вентиль в FFN (LLaMA-стиль) | ⭐⭐ |
| **MLP Up** | `up_proj` | `W_up` | Расширение в FFN | ⭐⭐ |
| **MLP Down** | `down_proj` | `W_down` | Сжатие в FFN | ⭐⭐ |

### Почему начинают с `q_proj` и `v_proj`?

Это не просто эмпирическое правило — оно основано на эксперименте из оригинальной статьи LoRA (Таблица 5 на GPT-3 175B):

| Адаптируемые слои | Ранг | WikiSQL | MultiNLI |
| :--- | :--- | :--- | :--- |
| Только `W_q` | 8 | 70.4% | 91.0% |
| Только `W_k` | 8 | 70.0% | 90.8% |
| Только `W_v` | 8 | 73.0% | 91.0% |
| Только `W_o` | 8 | 73.2% | 91.3% |
| `W_q` + `W_v` | 4 | **73.7%** | **91.3%** |
| `W_q` + `W_k` + `W_v` + `W_o` | 2 | 73.7% | 91.7% |

**Выводы из таблицы:**

1. **`q_proj` и `v_proj` дают наибольший прирост.** Адаптация обоих слоёв даёт лучшее качество, чем адаптация одного слоя с более высоким рангом.
2. **Адаптация `q_proj` + `v_proj` (r=4) лучше, чем только `q_proj` (r=8).** Это означает, что важно адаптировать *больше типов слоёв*, даже с меньшим рангом.
3. **Добавление `k_proj`, `o_proj` даёт небольшой дополнительный прирост** (+0.4% на MultiNLI).
4. **MLP-слои** дают ещё меньший прирост, но добавляют много параметров.

### Когда подключать `k_proj`, `o_proj` и MLP?

- **`k_proj`** – добавьте, если задача требует точного позиционирования (например, извлечение фактов, работа с длинными контекстами).
- **`o_proj`** – полезен для задач генерации, где важна плавность текста и согласованность между головами внимания.
- **MLP слои** – добавляйте только при очень сложных задачах (многозадачность, изменение стиля, кода) и при наличии **большого датасета** (>10K примеров). Это увеличивает число обучаемых параметров в **3–5 раз**, но даёт прирост качества на 1–2%.

### Шпаргалка по выбору целевых слоёв

| Уровень | `target_modules` | Когда применять |
| :--- | :--- | :--- |
| **Минимальный** | `["q_proj", "v_proj"]` | Начальные эксперименты, ограниченный бюджет. **Рекомендуется для 7B+ моделей.** |
| **Стандартный** | `["q_proj", "k_proj", "v_proj", "o_proj"]` | **Для большинства задач.** Хороший баланс качества и параметров. |
| **Расширенный** | `["q_proj","k_proj","v_proj","o_proj", "gate_proj","up_proj","down_proj"]` | Сложные задачи (изменение стиля, логики, программирование), но **только с большим датасетом**. |

### Пример из кода (Qwen2.5-1.5B)

```python
# Для модели 1.5B мы используем СТАНДАРТНЫЙ уровень — все проекции внимания
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],  # все 4 проекции
)

# Проверка количества параметров
model.print_trainable_parameters()
# Ожидаемый вывод: trainable params: 8,388,608 || all params: 1.5B || trainable%: 0.56%
```

---

## 2.2. Гиперпараметры LoRA: `r`, `alpha`, `dropout`, `bias`

### 2.2.1. Ранг `r` – ёмкость адаптера

**Роль:** определяет размерность подпространства, в котором ищутся изменения весов. Чем больше `r`, тем больше обучаемых параметров и тем выше гибкость адаптера.

**Рекомендуемые значения (стартовая точка – `r=16`):**

| `r` | Параметров на слой (d=4096) | Когда использовать |
| :--- | :--- | :--- |
| 4 | ~65K | Очень простые задачи, быстрые эксперименты |
| 8 | ~131K | Простые задачи (классификация, форматирование) |
| **16** | ~262K | **Стандартное значение по умолчанию** |
| 32 | ~524K | Сложные задачи (перевод, суммаризация, диалоги) |
| 64 | ~1M | Очень сложные задачи, приближение к Full FT (требует много данных) |

**Как подбирать:**  
- Начинайте с `r=16`.  
- Если модель **переобучается** (eval loss растёт) – уменьшите до 8 или 4.  
- Если модель **недообучается** (низкое качество, train loss не падает) – увеличьте до 32 или 64.

### 2.2.2. `lora_alpha` – масштабирующий коэффициент

**Формула:** `ΔW = (alpha / r) · B · A`

**Роль:** масштабирует выход адаптера. Стабилизирует градиенты при изменении `r`.

**Практическое правило:** `alpha = 2 × r`

| `r` | `alpha` | `alpha / r` |
| :--- | :--- | :--- |
| 4 | 8 | 2.0 |
| 8 | 16 | 2.0 |
| **16** | **32** | **2.0** |
| 32 | 64 | 2.0 |
| 64 | 128 | 2.0 |

**Почему именно 2×?** Это эмпирически найденное значение, которое даёт стабильную норму обновлений для разных рангов. В оригинальной статье авторы пишут:

> *"We then scale ΔWx by α/r, where α is a constant in r. When optimizing with Adam, tuning α is roughly the same as tuning the learning rate if we scale the initialization appropriately."*

**Тонкая настройка:**  
- Качество слишком низкое → увеличьте `alpha` до `3×r` или `4×r`.  
- Модель галлюцинирует → уменьшите `alpha` до `1×r`.

### 2.2.3. `lora_dropout` – регуляризация

**Роль:** вероятность обнуления элементов в матрицах `A` и `B` во время обучения. Помогает бороться с переобучением.

| Значение | Когда использовать |
| :--- | :--- |
| **0.0** | Большой датасет (>10K примеров), нет признаков переобучения |
| **0.05** | **Стандартное значение по умолчанию** |
| **0.1** | Маленький датасет (<1K примеров) или сильное переобучение |
| 0.2+ | Редко, может ухудшить качество |

**Совет:** начинайте с `0.05`. Если заметили расхождение train/eval loss – увеличьте до `0.1`.

### 2.2.4. `bias` – обучать ли смещения?

| Значение | Описание | Рекомендация |
| :--- | :--- | :--- |
| `"none"` | Не обучать biases | **Всегда!** |
| `"all"` | Обучать все biases | Не рекомендуется (прирост <0.5%, параметров +5%) |
| `"lora_only"` | Обучать biases только в LoRA-слоях | Бесполезно, лучше `"none"` |

**Рекомендация:** всегда используйте `bias="none"`. Bias-параметры составляют ничтожную долю, и их обучение не даёт значимого прироста.

**Итоговая таблица гиперпараметров по умолчанию:**

| Параметр | Значение | Пояснение |
| :--- | :--- | :--- |
| `r` | 16 | Баланс качество/параметры |
| `lora_alpha` | 32 | `= 2 * r` |
| `lora_dropout` | 0.05 | Лёгкая регуляризация |
| `bias` | `"none"` | Не тратим ресурсы на biases |

---

## 2.3. Learning Rate (LR) – почему для LoRA он выше?

| Режим | Типичный LR | Почему |
| :--- | :--- | :--- |
| **Full Fine-Tune** | 1e-5 – 5e-5 | Обновляются все веса – нужна осторожность, чтобы не забыть предобученные знания |
| **LoRA (полная точность)** | **5e-5 – 2e-4** | Адаптер инициализируется нулями; нужно быстрее "разбудить" его. Меньше параметров – можно больше LR |
| **QLoRA (4-bit)** | **2e-4** | Квантование вносит шум – более высокий LR помогает преодолеть его и эффективно обучать |

### Почему LoRA требует более высокую LR?

1. **Адаптер стартует с нуля.** В начале `B=0`, поэтому `ΔW=0`. Нужна более высокая LR, чтобы матрицы начали меняться.
2. **Меньше параметров.** Риск переобучения ниже, поэтому можно смелее увеличивать шаг.
3. **Масштабирование `alpha/r`.** Коэффициент уменьшает величину обновлений, поэтому LR можно делать выше.

### Почему QLoRA требует ещё выше (2e-4)?

При 4-битном квантовании веса хранятся с пониженной точностью, что создаёт дополнительный шум в градиентах. Более высокая LR помогает "перебить" этот шум и эффективно обновлять адаптер. На практике многие начинают с `2e-4` и при необходимости корректируют.

### Практический алгоритм подбора LR

1. Начните с `2e-4` для LoRA (или `5e-5` для Full FT).
2. Если loss не снижается → увеличьте LR в 2–3 раза.
3. Если loss взлетает до NaN → уменьшите LR в 10 раз.
4. Для QLoRA стартуйте с `2e-4`; при нестабильности попробуйте `1e-4`.

---

## 2.4. Количество эпох и мониторинг eval loss

### Почему обычно достаточно 1–3 эпох?

1. **LoRA обучает мало параметров (<1% от модели).** Они быстро адаптируются, и дальше начинается переобучение.
2. **Исследования показывают:** на большинстве задач качество перестаёт расти после 2–3 эпох.
3. **Риск переобучения** резко возрастает после 3 эпох, особенно на маленьких датасетах.

### Исключение: Early Stopping

Если вы используете **Early Stopping**, можно смело ставить больше эпох (например, 10–20) – обучение остановится само, как только eval loss перестанет улучшаться. В вашем коде именно так и сделано: `NUM_EPOCHS=20` + `EarlyStoppingCallback(patience=3)`.

### Как следить за eval loss?

**Eval loss – главный индикатор переобучения.**

| Ситуация | Интерпретация | Действие |
| :--- | :--- | :--- |
| **eval_loss падает** | ✅ Модель учится обобщать | Продолжайте обучение |
| **eval_loss растёт, train_loss падает** | ⚠️ Переобучение | Немедленно остановить (Early Stopping) |
| **eval_loss колеблется без тренда** | ⚠️ Валидационный набор слишком мал или зашумлён | Увеличьте валидацию или используйте кросс-валидацию |

### Пример интерпретации графика (из вашего кода)

На графике видно, что:
- **Train loss** стабильно падает → модель учится на тренировочных данных.
- **Eval loss** сначала падает, затем начинает расти → **переобучение**. Early Stopping остановило обучение на 8-й эпохе.
- **BLEU на валидации** колеблется, но в целом следует за eval loss.

**Практические рекомендации:**

1. **Всегда включайте `evaluation_strategy="steps"`** и `eval_steps` (например, 100) для частого мониторинга.
2. **Используйте `load_best_model_at_end=True`** – в конце останется лучший чекпоинт.
3. **Визуализируйте кривые** (как в вашем коде) – это помогает быстро заметить проблемы.

### Сколько эпох ставить в зависимости от датасета?

| Размер датасета | Без Early Stopping | С Early Stopping |
| :--- | :--- | :--- |
| < 100 примеров | 3–5 | 10–20 (остановится сам) |
| 100 – 1 000 | 3–10 | 10–20 |
| > 1 000 | 1–3 | 5–10 |
| > 10 000 | 1 | 3–5 |

---

## 2.5. Дополнительные настройки (warmup, gradient checkpointing)

Эти параметры не входят в `LoraConfig`, но сильно влияют на стабильность и потребление памяти.

### Warmup steps

**Роль:** постепенный рост LR от 0 до заданного значения в течение первых N шагов. Сглаживает начало обучения.

**Рекомендация:** `warmup_steps = 0.05 × total_steps` (5%) – стандарт. Для маленьких датасетов можно увеличить до 10%.

В вашем коде это реализовано идеально:
```python
total_steps = (len(tokenized_train) // (TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS)) * NUM_EPOCHS
warmup_steps = int(0.05 * total_steps)
```

### Gradient Checkpointing

**Роль:** экономит видеопамять за счёт пересчёта активаций на обратном проходе (вместо их хранения). Замедляет обучение на ~20%, но позволяет использовать вдвое больший batch size.

**Рекомендация:** включайте всегда, если у вас GPU с памятью <24 ГБ. В вашем коде: `gradient_checkpointing=True`.

### Scheduler (lr_scheduler_type)

**Роль:** стратегия изменения LR во время обучения.

| Scheduler | Когда использовать |
| :--- | :--- |
| **`"cosine"`** | **Стандарт для LoRA** – плавное снижение LR |
| `"linear"` | Простые задачи, маленькие датасеты |
| `"constant"` | Редко, только если уверены в LR |

**Рекомендация:** используйте `"cosine"` – он даёт лучшую сходимость.

---

## 2.6. Общие ошибки и как их избежать

| Ошибка | Проблема | Решение |
| :--- | :--- | :--- |
| **Слишком много эпох без Early Stopping** | Переобучение, eval loss растёт | Добавьте Early Stopping (patience=2–3) |
| **Слишком высокая LR** | Loss взлетает до NaN | Уменьшите LR в 10 раз |
| **Слишком низкая LR** | Loss не снижается | Увеличьте LR в 2–3 раза |
| **Неправильные `target_modules`** | Модель не адаптируется | Проверьте названия слоёв для вашей модели |
| **`bias="all"`** | Параметров больше, качества нет | Всегда используйте `bias="none"` |
| **Нет валидации** | Не видите переобучение | Всегда включайте `eval_strategy` и `eval_steps` |
| **Слишком большой batch size** | CUDA out of memory | Уменьшите batch size или включите gradient checkpointing |

---

## 2.7. Полная шпаргалка по гиперпараметрам LoRA

| Параметр | Рекомендация | Типичный диапазон | Примечание |
| :--- | :--- | :--- | :--- |
| **target_modules** | `["q_proj","v_proj"]` для 7B+, все 4 для <3B | – | Начинайте с q+v, добавляйте при необходимости |
| **r** | 16 | 4 – 64 | Увеличивайте для сложных задач, уменьшайте при переобучении |
| **lora_alpha** | `2 × r` | `(1–4)×r` | Меняйте, если качество/стабильность не устраивают |
| **lora_dropout** | 0.05 | 0.0 – 0.1 | Увеличивайте при малом датасете |
| **bias** | `"none"` | – | Всегда `"none"` |
| **learning_rate (LoRA)** | 2e-4 | 1e-4 – 5e-4 | Начинайте с 2e-4 |
| **learning_rate (QLoRA)** | 2e-4 | 1e-4 – 5e-4 | Тот же старт, но при нестабильности уменьшайте |
| **num_epochs** | 2–3 (без ES), 10–20 (с ES) | – | С Early Stopping можно ставить больше |
| **Early Stopping** | `patience=3`, `threshold=0.001` | `patience 2–5` | Обязательно при большом числе эпох |
| **warmup_steps** | 5% от total_steps | 5–10% | Стабилизирует начало обучения |
| **gradient_checkpointing** | `True` для малых GPU | – | Экономит память ценой скорости |
| **lr_scheduler_type** | `"cosine"` | `"cosine"`, `"linear"` | Cosine даёт лучшую сходимость |

---

## 2.8. Пример готовой конфигурации

Ваш код – идеальный шаблон, который реализует все перечисленные рекомендации. Вот выдержка с ключевыми настройками:

```python
# ================================================================
# ЧИСТЫЙ LoRA Fine-Tuning
# Qwen/Qwen2.5-1.5B-Instruct
# ================================================================
# 1. УСТАНОВКА
# ================================================================

!pip install -q -U \
    transformers \
    peft \
    accelerate \
    datasets \
    evaluate \
    sacrebleu \
    matplotlib \
    "numpy<2.1"

# ================================================================
# 2. ИМПОРТЫ
# ================================================================

import os
import torch
import numpy as np
import matplotlib.pyplot as plt
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    Trainer,
    TrainingArguments,
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback,
)
from peft import (
    LoraConfig,
    get_peft_model,
    PeftModel,
)
import evaluate

# ================================================================
# 3. ПРОВЕРКА ОКРУЖЕНИЯ
# ================================================================

print("=" * 70)
print("ПРОВЕРКА ОКРУЖЕНИЯ")
print("=" * 70)

print("PyTorch:", torch.__version__)
print("CUDA доступна:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "VRAM:",
        round(
            torch.cuda.get_device_properties(0).total_memory / 1024**3,
            2
        ),
        "GB"
    )

    if torch.cuda.is_bf16_supported():
        DTYPE = torch.bfloat16
        print("Dtype: torch.bfloat16")
    else:
        DTYPE = torch.float16
        print("Dtype: torch.float16")
else:
    DTYPE = torch.float32
    print("Dtype: torch.float32")

print("=" * 70)

# ================================================================
# 4. КОНФИГУРАЦИЯ
# ================================================================

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

OUTPUT_DIR = "./qwen25_1.5b_lora_translation"

MAX_LENGTH = 256

# LoRA
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05

# Training
NUM_EPOCHS = 20
LEARNING_RATE = 2e-4

TRAIN_BATCH_SIZE = 2
EVAL_BATCH_SIZE = 2

GRADIENT_ACCUMULATION_STEPS = 4

SEED = 42

# Early stopping
EARLY_STOPPING_PATIENCE = 3
EARLY_STOPPING_THRESHOLD = 0.001

# ================================================================
# 5. ДАТАСЕТ
# ================================================================

train_data = [
    {"instruction": "Переведи на английский: Привет, как дела?", "output": "Hello, how are you?"},
    {"instruction": "Переведи на английский: Сегодня отличная погода.", "output": "The weather is great today."},
    {"instruction": "Переведи на английский: Я люблю программирование.", "output": "I love programming."},
    {"instruction": "Переведи на английский: Который час?", "output": "What time is it?"},
    {"instruction": "Переведи на английский: Это очень интересно.", "output": "This is very interesting."},
    {"instruction": "Переведи на английский: Доброе утро!", "output": "Good morning!"},
    {"instruction": "Переведи на английский: Спокойной ночи.", "output": "Good night."},
    {"instruction": "Переведи на английский: Как тебя зовут?", "output": "What is your name?"},
]

eval_data = [
    {"instruction": "Переведи на английский: Я хочу пить.", "output": "I am thirsty."},
    {"instruction": "Переведи на английский: Где находится библиотека?", "output": "Where is the library?"},
]

train_dataset = Dataset.from_list(train_data)
eval_dataset = Dataset.from_list(eval_data)

print()
print("=" * 70)
print("DATASET")
print("=" * 70)

print("Train:", len(train_dataset))
print("Eval :", len(eval_dataset))

print()
print("Пример:")
print(train_dataset[0])

print("=" * 70)

# ================================================================
# 6. TOKENIZER
# ================================================================

print()
print("=" * 70)
print("ЗАГРУЗКА TOKENIZER")
print("=" * 70)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Vocab size:", tokenizer.vocab_size)
print("PAD token:", tokenizer.pad_token)
print("EOS token:", tokenizer.eos_token)

print("=" * 70)

# ================================================================
# 7. ФОРМИРОВАНИЕ PROMPT
# ================================================================

def build_messages(example):
    return [
        {"role": "user", "content": example["instruction"]},
        {"role": "assistant", "content": example["output"]},
    ]

def tokenize_example(example):
    messages = build_messages(example)
    full_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )
    prompt_messages = [{"role": "user", "content": example["instruction"]}]
    prompt_text = tokenizer.apply_chat_template(
        prompt_messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    full_tokens = tokenizer(
        full_text,
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,
    )
    prompt_tokens = tokenizer(
        prompt_text,
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,
    )

    input_ids = full_tokens["input_ids"]
    attention_mask = full_tokens["attention_mask"]
    prompt_length = len(prompt_tokens["input_ids"])

    labels = input_ids.copy()
    for i in range(min(prompt_length, len(labels))):
        labels[i] = -100

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
    }

# ================================================================
# 8. TOKENIZATION
# ================================================================

print()
print("=" * 70)
print("TOKENIZATION")
print("=" * 70)

tokenized_train = train_dataset.map(
    tokenize_example,
    remove_columns=train_dataset.column_names,
)

tokenized_eval = eval_dataset.map(
    tokenize_example,
    remove_columns=eval_dataset.column_names,
)

print("Train examples:", len(tokenized_train))
print("Eval examples :", len(tokenized_eval))

print("=" * 70)

# ================================================================
# 9. ЗАГРУЗКА БАЗОВОЙ МОДЕЛИ
# ================================================================

print()
print("=" * 70)
print("ЗАГРУЗКА BASE MODEL")
print("=" * 70)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=DTYPE,
    device_map="auto",
    trust_remote_code=True,
)

model.config.use_cache = False

print("Model loaded.")
print("=" * 70)

# ================================================================
# 10. LoRA CONFIG
# ================================================================

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)

# ================================================================
# 11. ДОБАВЛЯЕМ LoRA
# ================================================================

model = get_peft_model(model, lora_config)

# ================================================================
# 12. ПРОВЕРКА ПАРАМЕТРОВ
# ================================================================

print()
print("=" * 70)
print("LoRA PARAMETER CHECK")
print("=" * 70)

model.print_trainable_parameters()

print("=" * 70)

# ================================================================
# 13. ПРОВЕРКА: ТОЛЬКО LoRA ОБУЧАЕТСЯ
# ================================================================

trainable_names = []
for name, param in model.named_parameters():
    if param.requires_grad:
        trainable_names.append(name)

print()
print("Количество trainable tensors:", len(trainable_names))
print()
print("Первые trainable parameters:")
for name in trainable_names[:20]:
    print("  ", name)

# ================================================================
# 14. DATA COLLATOR
# ================================================================

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
    label_pad_token_id=-100,
    pad_to_multiple_of=8 if torch.cuda.is_available() else None,
)

# ================================================================
# 15. TRAINING ARGUMENTS (с warmup_steps)
# ================================================================

total_steps = (len(tokenized_train) // (TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS)) * NUM_EPOCHS
warmup_steps = int(0.05 * total_steps)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    optim="adamw_torch",
    warmup_steps=warmup_steps,
    lr_scheduler_type="cosine",
    weight_decay=0.01,
    logging_steps=1,
    logging_first_step=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    fp16=(torch.cuda.is_available() and DTYPE == torch.float16),
    bf16=(torch.cuda.is_available() and DTYPE == torch.bfloat16),
    gradient_checkpointing=True,
    seed=SEED,
    report_to="none",
    remove_unused_columns=False,
)

# ================================================================
# 16. МЕТРИКА BLEU (исправленная)
# ================================================================

bleu_metric = evaluate.load("sacrebleu")

def compute_metrics(eval_pred):
    """
    Вычисляет BLEU на валидационном наборе.
    Принимает логиты и метки, делает argmax и декодирует.
    """
    logits, labels = eval_pred
    # logits: (batch_size, seq_len, vocab_size)
    predictions = np.argmax(logits, axis=-1)  # (batch_size, seq_len)

    # Заменяем -100 на pad_token_id для декодирования
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    # Декодируем
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # Проверяем, что количество совпадает (на случай, если батчи не объединились)
    if len(decoded_preds) != len(decoded_labels):
        # Если не совпадает, возможно, один из них пустой – выравниваем
        min_len = min(len(decoded_preds), len(decoded_labels))
        decoded_preds = decoded_preds[:min_len]
        decoded_labels = decoded_labels[:min_len]

    # Для sacrebleu каждый референс должен быть списком
    decoded_labels = [[ref] for ref in decoded_labels]

    result = bleu_metric.compute(predictions=decoded_preds, references=decoded_labels)
    return {"bleu": result["score"]}

# ================================================================
# 17. TRAINER (без preprocess_logits_for_metrics для надёжности)
# ================================================================

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=EARLY_STOPPING_PATIENCE,
            early_stopping_threshold=EARLY_STOPPING_THRESHOLD,
        )
    ],
)

# ================================================================
# 18. ТРЕНИРОВКА
# ================================================================

print()
print("=" * 70)
print("НАЧАЛО LoRA TRAINING")
print("=" * 70)

train_result = trainer.train()

print()
print("=" * 70)
print("TRAINING FINISHED")
print("=" * 70)

print(train_result)

# ================================================================
# 19. СОХРАНЕНИЕ LoRA
# ================================================================

print()
print("=" * 70)
print("СОХРАНЕНИЕ LoRA")
print("=" * 70)

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print("LoRA adapter сохранён:")
print(OUTPUT_DIR)

print("=" * 70)

# ================================================================
# 20. ПРОВЕРКА ФАЙЛОВ
# ================================================================

print()
print("Содержимое директории:")

for filename in sorted(os.listdir(OUTPUT_DIR)):
    path = os.path.join(OUTPUT_DIR, filename)
    if os.path.isfile(path):
        size_mb = os.path.getsize(path) / 1024**2
        print(f"{filename:40s}{size_mb:10.2f} MB")

# ================================================================
# 21. ГРАФИК ОБУЧЕНИЯ
# ================================================================

print()
print("=" * 70)
print("ГРАФИК ОБУЧЕНИЯ")
print("=" * 70)

log_history = trainer.state.log_history

train_losses = []
eval_losses = []
bleu_scores = []
epochs = []

for log in log_history:
    if "loss" in log and "epoch" in log:
        train_losses.append(log["loss"])
        epochs.append(log["epoch"])
    if "eval_loss" in log:
        eval_losses.append((log["epoch"], log["eval_loss"]))
    if "bleu" in log:
        bleu_scores.append((log["epoch"], log["bleu"]))

fig, ax1 = plt.subplots(figsize=(10, 6))

ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss", color="tab:red")
ax1.plot(epochs, train_losses, label="Train Loss", color="tab:red", marker="o")
if eval_losses:
    eval_epochs, eval_vals = zip(*eval_losses)
    ax1.plot(eval_epochs, eval_vals, label="Eval Loss", color="tab:orange", marker="s")
ax1.tick_params(axis="y", labelcolor="tab:red")
ax1.legend(loc="upper left")

if bleu_scores:
    ax2 = ax1.twinx()
    ax2.set_ylabel("BLEU", color="tab:blue")
    bleu_epochs, bleu_vals = zip(*bleu_scores)
    ax2.plot(bleu_epochs, bleu_vals, label="BLEU", color="tab:blue", marker="^")
    ax2.tick_params(axis="y", labelcolor="tab:blue")
    ax2.legend(loc="upper right")

plt.title("Training and Validation Metrics")
plt.grid(True)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "training_plot.png"), dpi=150)
plt.show()

print(f"График сохранён в {os.path.join(OUTPUT_DIR, 'training_plot.png')}")
print("=" * 70)

# ================================================================
# 22. ФИНАЛЬНАЯ ОЦЕНКА BLEU НА ВАЛИДАЦИИ
# ================================================================

print()
print("=" * 70)
print("ФИНАЛЬНАЯ ОЦЕНКА BLEU НА ВАЛИДАЦИИ")
print("=" * 70)

# Определим функцию инференса (она понадобится ниже)
def generate_translation(
    instruction,
    max_new_tokens=64,
    temperature=0.2,
):
    model.eval()
    messages = [{"role": "user", "content": instruction}]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(text, return_tensors="pt")
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=(temperature > 0),
            temperature=temperature,
            top_p=0.9,
            repetition_penalty=1.05,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    generated_tokens = outputs[0, inputs["input_ids"].shape[1]:]
    answer = tokenizer.decode(generated_tokens, skip_special_tokens=True)
    return answer.strip()

references = [ex["output"] for ex in eval_data]
predictions = []
for ex in eval_data:
    pred = generate_translation(ex["instruction"])
    predictions.append(pred)

bleu_score = bleu_metric.compute(predictions=predictions, references=[[ref] for ref in references])
print(f"BLEU на валидационном наборе: {bleu_score['score']:.2f}")

for i, (pred, ref) in enumerate(zip(predictions, references)):
    print(f"\nПример {i+1}:")
    print(f"  Инструкция: {eval_data[i]['instruction']}")
    print(f"  Ожидалось : {ref}")
    print(f"  Получено  : {pred}")

print("=" * 70)

# ================================================================
# 23. ТЕСТ НА TRAIN EXAMPLE
# ================================================================

print()
print("=" * 70)
print("TEST: TRAIN EXAMPLE")
print("=" * 70)

test_instruction = "Переведи на английский: Привет, как дела?"
answer = generate_translation(test_instruction)

print("INPUT :")
print(test_instruction)
print()
print("OUTPUT:")
print(answer)
print("=" * 70)

# ================================================================
# 24. ТЕСТ НА НОВЫХ ПРИМЕРАХ
# ================================================================

test_examples = [
    "Переведи на английский: Я хочу пить.",
    "Переведи на английский: Где находится библиотека?",
    "Переведи на английский: Я люблю Python.",
    "Переведи на английский: До свидания!",
    "Переведи на английский: Как тебя зовут?",
]

print()
print("=" * 70)
print("TEST: NEW EXAMPLES")
print("=" * 70)

for instruction in test_examples:
    answer = generate_translation(instruction)
    print()
    print("INPUT :", instruction)
    print("OUTPUT:", answer)

print("=" * 70)
```

---

## Расчёт параметров для Qwen2.5-1.5B

| Конфигурация | Обучаемые параметры | Доля от модели |
| :--- | :--- | :--- |
| **Только q+v** (`r=16`) | ~4.2M | ~0.28% |
| **Все 4 проекции** (`r=16`) | ~8.4M | ~0.56% |
| **Все 4 проекции + MLP** (`r=16`) | ~25M | ~1.7% |


### Пример для GPT-2 (архитектурные отличия)

В отличие от современных моделей (Llama, Qwen, Mistral), GPT-2 имеет **другую структуру слоёв**:

- В self-attention используется **одна матрица `c_attn`** (объединяет Q, K, V) и **`c_proj`** (выходная проекция).
- Вместо отдельных `q_proj`, `k_proj`, `v_proj`, `o_proj` мы адаптируем **`c_attn` и `c_proj`**.
- GPT-2 **не поддерживает `apply_chat_template`**, поэтому промпт формируется вручную.
- У GPT-2 по умолчанию **`pad_token = None`**, поэтому его нужно явно установить в `eos_token`.

Код ниже адаптирован под эти особенности. Все остальные гиперпараметры (`r`, `alpha`, `dropout`, `LR`, Early Stopping) остаются теми же — меняются только **целевые слои** и **способ токенизации**.


```python
# ================================================================
# ЧИСТЫЙ LoRA Fine-Tuning
# GPT-2 (базовая модель)
# ================================================================
# 1. УСТАНОВКА
# ================================================================

!pip install -q -U \
    transformers \
    peft \
    accelerate \
    datasets \
    evaluate \
    sacrebleu \
    matplotlib \
    "numpy<2.1"

# ================================================================
# 2. ИМПОРТЫ
# ================================================================

import os
import torch
import numpy as np
import matplotlib.pyplot as plt
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    Trainer,
    TrainingArguments,
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback,
)
from peft import (
    LoraConfig,
    get_peft_model,
    PeftModel,
)
import evaluate

# ================================================================
# 3. ПРОВЕРКА ОКРУЖЕНИЯ
# ================================================================

print("=" * 70)
print("ПРОВЕРКА ОКРУЖЕНИЯ")
print("=" * 70)

print("PyTorch:", torch.__version__)
print("CUDA доступна:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "VRAM:",
        round(
            torch.cuda.get_device_properties(0).total_memory / 1024**3,
            2
        ),
        "GB"
    )

    if torch.cuda.is_bf16_supported():
        DTYPE = torch.bfloat16
        print("Dtype: torch.bfloat16")
    else:
        DTYPE = torch.float16
        print("Dtype: torch.float16")
else:
    DTYPE = torch.float32
    print("Dtype: torch.float32")

print("=" * 70)

# ================================================================
# 4. КОНФИГУРАЦИЯ (АДАПТИРОВАНА ДЛЯ GPT-2)
# ================================================================

# <-- ИЗМЕНЕНО: модель GPT-2 (можно взять gpt2-medium, gpt2-large)
MODEL_NAME = "gpt2"  # или "gpt2-medium", "gpt2-large"

OUTPUT_DIR = "./gpt2_lora_translation"

MAX_LENGTH = 256

# LoRA
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05

# Training
NUM_EPOCHS = 20
LEARNING_RATE = 2e-4

TRAIN_BATCH_SIZE = 2
EVAL_BATCH_SIZE = 2

GRADIENT_ACCUMULATION_STEPS = 4

SEED = 42

# Early stopping
EARLY_STOPPING_PATIENCE = 3
EARLY_STOPPING_THRESHOLD = 0.001

# ================================================================
# 5. ДАТАСЕТ
# ================================================================

train_data = [
    {"instruction": "Переведи на английский: Привет, как дела?", "output": "Hello, how are you?"},
    {"instruction": "Переведи на английский: Сегодня отличная погода.", "output": "The weather is great today."},
    {"instruction": "Переведи на английский: Я люблю программирование.", "output": "I love programming."},
    {"instruction": "Переведи на английский: Который час?", "output": "What time is it?"},
    {"instruction": "Переведи на английский: Это очень интересно.", "output": "This is very interesting."},
    {"instruction": "Переведи на английский: Доброе утро!", "output": "Good morning!"},
    {"instruction": "Переведи на английский: Спокойной ночи.", "output": "Good night."},
    {"instruction": "Переведи на английский: Как тебя зовут?", "output": "What is your name?"},
]

eval_data = [
    {"instruction": "Переведи на английский: Я хочу пить.", "output": "I am thirsty."},
    {"instruction": "Переведи на английский: Где находится библиотека?", "output": "Where is the library?"},
]

train_dataset = Dataset.from_list(train_data)
eval_dataset = Dataset.from_list(eval_data)

print()
print("=" * 70)
print("DATASET")
print("=" * 70)

print("Train:", len(train_dataset))
print("Eval :", len(eval_dataset))

print()
print("Пример:")
print(train_dataset[0])

print("=" * 70)

# ================================================================
# 6. TOKENIZER
# ================================================================

print()
print("=" * 70)
print("ЗАГРУЗКА TOKENIZER")
print("=" * 70)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
)

# <-- ИЗМЕНЕНО: для GPT-2 нужно явно установить pad_token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Vocab size:", tokenizer.vocab_size)
print("PAD token:", tokenizer.pad_token)
print("EOS token:", tokenizer.eos_token)

print("=" * 70)

# ================================================================
# 7. ФОРМИРОВАНИЕ PROMPT (РУЧНОЕ, БЕЗ CHAT_TEMPLATE)
# ================================================================

# <-- ИЗМЕНЕНО: GPT-2 не поддерживает apply_chat_template,
# поэтому формируем промпт вручную

def build_prompt(instruction, output=None):
    """
    Формирует промпт для GPT-2.
    Если output=None — только инструкция (для инференса).
    """
    prompt = f"Инструкция: {instruction}\nОтвет:"
    if output is not None:
        prompt += f" {output}"
    return prompt

def tokenize_example(example):
    # Полный текст (инструкция + ответ)
    full_text = build_prompt(example["instruction"], example["output"])
    # Только инструкция (для вычисления длины промпта)
    prompt_text = build_prompt(example["instruction"], None)

    full_tokens = tokenizer(
        full_text,
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,
    )
    prompt_tokens = tokenizer(
        prompt_text,
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,
    )

    input_ids = full_tokens["input_ids"]
    attention_mask = full_tokens["attention_mask"]
    prompt_length = len(prompt_tokens["input_ids"])

    labels = input_ids.copy()
    for i in range(min(prompt_length, len(labels))):
        labels[i] = -100

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
    }

# ================================================================
# 8. TOKENIZATION
# ================================================================

print()
print("=" * 70)
print("TOKENIZATION")
print("=" * 70)

tokenized_train = train_dataset.map(
    tokenize_example,
    remove_columns=train_dataset.column_names,
)

tokenized_eval = eval_dataset.map(
    tokenize_example,
    remove_columns=eval_dataset.column_names,
)

print("Train examples:", len(tokenized_train))
print("Eval examples :", len(tokenized_eval))

print("=" * 70)

# ================================================================
# 9. ЗАГРУЗКА БАЗОВОЙ МОДЕЛИ
# ================================================================

print()
print("=" * 70)
print("ЗАГРУЗКА BASE MODEL")
print("=" * 70)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=DTYPE,
    device_map="auto",
)

model.config.use_cache = False

print("Model loaded.")
print("=" * 70)

# ================================================================
# 10. LoRA CONFIG (АДАПТИРОВАНА ДЛЯ GPT-2)
# ================================================================

# <-- ИЗМЕНЕНО: target_modules для GPT-2
# У GPT-2 в self-attention одна матрица c_attn (QKV вместе) и c_proj (выходная)
# Также можно добавить MLP-слои: c_fc, c_proj
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["c_attn", "c_proj"],  # для GPT-2
)

# ================================================================
# 11. ДОБАВЛЯЕМ LoRA
# ================================================================

model = get_peft_model(model, lora_config)

# ================================================================
# 12. ПРОВЕРКА ПАРАМЕТРОВ
# ================================================================

print()
print("=" * 70)
print("LoRA PARAMETER CHECK")
print("=" * 70)

model.print_trainable_parameters()

print("=" * 70)

# ================================================================
# 13. ПРОВЕРКА: ТОЛЬКО LoRA ОБУЧАЕТСЯ
# ================================================================

trainable_names = []
for name, param in model.named_parameters():
    if param.requires_grad:
        trainable_names.append(name)

print()
print("Количество trainable tensors:", len(trainable_names))
print()
print("Первые trainable parameters:")
for name in trainable_names[:20]:
    print("  ", name)

# ================================================================
# 14. DATA COLLATOR
# ================================================================

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
    label_pad_token_id=-100,
    pad_to_multiple_of=8 if torch.cuda.is_available() else None,
)

# ================================================================
# 15. TRAINING ARGUMENTS
# ================================================================

total_steps = (len(tokenized_train) // (TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS)) * NUM_EPOCHS
warmup_steps = int(0.05 * total_steps)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    optim="adamw_torch",
    warmup_steps=warmup_steps,
    lr_scheduler_type="cosine",
    weight_decay=0.01,
    logging_steps=1,
    logging_first_step=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    fp16=(torch.cuda.is_available() and DTYPE == torch.float16),
    bf16=(torch.cuda.is_available() and DTYPE == torch.bfloat16),
    gradient_checkpointing=True,
    seed=SEED,
    report_to="none",
    remove_unused_columns=False,
)

# ================================================================
# 16. МЕТРИКА BLEU
# ================================================================

bleu_metric = evaluate.load("sacrebleu")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    if len(decoded_preds) != len(decoded_labels):
        min_len = min(len(decoded_preds), len(decoded_labels))
        decoded_preds = decoded_preds[:min_len]
        decoded_labels = decoded_labels[:min_len]

    decoded_labels = [[ref] for ref in decoded_labels]

    result = bleu_metric.compute(predictions=decoded_preds, references=decoded_labels)
    return {"bleu": result["score"]}

# ================================================================
# 17. TRAINER
# ================================================================

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=EARLY_STOPPING_PATIENCE,
            early_stopping_threshold=EARLY_STOPPING_THRESHOLD,
        )
    ],
)

# ================================================================
# 18. ТРЕНИРОВКА
# ================================================================

print()
print("=" * 70)
print("НАЧАЛО LoRA TRAINING")
print("=" * 70)

train_result = trainer.train()

print()
print("=" * 70)
print("TRAINING FINISHED")
print("=" * 70)

print(train_result)

# ================================================================
# 19. СОХРАНЕНИЕ LoRA
# ================================================================

print()
print("=" * 70)
print("СОХРАНЕНИЕ LoRA")
print("=" * 70)

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print("LoRA adapter сохранён:")
print(OUTPUT_DIR)

print("=" * 70)

# ================================================================
# 20. ПРОВЕРКА ФАЙЛОВ
# ================================================================

print()
print("Содержимое директории:")

for filename in sorted(os.listdir(OUTPUT_DIR)):
    path = os.path.join(OUTPUT_DIR, filename)
    if os.path.isfile(path):
        size_mb = os.path.getsize(path) / 1024**2
        print(f"{filename:40s}{size_mb:10.2f} MB")

# ================================================================
# 21. ГРАФИК ОБУЧЕНИЯ
# ================================================================

print()
print("=" * 70)
print("ГРАФИК ОБУЧЕНИЯ")
print("=" * 70)

log_history = trainer.state.log_history

train_losses = []
eval_losses = []
bleu_scores = []
epochs = []

for log in log_history:
    if "loss" in log and "epoch" in log:
        train_losses.append(log["loss"])
        epochs.append(log["epoch"])
    if "eval_loss" in log:
        eval_losses.append((log["epoch"], log["eval_loss"]))
    if "bleu" in log:
        bleu_scores.append((log["epoch"], log["bleu"]))

fig, ax1 = plt.subplots(figsize=(10, 6))

ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss", color="tab:red")
ax1.plot(epochs, train_losses, label="Train Loss", color="tab:red", marker="o")
if eval_losses:
    eval_epochs, eval_vals = zip(*eval_losses)
    ax1.plot(eval_epochs, eval_vals, label="Eval Loss", color="tab:orange", marker="s")
ax1.tick_params(axis="y", labelcolor="tab:red")
ax1.legend(loc="upper left")

if bleu_scores:
    ax2 = ax1.twinx()
    ax2.set_ylabel("BLEU", color="tab:blue")
    bleu_epochs, bleu_vals = zip(*bleu_scores)
    ax2.plot(bleu_epochs, bleu_vals, label="BLEU", color="tab:blue", marker="^")
    ax2.tick_params(axis="y", labelcolor="tab:blue")
    ax2.legend(loc="upper right")

plt.title("Training and Validation Metrics")
plt.grid(True)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "training_plot.png"), dpi=150)
plt.show()

print(f"График сохранён в {os.path.join(OUTPUT_DIR, 'training_plot.png')}")
print("=" * 70)

# ================================================================
# 22. ФИНАЛЬНАЯ ОЦЕНКА BLEU НА ВАЛИДАЦИИ
# ================================================================

print()
print("=" * 70)
print("ФИНАЛЬНАЯ ОЦЕНКА BLEU НА ВАЛИДАЦИИ")
print("=" * 70)

def generate_translation(
    instruction,
    max_new_tokens=64,
    temperature=0.2,
):
    model.eval()
    prompt = build_prompt(instruction, None)  # только инструкция
    inputs = tokenizer(prompt, return_tensors="pt")
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=(temperature > 0),
            temperature=temperature,
            top_p=0.9,
            repetition_penalty=1.05,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    generated_tokens = outputs[0, inputs["input_ids"].shape[1]:]
    answer = tokenizer.decode(generated_tokens, skip_special_tokens=True)
    return answer.strip()

references = [ex["output"] for ex in eval_data]
predictions = []
for ex in eval_data:
    pred = generate_translation(ex["instruction"])
    predictions.append(pred)

bleu_score = bleu_metric.compute(predictions=predictions, references=[[ref] for ref in references])
print(f"BLEU на валидационном наборе: {bleu_score['score']:.2f}")

for i, (pred, ref) in enumerate(zip(predictions, references)):
    print(f"\nПример {i+1}:")
    print(f"  Инструкция: {eval_data[i]['instruction']}")
    print(f"  Ожидалось : {ref}")
    print(f"  Получено  : {pred}")

print("=" * 70)

# ================================================================
# 23. ТЕСТ НА TRAIN EXAMPLE
# ================================================================

print()
print("=" * 70)
print("TEST: TRAIN EXAMPLE")
print("=" * 70)

test_instruction = "Переведи на английский: Привет, как дела?"
answer = generate_translation(test_instruction)

print("INPUT :")
print(test_instruction)
print()
print("OUTPUT:")
print(answer)
print("=" * 70)

# ================================================================
# 24. ТЕСТ НА НОВЫХ ПРИМЕРАХ
# ================================================================

test_examples = [
    "Переведи на английский: Я хочу пить.",
    "Переведи на английский: Где находится библиотека?",
    "Переведи на английский: Я люблю Python.",
    "Переведи на английский: До свидания!",
    "Переведи на английский: Как тебя зовут?",
]

print()
print("=" * 70)
print("TEST: NEW EXAMPLES")
print("=" * 70)

for instruction in test_examples:
    answer = generate_translation(instruction)
    print()
    print("INPUT :", instruction)
    print("OUTPUT:", answer)

print("=" * 70)
```


## Итоговые выводы по теме 2

1. **Целевые слои:** начинайте с `["q_proj", "v_proj"]`. Для сложных задач добавляйте `k_proj`, `o_proj` и MLP.
2. **Гиперпараметры:** `r=16`, `lora_alpha=32` (2 × r), `lora_dropout=0.05`, `bias="none"`.
3. **Learning Rate:** LoRA требует более высокой LR (5e-5 – 2e-4) чем Full FT. QLoRA — ещё выше (2e-4).
4. **Эпохи:** 1–3 без Early Stopping, 10–20 с Early Stopping. Следите за eval loss — это главный индикатор переобучения.
5. **Дополнительные настройки:** warmup (5% от total_steps), cosine scheduler, gradient checkpointing для экономии памяти.

**Золотое правило:** если вы не знаете, с чего начать — используйте значения по умолчанию из этой шпаргалки. В 90% случаев они будут работать хорошо.

**Помните:** все параметры взаимосвязаны. Если вы меняете `r`, пересчитайте `alpha`. Если меняете `LR`, проверьте стабильность. И всегда мониторьте eval loss – это ваш главный компас.



# Тема 3. QLoRA: 4-битное квантование для работы на ограниченном железе

**Цель:** объяснить, как QLoRA позволяет дообучать большие модели (65B+) на одном 48GB GPU, разобрать математические основы 4-битного квантования и показать практическую реализацию.

---

## 3.1. Зачем нужен QLoRA?

QLoRA (Quantized Low-Rank Adaptation) — это метод, который **сочетает 4-битное квантование базовой модели с LoRA-адаптерами в полной точности**. Его главная цель — сделать дообучение больших моделей доступным на обычных GPU.

**Ключевая проблема**, которую решает QLoRA:

| Модель | Полная точность (FP16) | QLoRA (4-bit NF4) |
| :--- | :--- | :--- |
| **3B** | ~12 GB (веса + оптимизатор) | ~2 GB |
| **7B** | Требует ~24 GB+ | ~4–5 GB |
| **65B** | Невозможно на одном GPU | **Влезает на один 48GB GPU** |
| **70B** | Кластер из 4+ GPU | **Один 48GB GPU** |

Без QLoRA дообучение 70B модели требует кластера из нескольких дорогих GPU. С QLoRA это становится возможным на одном 48GB GPU. Именно поэтому QLoRA называют **демократизацией доступа к большим языковым моделям**.

---

## 3.2. Математические основы квантования: NF4

### 3.2.1. Что такое квантование?

Квантование — это процесс отображения непрерывных значений (или значений с высокой точностью) в дискретный набор значений с меньшей точностью.

Формально, для веса $w \in \mathbb{R}$ и его квантованного представления $\hat{w}$:

$$
\hat{w} = \text{quantize}(w) = \arg\min_{q \in \mathcal{Q}} |w - q|,
$$

где $\mathcal{Q}$ — конечное множество квантованных значений (кодов).

### 3.2.2. NF4 (NormalFloat 4)

**NF4 (NormalFloat 4)** — это специальный 4-битный формат (16 возможных значений), оптимизированный для **нормально распределённых данных**. Веса нейросетей, как правило, имеют распределение, близкое к нормальному, и NF4 использует это.

#### Математическое обоснование NF4

Пусть веса $W$ следуют нормальному распределению: $W \sim \mathcal{N}(0, \sigma^2)$. Мы хотим разбить диапазон значений на 16 интервалов так, чтобы **каждый интервал имел равную вероятность** (равную площадь под кривой нормального распределения).

**Для стандартного нормального распределения $\mathcal{N}(0,1)$:**

1. Находим квантили $q_i$ такие, что:
   $$
   \int_{-\infty}^{q_i} \frac{1}{\sqrt{2\pi}} e^{-t^2/2} dt = \frac{i}{16}, \quad i = 0, \dots, 16.
   $$

2. Центры интервалов (квантованные значения) — это **средние значения в каждом интервале**:
   $$
   c_i = \mathbb{E}[X \mid X \in [q_{i-1}, q_i]], \quad X \sim \mathcal{N}(0,1).
   $$

**16 бинов NF4** для стандартного нормального распределения (значения округлены):

```
[-1.0, -0.6962, -0.5251, -0.3949, -0.2844, -0.1848, -0.0911, 0.0,
  0.0796,  0.1609,  0.2461,  0.3379,  0.4407,  0.5626,  0.7230, 1.0]
```

#### Преимущество NF4 перед FP4

| Характеристика | FP4 | NF4 |
| :--- | :--- | :--- |
| **Формат** | Стандартный 4-битный float (знак, экспонента, мантисса) | Информационно-оптимальный для нормального распределения |
| **Размещение значений** | Равномерное в диапазоне | **Сгущено вокруг нуля** (там, где больше весов) |
| **Ошибка квантования** | Выше для значений около нуля | **Минимальна** для нормального распределения |

**Математическое сравнение:**

Ошибка квантования $L_2$ для NF4 значительно меньше, чем для FP4:

$$
\mathbb{E}[(w - \hat{w})^2]_{\text{NF4}} \approx 0.75 \cdot \mathbb{E}[(w - \hat{w})^2]_{\text{FP4}}.
$$

Это означает, что NF4 на **~25% точнее** FP4 для нормально распределённых данных — именно таких, как веса LLM.

### 3.2.3. Double Quantization

Double Quantization — это техника, которая **квантует константы квантования** (масштабирующие коэффициенты), экономя дополнительную память.

**Как это работает:**

1. **Первый уровень квантования:** веса квантуются в 4-bit.
2. **Второй уровень квантования:** константы квантования (обычно 32-bit) квантуются в 8-bit.

Это даёт экономию **~0.37 бита на параметр**. Для 70B модели это ~3.2 GB дополнительной экономии.

---

## 3.3. Механизм QLoRA: базовая модель в 4-bit, адаптеры в bf16

QLoRA использует **смешанную точность**:

```
┌─────────────────────────────────────────────────────────────────────┐
│                      QLoRA: СМЕШАННАЯ ТОЧНОСТЬ                     │
│                                                                     │
│  ┌─────────────────────────────────────────────────────────────┐   │
│  │  БАЗОВАЯ МОДЕЛЬ (заморожена)                                │   │
│  │  ┌─────────────────────────────────────────────────────┐   │   │
│  │  │  Веса хранятся в 4-bit NF4                          │   │   │
│  │  │  Память: ~25% от FP16                               │   │   │
│  │  └─────────────────────────────────────────────────────┘   │   │
│  │                    ⬇️  деквантование "на лету"              │   │   │
│  │  ┌─────────────────────────────────────────────────────┐   │   │
│  │  │  Вычисления в bf16 (для forward/backward)           │   │   │
│  │  └─────────────────────────────────────────────────────┘   │   │
│  └─────────────────────────────────────────────────────────────┘   │
│                              +                                      │
│  ┌─────────────────────────────────────────────────────────────┐   │
│  │  LoRA-АДАПТЕРЫ (обучаются)                                  │   │
│  │  ┌─────────────────────────────────────────────────────┐   │   │
│  │  │  Матрицы A и B в полной точности (bf16)            │   │   │
│  │  │  Память: ~0.1–0.5% от модели                       │   │   │
│  │  └─────────────────────────────────────────────────────┘   │   │
│  └─────────────────────────────────────────────────────────────┘   │
└─────────────────────────────────────────────────────────────────────┘
```

### 3.3.1. Процесс квантования и деквантования

**Формальное описание процесса:**

1. **Квантование (сохранение):**
   $$
   w_{\text{int}} = \text{round}\left(\frac{w}{\Delta}\right) + \text{offset},
   $$
   где $\Delta$ — масштабирующий коэффициент, определяющий диапазон значений.

2. **Деквантование (вычисления):**
   $$
   w_{\text{dequant}} = (w_{\text{int}} - \text{offset}) \cdot \Delta.
   $$

3. **Вычисления в bf16:** деквантованные веса участвуют в forward/backward в bf16, обеспечивая численную стабильность.

4. **Обновление адаптеров:** градиенты проходят через замороженную модель и обновляют только $A$ и $B$.

### 3.3.2. Дополнительные техники

**Paged Optimizers** — используют unified memory NVIDIA для автоматической передачи состояния оптимизатора между GPU и CPU при пиках памяти. Это позволяет работать с большими моделями даже при нехватке VRAM.

---

## 3.4. Конфигурация BitsAndBytesConfig

Вот полный конфиг для QLoRA с комментариями:

```python
from transformers import BitsAndBytesConfig
import torch

bnb_config = BitsAndBytesConfig(
    # 1. Включаем 4-битное квантование
    load_in_4bit=True,
    
    # 2. Тип квантования: NF4 (рекомендуется)
    bnb_4bit_quant_type="nf4",
    
    # 3. Double Quantization: квантуем константы квантования
    bnb_4bit_use_double_quant=True,
    
    # 4. Тип данных для вычислений (девантованные веса)
    bnb_4bit_compute_dtype=torch.bfloat16,
)
```

**Что делает каждый параметр:**

| Параметр | Значение | Объяснение |
| :--- | :--- | :--- |
| `load_in_4bit=True` | Включить 4-bit | Активирует квантование модели |
| `bnb_4bit_quant_type="nf4"` | NF4 | Используем NormalFloat 4 — оптимален для LLM |
| `bnb_4bit_use_double_quant=True` | Двойное квантование | Квантуем константы масштабирования → экономия памяти |
| `bnb_4bit_compute_dtype=torch.bfloat16` | bf16 для вычислений | Деквантуем веса до bf16 для forward/backward |

**Использование с моделью:**

```python
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(
    "meta-llama/Llama-2-7b",
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)
```

---

## 3.5. Сравнение LoRA и QLoRA: качество и время

### Качество

QLoRA практически не уступает LoRA по качеству, несмотря на 4-битное квантование:

| Метод | Улучшение метрик | Относительное качество |
| :--- | :--- | :--- |
| **LoRA** | +10–20 пунктов над базовой модельой | 100% (эталон) |
| **QLoRA (4-bit)** | +8–14 пунктов над базовой моделью | ~95–98% от LoRA |

В оригинальной статье QLoRA показано, что метод **сохраняет качество полноценной 16-битной настройки**.

### Память и оборудование

| Параметр | LoRA | QLoRA (4-bit) |
| :--- | :--- | :--- |
| **Требуемые GPU** | 4 GPU | 2 GPU |
| **Пиковая RAM** | 100% | ~66% от LoRA |
| **Максимальная модель** | Ограничена памятью | **65B на одном 48GB GPU** |

### Время обучения

| Аспект | LoRA | QLoRA (4-bit) |
| :--- | :--- | :--- |
| **Скорость обучения** | Быстрее | **На 28–32% медленнее** |
| **Причина** | — | Деквантование "на лету" + менее оптимизированные библиотеки |

### Сравнительная таблица

| Критерий | LoRA | QLoRA (4-bit NF4) |
| :--- | :--- | :--- |
| **Качество** | ⭐⭐⭐⭐⭐ (эталон) | ⭐⭐⭐⭐ (95–98% от LoRA) |
| **Память** | ⭐⭐⭐ | ⭐⭐⭐⭐⭐ (~66% от LoRA) |
| **Скорость** | ⭐⭐⭐⭐⭐ | ⭐⭐⭐⭐ (на 28–32% медленнее) |
| **Макс. размер модели** | Ограничен VRAM | **65B на 48GB GPU** |
| **Когда выбрать** | Есть достаточно VRAM, важна скорость | **VRAM ограничена, нужно дообучать большие модели** |

---

## 3.6. Математический итог: почему QLoRA работает?

QLoRA эффективен благодаря комбинации двух математических идей:

1. **Низкоранговая адаптация (LoRA):**
   $$
   W' = W_0 + B A, \quad B \in \mathbb{R}^{d \times r}, A \in \mathbb{R}^{r \times k}, r \ll \min(d,k).
   $$
   Только $A$ и $B$ обучаются, основная модель заморожена.

2. **Информационно-оптимальное квантование (NF4):**
   $$
   \text{NF4}(w) = \arg\min_{c \in \mathcal{C}} |w - c|,
   $$
   где $\mathcal{C}$ — множество из 16 значений, оптимизированных для нормального распределения.

Вместе они дают:
- **Минимальную потерю качества:** LoRA сохраняет выразительность, NF4 сохраняет точность квантования.
- **Максимальную экономию памяти:** LoRA обучает мало параметров, NF4 сжимает основную модель.
- **Доступность:** 70B модели становятся реальностью на одном 48GB GPU.

---

## 3.7. Практический пример: QLoRA на TinyLlama 1.1B

Ниже приведён полный код для дообучения TinyLlama 1.1B с использованием QLoRA. Этот код демонстрирует все ключевые элементы QLoRA: 4-битное квантование, LoRA-адаптеры, обучение и инференс.

```python
# ================================================================
# QLoRA Fine-Tuning (4-bit Quantized)
# TinyLlama/TinyLlama-1.1B-Chat-v1.0
# ================================================================
# 1. УСТАНОВКА
# ================================================================

# <-- ИЗМЕНЕНО: добавлена библиотека bitsandbytes
!pip install -q -U \
    transformers \
    peft \
    accelerate \
    datasets \
    evaluate \
    sacrebleu \
    matplotlib \
    bitsandbytes \
    "numpy<2.1"

# ================================================================
# 2. ИМПОРТЫ
# ================================================================

import os
import torch
import numpy as np
import matplotlib.pyplot as plt
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    Trainer,
    TrainingArguments,
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback,
    BitsAndBytesConfig,          # <-- НОВОЕ: для 4-битного квантования
)
from peft import (
    LoraConfig,
    get_peft_model,
    PeftModel,
)
import evaluate

# ================================================================
# 3. ПРОВЕРКА ОКРУЖЕНИЯ
# ================================================================

print("=" * 70)
print("ПРОВЕРКА ОКРУЖЕНИЯ")
print("=" * 70)

print("PyTorch:", torch.__version__)
print("CUDA доступна:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "VRAM:",
        round(
            torch.cuda.get_device_properties(0).total_memory / 1024**3,
            2
        ),
        "GB"
    )

    if torch.cuda.is_bf16_supported():
        DTYPE = torch.bfloat16
        print("Dtype: torch.bfloat16")
    else:
        DTYPE = torch.float16
        print("Dtype: torch.float16")
else:
    DTYPE = torch.float32
    print("Dtype: torch.float32")

print("=" * 70)

# ================================================================
# 4. КОНФИГУРАЦИЯ (АДАПТИРОВАНА ДЛЯ QLoRA)
# ================================================================

# <-- ИЗМЕНЕНО: можно подставить любую модель
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
# Альтернативы:
# MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
# MODEL_NAME = "microsoft/Phi-3-mini-4k-instruct"

OUTPUT_DIR = "./tinyllama_1.1b_qlora_translation"   # <-- папка для QLoRA

MAX_LENGTH = 256

# LoRA (те же параметры, что и для обычного LoRA)
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05

# Training
NUM_EPOCHS = 20
LEARNING_RATE = 2e-4             # QLoRA использует ту же LR

TRAIN_BATCH_SIZE = 2
EVAL_BATCH_SIZE = 2

GRADIENT_ACCUMULATION_STEPS = 4

SEED = 42

# Early stopping
EARLY_STOPPING_PATIENCE = 3
EARLY_STOPPING_THRESHOLD = 0.001

# ================================================================
# 5. ДАТАСЕТ
# ================================================================

train_data = [
    {"instruction": "Переведи на английский: Привет, как дела?", "output": "Hello, how are you?"},
    {"instruction": "Переведи на английский: Сегодня отличная погода.", "output": "The weather is great today."},
    {"instruction": "Переведи на английский: Я люблю программирование.", "output": "I love programming."},
    {"instruction": "Переведи на английский: Который час?", "output": "What time is it?"},
    {"instruction": "Переведи на английский: Это очень интересно.", "output": "This is very interesting."},
    {"instruction": "Переведи на английский: Доброе утро!", "output": "Good morning!"},
    {"instruction": "Переведи на английский: Спокойной ночи.", "output": "Good night."},
    {"instruction": "Переведи на английский: Как тебя зовут?", "output": "What is your name?"},
]

eval_data = [
    {"instruction": "Переведи на английский: Я хочу пить.", "output": "I am thirsty."},
    {"instruction": "Переведи на английский: Где находится библиотека?", "output": "Where is the library?"},
]

train_dataset = Dataset.from_list(train_data)
eval_dataset = Dataset.from_list(eval_data)

print()
print("=" * 70)
print("DATASET")
print("=" * 70)

print("Train:", len(train_dataset))
print("Eval :", len(eval_dataset))

print()
print("Пример:")
print(train_dataset[0])

print("=" * 70)

# ================================================================
# 6. TOKENIZER
# ================================================================

print()
print("=" * 70)
print("ЗАГРУЗКА TOKENIZER")
print("=" * 70)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Vocab size:", tokenizer.vocab_size)
print("PAD token:", tokenizer.pad_token)
print("EOS token:", tokenizer.eos_token)

print("=" * 70)

# ================================================================
# 7. ФОРМИРОВАНИЕ PROMPT
# ================================================================

def build_messages(example):
    return [
        {"role": "user", "content": example["instruction"]},
        {"role": "assistant", "content": example["output"]},
    ]

def tokenize_example(example):
    messages = build_messages(example)
    full_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )
    prompt_messages = [{"role": "user", "content": example["instruction"]}]
    prompt_text = tokenizer.apply_chat_template(
        prompt_messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    full_tokens = tokenizer(
        full_text,
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,
    )
    prompt_tokens = tokenizer(
        prompt_text,
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,
    )

    input_ids = full_tokens["input_ids"]
    attention_mask = full_tokens["attention_mask"]
    prompt_length = len(prompt_tokens["input_ids"])

    labels = input_ids.copy()
    for i in range(min(prompt_length, len(labels))):
        labels[i] = -100

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
    }

# ================================================================
# 8. TOKENIZATION
# ================================================================

print()
print("=" * 70)
print("TOKENIZATION")
print("=" * 70)

tokenized_train = train_dataset.map(
    tokenize_example,
    remove_columns=train_dataset.column_names,
)

tokenized_eval = eval_dataset.map(
    tokenize_example,
    remove_columns=eval_dataset.column_names,
)

print("Train examples:", len(tokenized_train))
print("Eval examples :", len(tokenized_eval))

print("=" * 70)

# ================================================================
# 9. КОНФИГУРАЦИЯ КВАНТОВАНИЯ (QLoRA)
# ================================================================

print()
print("=" * 70)
print("НАСТРОЙКА 4-BIT КВАНТОВАНИЯ")
print("=" * 70)

# <-- НОВОЕ: BitsAndBytesConfig для QLoRA
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,                         # Включаем 4-битное квантование
    bnb_4bit_quant_type="nf4",                 # NF4 — оптимальный для LLM
    bnb_4bit_use_double_quant=True,            # Double Quantization — экономия памяти
    bnb_4bit_compute_dtype=DTYPE,              # Деквантуем до bf16/fp16 для вычислений
)

print("4-bit квантование включено:")
print(f"  Тип: NF4")
print(f"  Double Quantization: True")
print(f"  Compute dtype: {DTYPE}")

print("=" * 70)

# ================================================================
# 10. ЗАГРУЗКА КВАНТОВАННОЙ МОДЕЛИ
# ================================================================

print()
print("=" * 70)
print("ЗАГРУЗКА КВАНТОВАННОЙ МОДЕЛИ (4-bit)")
print("=" * 70)

# <-- ИЗМЕНЕНО: добавлен quantization_config
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,            # <-- КЛЮЧЕВОЕ ОТЛИЧИЕ QLoRA
    device_map="auto",
    trust_remote_code=True,
)

model.config.use_cache = False

print("Квантованная модель загружена.")
print("=" * 70)

# ================================================================
# 11. LoRA CONFIG
# ================================================================

# <-- target_modules зависят от архитектуры модели.
# Для TinyLlama (как и LLaMA, Mistral, Qwen) используются:
target_modules = ["q_proj", "k_proj", "v_proj", "o_proj"]
# Если модель другая (например, GPT-2), нужно изменить на ["c_attn", "c_proj"]

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=target_modules,
)

# ================================================================
# 12. ДОБАВЛЯЕМ LoRA
# ================================================================

model = get_peft_model(model, lora_config)

# ================================================================
# 13. ПРОВЕРКА ПАРАМЕТРОВ
# ================================================================

print()
print("=" * 70)
print("LoRA PARAMETER CHECK")
print("=" * 70)

model.print_trainable_parameters()

print("=" * 70)

# ================================================================
# 14. DATA COLLATOR
# ================================================================

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
    label_pad_token_id=-100,
    pad_to_multiple_of=8 if torch.cuda.is_available() else None,
)

# ================================================================
# 15. TRAINING ARGUMENTS
# ================================================================

total_steps = (len(tokenized_train) // (TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS)) * NUM_EPOCHS
warmup_steps = int(0.05 * total_steps)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    optim="adamw_torch",
    warmup_steps=warmup_steps,
    lr_scheduler_type="cosine",
    weight_decay=0.01,
    logging_steps=1,
    logging_first_step=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    fp16=(torch.cuda.is_available() and DTYPE == torch.float16),
    bf16=(torch.cuda.is_available() and DTYPE == torch.bfloat16),
    gradient_checkpointing=True,
    seed=SEED,
    report_to="none",
    remove_unused_columns=False,
)

# ================================================================
# 16. МЕТРИКА BLEU
# ================================================================

bleu_metric = evaluate.load("sacrebleu")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    if len(decoded_preds) != len(decoded_labels):
        min_len = min(len(decoded_preds), len(decoded_labels))
        decoded_preds = decoded_preds[:min_len]
        decoded_labels = decoded_labels[:min_len]

    decoded_labels = [[ref] for ref in decoded_labels]

    result = bleu_metric.compute(predictions=decoded_preds, references=decoded_labels)
    return {"bleu": result["score"]}

# ================================================================
# 17. TRAINER
# ================================================================

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=EARLY_STOPPING_PATIENCE,
            early_stopping_threshold=EARLY_STOPPING_THRESHOLD,
        )
    ],
)

# ================================================================
# 18. ТРЕНИРОВКА
# ================================================================

print()
print("=" * 70)
print("НАЧАЛО QLoRA TRAINING (4-bit)")
print("=" * 70)

train_result = trainer.train()

print()
print("=" * 70)
print("TRAINING FINISHED")
print("=" * 70)

print(train_result)

# ================================================================
# 19. СОХРАНЕНИЕ LoRA
# ================================================================

print()
print("=" * 70)
print("СОХРАНЕНИЕ QLoRA АДАПТЕРА")
print("=" * 70)

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print("QLoRA adapter сохранён:")
print(OUTPUT_DIR)

print("=" * 70)

# ================================================================
# 20. ПРОВЕРКА ФАЙЛОВ
# ================================================================

print()
print("Содержимое директории:")

for filename in sorted(os.listdir(OUTPUT_DIR)):
    path = os.path.join(OUTPUT_DIR, filename)
    if os.path.isfile(path):
        size_mb = os.path.getsize(path) / 1024**2
        print(f"{filename:40s}{size_mb:10.2f} MB")

# ================================================================
# 21. ГРАФИК ОБУЧЕНИЯ
# ================================================================

print()
print("=" * 70)
print("ГРАФИК ОБУЧЕНИЯ")
print("=" * 70)

log_history = trainer.state.log_history

train_losses = []
eval_losses = []
bleu_scores = []
epochs = []

for log in log_history:
    if "loss" in log and "epoch" in log:
        train_losses.append(log["loss"])
        epochs.append(log["epoch"])
    if "eval_loss" in log:
        eval_losses.append((log["epoch"], log["eval_loss"]))
    if "bleu" in log:
        bleu_scores.append((log["epoch"], log["bleu"]))

fig, ax1 = plt.subplots(figsize=(10, 6))

ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss", color="tab:red")
ax1.plot(epochs, train_losses, label="Train Loss", color="tab:red", marker="o")
if eval_losses:
    eval_epochs, eval_vals = zip(*eval_losses)
    ax1.plot(eval_epochs, eval_vals, label="Eval Loss", color="tab:orange", marker="s")
ax1.tick_params(axis="y", labelcolor="tab:red")
ax1.legend(loc="upper left")

if bleu_scores:
    ax2 = ax1.twinx()
    ax2.set_ylabel("BLEU", color="tab:blue")
    bleu_epochs, bleu_vals = zip(*bleu_scores)
    ax2.plot(bleu_epochs, bleu_vals, label="BLEU", color="tab:blue", marker="^")
    ax2.tick_params(axis="y", labelcolor="tab:blue")
    ax2.legend(loc="upper right")

plt.title("QLoRA Training and Validation Metrics (4-bit)")
plt.grid(True)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "training_plot.png"), dpi=150)
plt.show()

print(f"График сохранён в {os.path.join(OUTPUT_DIR, 'training_plot.png')}")
print("=" * 70)

# ================================================================
# 22. ФИНАЛЬНАЯ ОЦЕНКА BLEU НА ВАЛИДАЦИИ
# ================================================================

print()
print("=" * 70)
print("ФИНАЛЬНАЯ ОЦЕНКА BLEU НА ВАЛИДАЦИИ (QLoRA)")
print("=" * 70)

def generate_translation(
    instruction,
    max_new_tokens=64,
    temperature=0.2,
):
    model.eval()
    messages = [{"role": "user", "content": instruction}]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(text, return_tensors="pt")
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=(temperature > 0),
            temperature=temperature,
            top_p=0.9,
            repetition_penalty=1.05,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    generated_tokens = outputs[0, inputs["input_ids"].shape[1]:]
    answer = tokenizer.decode(generated_tokens, skip_special_tokens=True)
    return answer.strip()

references = [ex["output"] for ex in eval_data]
predictions = []
for ex in eval_data:
    pred = generate_translation(ex["instruction"])
    predictions.append(pred)

bleu_score = bleu_metric.compute(predictions=predictions, references=[[ref] for ref in references])
print(f"BLEU на валидационном наборе: {bleu_score['score']:.2f}")

for i, (pred, ref) in enumerate(zip(predictions, references)):
    print(f"\nПример {i+1}:")
    print(f"  Инструкция: {eval_data[i]['instruction']}")
    print(f"  Ожидалось : {ref}")
    print(f"  Получено  : {pred}")

print("=" * 70)

# ================================================================
# 23. ТЕСТ НА TRAIN EXAMPLE
# ================================================================

print()
print("=" * 70)
print("TEST: TRAIN EXAMPLE (QLoRA)")
print("=" * 70)

test_instruction = "Переведи на английский: Привет, как дела?"
answer = generate_translation(test_instruction)

print("INPUT :")
print(test_instruction)
print()
print("OUTPUT:")
print(answer)
print("=" * 70)

# ================================================================
# 24. ТЕСТ НА НОВЫХ ПРИМЕРАХ
# ================================================================

test_examples = [
    "Переведи на английский: Я хочу пить.",
    "Переведи на английский: Где находится библиотека?",
    "Переведи на английский: Я люблю Python.",
    "Переведи на английский: До свидания!",
    "Переведи на английский: Как тебя зовут?",
]

print()
print("=" * 70)
print("TEST: NEW EXAMPLES (QLoRA)")
print("=" * 70)

for instruction in test_examples:
    answer = generate_translation(instruction)
    print()
    print("INPUT :", instruction)
    print("OUTPUT:", answer)

print("=" * 70)
```

---

## Итоговые выводы по теме 3

1. **QLoRA** — это LoRA + 4-битное квантование базовой модели. Позволяет дообучать **65B модели на одном 48GB GPU**.

2. **NF4** — информационно-оптимальный 4-битный формат для нормального распределения. Основан на квантилях стандартного нормального распределения, что даёт **~25% меньшую ошибку квантования** по сравнению с FP4.

3. **Double Quantization** квантует константы масштабирования, экономя ~0.37 бита на параметр (~3.2 GB для 70B модели).

4. **Механизм:** базовая модель хранится в 4-bit NF4, деквантуется "на лету" до bf16 для вычислений, LoRA-адаптеры обучаются в полной точности.

5. **QLoRA почти не уступает LoRA по качеству** (95–98%), но требует **на 28–32% больше времени** из-за деквантования. При этом экономит **~34% памяти** и требует **вдвое меньше GPU**.

**Практический совет:** если у вас есть GPU с 24+ GB VRAM и вам важна скорость — используйте LoRA. Если вы хотите дообучать 13B+ модели на ограниченном железе — QLoRA ваш выбор.

## Тема 4. Подготовка данных для тонкой настройки – практический гайд

**Цель:** научиться подготавливать датасеты для обучения с учителем (SFT). Мы разберём форматы данных, преобразование, очистку, аугментацию, негативные примеры и packing. Все примеры привязаны к реальному коду.

---

### Введение: почему качество данных — ключевой фактор?

Гиперпараметры LoRA — важны, но **качество данных определяет качество модели**. Плохие данные — плохая модель, даже с идеальными гиперпараметрами. В этой теме мы разберём, как подготовить датасет для SFT: от формата до финальной очистки.

---

## 4.1. Форматы данных: Alpaca, ShareGPT, ChatML

### 4.1.1. Формат Alpaca (базовый)

**Структура:**
```json
{
  "instruction": "Переведи на английский: Привет, как дела?",
  "input": "",
  "output": "Hello, how are you?"
}
```

**Когда использовать:** простые задачи, где один запрос → один ответ.

**Пример конвертации из CSV в Alpaca:**
```python
import pandas as pd

def csv_to_alpaca(csv_path, output_path):
    """Конвертирует CSV в формат Alpaca."""
    df = pd.read_csv(csv_path)
    alpaca_data = []
    
    for _, row in df.iterrows():
        alpaca_data.append({
            "instruction": row["question"],
            "input": row.get("context", ""),
            "output": row["answer"]
        })
    
    import json
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(alpaca_data, f, ensure_ascii=False, indent=2)
    
    print(f"✅ Конвертировано {len(alpaca_data)} примеров в {output_path}")
```

---

### 4.1.2. Формат ShareGPT (диалоговый)

**Структура:**
```json
{
  "conversations": [
    {"from": "human", "value": "Привет, как дела?"},
    {"from": "gpt", "value": "Привет! У меня всё отлично, спасибо."},
    {"from": "human", "value": "Что нового?"},
    {"from": "gpt", "value": "Сегодня отличная погода, и я изучаю AI."}
  ]
}
```

**Когда использовать:** многооборотные диалоги, чаты, техподдержка.

**Конвертация из Alpaca в ShareGPT:**
```python
def alpaca_to_shareGPT(alpaca_data):
    """Конвертирует Alpaca в ShareGPT."""
    sharegpt_data = []
    
    for ex in alpaca_data:
        messages = []
        if ex.get("instruction"):
            messages.append({"from": "human", "value": ex["instruction"]})
        if ex.get("input"):
            messages[-1]["value"] += "\n" + ex["input"]
        messages.append({"from": "gpt", "value": ex["output"]})
        
        sharegpt_data.append({"conversations": messages})
    
    return sharegpt_data
```

---

### 4.1.3. Формат ChatML (стандарт OpenAI)

**Структура (текстовое представление):**
```
<|im_start|>system
Ты — полезный ассистент.<|im_end|>
<|im_start|>user
Привет, как дела?<|im_end|>
<|im_start|>assistant
Привет! У меня всё отлично.<|im_end|>
```

**Когда использовать:** все современные чат-модели (GPT, Qwen, Llama с chat-шаблоном).

**Конвертация ShareGPT в ChatML:**
```python
def sharegpt_to_chatml(sharegpt_data):
    """Конвертирует ShareGPT в ChatML (строки)."""
    chatml_data = []
    
    for ex in sharegpt_data:
        lines = []
        for msg in ex["conversations"]:
            role = "user" if msg["from"] == "human" else "assistant"
            lines.append(f"<|im_start|>{role}\n{msg['value']}<|im_end|>")
        
        # Добавляем системное сообщение
        system = "<|im_start|>system\nТы — полезный ассистент.<|im_end|>"
        chatml_data.append(system + "\n".join(lines))
    
    return chatml_data
```

---

## 4.2. `apply_chat_template()`: почему это критично?

**Проблема:** каждая модель использует **свой формат** специальных токенов. Если вы просто объедините сообщения через "\n", модель не поймёт, где чья роль.

**Примеры разных шаблонов:**

| Модель | Формат |
| :--- | :--- |
| **Llama 3** | `<|begin_of_text|><|start_header_id|>user<|end_header_id|>...<|eot_id|>` |
| **Qwen 2.5** | `<|im_start|>user\n...<|im_end|>` |
| **Phi-3** | `<|user|>\n...<|end|><|assistant|>\n...<|end|>` |
| **GPT-4** | `{"role": "user", "content": "..."}` (токенизируется специально) |

**Решение:** всегда используйте `tokenizer.apply_chat_template()`.

```python
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct")

messages = [
    {"role": "system", "content": "Ты — полезный ассистент."},
    {"role": "user", "content": "Привет, как дела?"},
]

# ❌ НЕПРАВИЛЬНО — ручное форматирование
wrong_text = "System: Ты — полезный ассистент.\nUser: Привет, как дела?\nAssistant:"

# ✅ ПРАВИЛЬНО — apply_chat_template
correct_text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

print(correct_text)
# Вывод: <|im_start|>system\nТы — полезный ассистент.<|im_end|>
#        <|im_start|>user\nПривет, как дела?<|im_end|>
#        <|im_start|>assistant\n
```

**Почему это критично:**
1. **Без apply_chat_template** модель не знает, где системная инструкция, где пользователь, где ассистент.
2. **Специальные токены** (например, `<|im_end|>`) — это часть обучения модели. Их пропуск = потеря контекста.
3. **Ошибки форматирования** — частая причина NaN loss и плохого качества.

**Пример ошибки в коде (из реальной практики):**
```python
# ❌ Забыли добавить add_generation_prompt=True
text = tokenizer.apply_chat_template(messages, tokenize=False)
# Модель не знает, что после user должен идти assistant
```

**Исправление:**
```python
# ✅ Всегда указывайте add_generation_prompt=True
text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
```

---

## 4.3. Качество данных: чистка, дедупликация

### 4.3.1. Очистка текста

```python
import re

def clean_text(text):
    """Очищает текст от мусора."""
    if not text:
        return ""
    
    # Удаляем лишние пробелы
    text = re.sub(r'\s+', ' ', text).strip()
    
    # Удаляем невидимые символы
    text = re.sub(r'[\x00-\x1F\x7F]', '', text)
    
    # Удаляем URL (опционально)
    # text = re.sub(r'https?://\S+', '', text)
    
    return text

def clean_dataset(dataset):
    """Применяет очистку ко всем текстам."""
    for ex in dataset:
        if "instruction" in ex:
            ex["instruction"] = clean_text(ex["instruction"])
        if "input" in ex:
            ex["input"] = clean_text(ex["input"])
        if "output" in ex:
            ex["output"] = clean_text(ex["output"])
        if "conversations" in ex:
            for msg in ex["conversations"]:
                msg["value"] = clean_text(msg["value"])
    return dataset
```

### 4.3.2. Дедупликация

```python
def deduplicate_dataset(dataset, key="instruction"):
    """Удаляет дубликаты по указанному ключу."""
    seen = set()
    dedup = []
    
    for ex in dataset:
        value = ex.get(key, "")
        # Берём первые 50 символов для сравнения
        hash_key = value[:50].strip()
        if hash_key and hash_key not in seen:
            seen.add(hash_key)
            dedup.append(ex)
    
    removed = len(dataset) - len(dedup)
    print(f"✅ Удалено {removed} дубликатов, осталось {len(dedup)}")
    return dedup
```

### 4.3.3. Фильтрация по длине

```python
def filter_by_length(dataset, min_len=5, max_len=2048):
    """Удаляет слишком короткие и слишком длинные примеры."""
    filtered = []
    
    for ex in dataset:
        text = ex.get("instruction", "") + " " + ex.get("output", "")
        length = len(text)
        if min_len <= length <= max_len:
            filtered.append(ex)
    
    removed = len(dataset) - len(filtered)
    print(f"✅ Отфильтровано {removed} примеров, осталось {len(filtered)}")
    return filtered
```

---

## 4.4. Аугментация данных

### 4.4.1. Перефразирование

Используем LLM для создания вариаций:

```python
def paraphrase_example(example, llm_func, num_variations=2):
    """
    Генерирует вариации инструкции с помощью LLM.
    llm_func — функция, которая принимает промпт и возвращает текст.
    """
    prompt = f"""Перефразируй следующий запрос, сохранив смысл.
    Запрос: {example['instruction']}
    
    Дай {num_variations} вариантов перефразирования, каждый с новой строки:"""
    
    variations = llm_func(prompt).strip().split('\n')
    return [example["instruction"]] + variations[:num_variations]
```

### 4.4.2. Back-translation

Перевод на другой язык и обратно для увеличения разнообразия:

```python
def back_translate(text, translator, target_lang="ru"):
    """
    Делает back-translation: текст → целевой язык → обратно.
    """
    # Переводим на целевой язык
    translated = translator.translate(text, target_lang)
    # Переводим обратно
    back_translated = translator.translate(translated, "en")
    return back_translated
```

### 4.4.3. Генерация синтетических данных через GPT-4

```python
def generate_synthetic_data(topic, num_examples=10, gpt4_func=None):
    """
    Генерирует синтетические примеры для заданной темы.
    """
    prompt = f"""Сгенерируй {num_examples} пар вопрос-ответ на тему "{topic}".
    Формат: каждый пример в виде JSON с полями 'instruction' и 'output'.
    """
    
    response = gpt4_func(prompt)
    # Парсим JSON из ответа
    import json
    synthetic_data = json.loads(response)
    return synthetic_data
```

---

## 4.5. Negative sampling: отказы и границы компетенции

### Зачем это нужно?

Модель должна уметь говорить **«Я не знаю»**, если вопрос выходит за рамки её знаний. Без негативных примеров модель будет галлюцинировать на незнакомые темы.

### Примеры негативных примеров

```python
negative_examples = [
    {
        "instruction": "Как взломать банковский аккаунт?",
        "output": "Извините, я не могу ответить на этот вопрос. Это незаконно."
    },
    {
        "instruction": "Какая будет погода завтра в Москве?",
        "output": "Я не знаю прогноз погоды. У меня нет доступа к актуальной информации."
    },
    {
        "instruction": "Расскажи последние новости о политике.",
        "output": "Я не могу предоставить актуальные новости. Мои знания ограничены."
    },
    {
        "instruction": "Что думаешь о компании X?",
        "output": "У меня нет информации о конкретных компаниях."
    },
    {
        "instruction": "Как сделать взрывчатку?",
        "output": "Я не могу предоставить информацию о создании опасных веществ."
    }
]
```

### Добавление негативных примеров в датасет

```python
def add_negative_samples(dataset, negative_examples, ratio=0.05):
    """
    Добавляет негативные примеры в датасет.
    ratio — доля негативных примеров от общего числа.
    """
    import random
    
    num_negative = int(len(dataset) * ratio)
    selected_negatives = random.sample(
        negative_examples,
        min(num_negative, len(negative_examples))
    )
    
    final_dataset = dataset + selected_negatives
    random.shuffle(final_dataset)
    
    print(f"✅ Добавлено {len(selected_negatives)} негативных примеров")
    print(f"   Доля негативных примеров: {len(selected_negatives)/len(final_dataset):.1%}")
    
    return final_dataset
```

---

## 4.6. Packing: объединение коротких примеров

### Что такое packing?

**Packing** — это техника объединения нескольких коротких примеров в одну последовательность для увеличения эффективности обучения.

### Почему это важно?

1. **Ускорение обучения:** GPU обрабатывает фиксированное количество токенов. Если примеры короткие — эффективность падает.
2. **Экономия памяти:** меньше паддинга = меньше лишней памяти.
3. **Улучшение качества:** модель видит больше разнообразных примеров за один шаг.

### Пример packing (с использованием библиотеки)

```python
from transformers import DataCollatorForLanguageModeling
from transformers import DataCollatorForSeq2Seq

def create_packed_dataset(dataset, tokenizer, max_length=2048):
    """
    Создаёт датасет с packing для коротких примеров.
    Альтернативно: использовать DataCollatorForSeq2Seq с padding=False.
    """
    # Этот подход лучше делать через dataloader, не через датасет
    # Используйте datacollator:
    
    from transformers import DataCollatorForSeq2Seq
    
    data_collator = DataCollatorForSeq2Seq(
        tokenizer=tokenizer,
        model=None,
        padding=True,  # pad до макс. длины
        label_pad_token_id=-100,
        pad_to_multiple_of=8,  # выравнивание для эффективности
    )
    
    # Альтернатива: использовать свой collator с packing
    return data_collator
```

### Реализация packing вручную

```python
def pack_examples(examples, tokenizer, max_length=2048):
    """
    Упаковывает несколько коротких примеров в одну последовательность.
    """
    packed_input_ids = []
    packed_labels = []
    packed_attention_mask = []
    
    for ex in examples:
        input_ids = ex["input_ids"]
        labels = ex["labels"]
        
        if len(packed_input_ids) + len(input_ids) > max_length:
            # Если не влезает — завершаем текущий пакет
            yield {
                "input_ids": packed_input_ids,
                "labels": packed_labels,
                "attention_mask": [1] * len(packed_input_ids)
            }
            packed_input_ids = []
            packed_labels = []
        
        packed_input_ids.extend(input_ids)
        packed_labels.extend(labels)
    
    if packed_input_ids:
        yield {
            "input_ids": packed_input_ids,
            "labels": packed_labels,
            "attention_mask": [1] * len(packed_input_ids)
        }
```

### Советы по packing

1. **Не смешивайте системные сообщения с user/assistant** при packing.
2. **Добавляйте специальный разделитель** между примерами (`<|im_end|>` или `eos_token`).
3. **Используйте attention_mask**, чтобы модель знала, где реальные токены.

---

## Итоговый чек-лист подготовки данных

- [ ] **Выбран формат** (Alpaca, ShareGPT, ChatML) под задачу.
- [ ] **Применён `apply_chat_template()`** для форматирования диалогов.
- [ ] **Проведена очистка текста** (удаление мусора, нормализация).
- [ ] **Удалены дубликаты** (дедупликация).
- [ ] **Отфильтрованы слишком короткие/длинные** примеры.
- [ ] **Добавлены негативные примеры** (отказы) — не менее 5%.
- [ ] **Проведена аугментация** (перефразирование, синтетика) при необходимости.
- [ ] **Настроен packing** для эффективного обучения.

---

## Итоговые выводы по теме 4

1. **Форматы:** Alpaca — для простых задач, ShareGPT — для диалогов, ChatML — для современных чат-моделей.
2. **apply_chat_template()** — критически важен для правильной токенизации. Без него модель не поймёт роли.
3. **Качество данных:** чистка, дедупликация, фильтрация по длине — база.
4. **Аугментация:** перефразирование, back-translation, синтетика через GPT-4 увеличивают разнообразие.
5. **Negative sampling:** обязательный приём для обучения модели говорить «Я не знаю».
6. **Packing:** ускоряет обучение и экономит память на коротких примерах.

**Золотое правило:** потратьте 80% времени на данные и 20% на модель. Качество данных определяет качество результата.

# Лекция 4.2. LoRA и QLoRA: практическая параметрически-эффективная настройка

## Тема 5. Практическая реализация с PEFT (код, шаг за шагом)

**Цель:** дать пошаговую инструкцию по реализации LoRA/QLoRA в коде — от установки библиотек до инференса. Все примеры основаны на актуальных версиях библиотек (2026) и используют современный API `SFTTrainer` / `SFTConfig` из `trl`.

---

## 5.1. Установка библиотек

### Минимальный набор для LoRA / QLoRA

```bash
pip install -q -U \
    transformers \           # Загрузка моделей и токенизаторов
    peft \                   # LoRA, QLoRA
    accelerate \             # Оптимизация GPU/CPU
    bitsandbytes \           # 4-битное квантование (QLoRA)
    datasets \               # Работа с датасетами
    evaluate \               # Метрики (BLEU, ROUGE)
    trl \                    # SFTTrainer, SFTConfig
    wandb \                  # Логирование (опционально)
    "numpy<2.1"              # Совместимость
```

### Проверка установки

```python
import transformers, peft, accelerate, bitsandbytes, trl, datasets
print(f"Transformers: {transformers.__version__}")
print(f"PEFT: {peft.__version__}")
print(f"TRL: {trl.__version__}")
```

---

## 5.2. Загрузка модели с квантованием (QLoRA)

### 5.2.1. Конфигурация квантования (BitsAndBytesConfig)

```python
from transformers import BitsAndBytesConfig
import torch

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,                         # Включаем 4-битное квантование
    bnb_4bit_quant_type="nf4",                 # NF4 — оптимален для LLM
    bnb_4bit_use_double_quant=True,            # Double Quantization — экономия памяти
    bnb_4bit_compute_dtype=torch.bfloat16,     # Деквантуем до bf16 для вычислений
)

# Для обычного LoRA (без квантования) — просто не передаём quantization_config
```

### 5.2.2. Определение типа данных (DTYPE)

```python
if torch.cuda.is_available():
    if torch.cuda.is_bf16_supported():
        DTYPE = torch.bfloat16
        USE_BF16 = True
        USE_FP16 = False
    else:
        DTYPE = torch.float16
        USE_BF16 = False
        USE_FP16 = True
else:
    DTYPE = torch.float32
    USE_BF16 = False
    USE_FP16 = False
```

### 5.2.3. Загрузка модели

```python
from transformers import AutoModelForCausalLM, AutoTokenizer

# Для QLoRA
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,       # только для QLoRA
    device_map="auto",
    torch_dtype=DTYPE,
    trust_remote_code=True,
)

# Для обычного LoRA
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=DTYPE,
    device_map="auto",
    trust_remote_code=True,
)

model.config.use_cache = False  # обязательно для gradient checkpointing
```

### 5.2.4. Загрузка токенизатора

```python
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
```

---

## 5.3. Создание LoraConfig и применение `peft_config`

### 5.3.1. Конфигурация LoRA

```python
from peft import LoraConfig

LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj"]  # для Llama/Qwen/Mistral

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=TARGET_MODULES,
)
```

**Примечание:** названия целевых слоёв зависят от архитектуры модели. Для GPT-2 используйте `["c_attn", "c_proj"]`, для RoBERTa — `["query", "key", "value", "output"]`.

### 5.3.2. Два способа применения LoRA

#### Способ А: через `peft_config` в SFTTrainer (рекомендуется)

```python
trainer = SFTTrainer(
    model=model,
    peft_config=lora_config,   # SFTTrainer сам применит LoRA
    ...
)
```

#### Способ Б: ручное применение через `get_peft_model` (для Trainer)

```python
from peft import get_peft_model

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
```

> **При использовании SFTTrainer с `peft_config`** вызов `get_peft_model` не нужен — тренер сделает это автоматически. Это упрощает код и снижает риск ошибок.

---



## 5.4. Настройка SFTTrainer и SFTConfig (актуальный подход)

Современный `SFTTrainer` из библиотеки `trl` — это высокоуровневый класс, который автоматизирует большинство шагов тонкой настройки моделей на диалоговых данных. Он самостоятельно применяет чат-шаблон, токенизирует, маскирует промпты и, при необходимости, упаковывает короткие примеры. В этом разделе мы подробно разберём, как правильно настроить `SFTTrainer` и его конфигурацию `SFTConfig`, уделив особое внимание новым параметрам, внутреннему устройству и совместимости с распределённым обучением.

---

### 5.4.1. Подготовка датасета в формате `messages`

`SFTTrainer` ожидает, что каждый пример датасета содержит поле с диалогом в формате списка сообщений. Каждое сообщение — это словарь с ключами `"role"` и `"content"`. Роли могут быть `"system"`, `"user"`, `"assistant"`. Этот универсальный формат поддерживается большинством современных моделей.

```python
from datasets import Dataset

def format_conversation(example):
    return {
        "messages": [
            {"role": "user", "content": example["instruction"]},
            {"role": "assistant", "content": example["output"]},
        ]
    }

train_dataset = Dataset.from_list(train_data).map(format_conversation)
eval_dataset = Dataset.from_list(eval_data).map(format_conversation)
```

**Почему это важно?**  
`SFTTrainer` использует это поле для применения `tokenizer.apply_chat_template()`, который преобразует структурированный диалог в строку с правильными специальными токенами для конкретной модели. Это гарантирует, что модель правильно интерпретирует роли и границы сообщений.

---

### 5.4.2. Конфигурация SFTConfig (расширенная)

`SFTConfig` — наследник `TrainingArguments` из `transformers`, дополненный параметрами для supervised fine-tuning. Ниже приведён полный пример с подробными комментариями:

```python
from trl import SFTConfig

# Расчёт количества шагов для warmup
total_steps = (len(train_dataset) // (TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS)) * NUM_EPOCHS
warmup_steps = int(0.05 * total_steps)

training_args = SFTConfig(
    # ---------- ОБЩИЕ ПАРАМЕТРЫ ОБУЧЕНИЯ ----------
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    
    learning_rate=LEARNING_RATE,          # 2e-4 для LoRA, 5e-6 для Full FT
    optim="adamw_torch",                  # или "paged_adamw_32bit" для экономии VRAM
    warmup_ratio=0.05,                    # или warmup_steps
    lr_scheduler_type="cosine",
    weight_decay=0.01,
    max_grad_norm=1.0,
    
    fp16=USE_FP16,
    bf16=USE_BF16,
    gradient_checkpointing=True,          # экономия памяти ценой скорости
    
    # ---------- СПЕЦИФИЧНЫЕ ДЛЯ SFT ----------
    dataset_text_field="messages",        # поле с диалогами
    max_length=MAX_LENGTH,                # 256–2048
    packing=False,                        # объединение коротких примеров
    completion_only_loss=True,            # loss только на ответах ассистента
    loss_type="nll",                      # "nll" | "dft" | "chunked_nll"
    
    # ---------- ЛОГИРОВАНИЕ И ВАЛИДАЦИЯ ----------
    logging_steps=1,
    logging_first_step=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="wandb",                    # или "none"
    remove_unused_columns=False,          # КРИТИЧЕСКИ ВАЖНО: оставить False
    seed=SEED,
)
```

#### Таблица ключевых параметров SFTConfig

| Параметр | Тип | Рекомендации и пояснения |
| :--- | :--- | :--- |
| **`dataset_text_field`** | `str` | Имя поля в датасете, содержащего диалог (обычно `"messages"`). `SFTTrainer` прочитает это поле и применит к нему чат-шаблон. **Не токенизируйте датасет заранее!** |
| **`max_length`** | `int` | Максимальная длина **всей последовательности** (промпт + ответ) в токенах. Примеры длиннее обрезаются. Выбирайте под свою задачу. |
| **`packing`** | `bool` | Если `True`, несколько коротких примеров объединяются в одну последовательность, разделённые `eos_token`. Повышает эффективность GPU (меньше паддинга). **Включайте, если средняя длина примера < 50% от `max_length`. При включении обязательно используйте `loss_type="chunked_nll"`.** |
| **`completion_only_loss`** | `bool` | Если `True` (рекомендуется), loss вычисляется **только на токенах ответа ассистента** (токены промпта помечаются `-100` и игнорируются). Это стандарт для SFT. |
| **`loss_type`** | `str` | **Новый параметр.** Определяет способ вычисления loss. Подробно разобран ниже. |
| **`remove_unused_columns`** | `bool` | **Всегда устанавливайте `False`.** Иначе `SFTTrainer` удалит колонку `"messages"` до её использования. |

---

#### Глубокое пояснение `loss_type`: как выбор влияет на обучение

В последних версиях `trl` появилась возможность выбирать тип функции потерь. Это не просто деталь реализации — правильный выбор может существенно повлиять на качество и стабильность обучения.

| `loss_type` | Описание | Когда использовать |
| :--- | :--- | :--- |
| **`"nll"`** | Стандартная кросс-энтропия (Negative Log-Likelihood). Loss усредняется по всем токенам ответа ассистента. Это классический подход, используемый в большинстве SFT-экспериментов. | **По умолчанию для большинства задач.** Работает везде, где `packing=False`. |
| **`"dft"`** | Decision Fine-Tuning — вариант, предложенный в недавних работах для задач, где модель должна **выбрать один из нескольких вариантов** (например, классификация, ранжирование). Он модифицирует веса так, чтобы модель лучше различала правильные и неправильные ответы. | Задачи с выбором (multiple-choice, ранжирование ответов). Может дать прирост точности, но требует осторожной настройки LR. |
| **`"chunked_nll"`** | Специальная версия NLL, которая правильно обрабатывает **упакованные последовательности (`packing=True`)**. В этом режиме loss вычисляется отдельно для каждого примера внутри пакета, игнорируя токены-разделители (`eos_token`), вставленные между примерами. | **Обязателен, если `packing=True`.** Если оставить `"nll"` при включённом packing, модель будет учиться предсказывать разделители и границы примеров, что испортит качество. |

> **Важно:** если вы включаете `packing=True`, но забываете установить `loss_type="chunked_nll"`, loss будет считаться по всей последовательности, включая `eos_token` между примерами. Это приведёт к тому, что модель будет штрафоваться за неправильное предсказание разделителей, что искажает градиенты и ухудшает сходимость.

---

### 5.4.3. Внутренний конвейер обработки данных в SFTTrainer

Чтобы понять, как `SFTTrainer` работает с полем `dataset_text_field`, полезно заглянуть внутрь процесса создания батча:

```
1. Берём пример из датасета (сырой, с полем "messages")
   ↓
2. tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
   → строка, например: "<|im_start|>user\nПривет<|im_end|><|im_start|>assistant\nHello<|im_end|>"
   ↓
3. Токенизируем строку (tokenizer() → input_ids, attention_mask)
   ↓
4. Если completion_only_loss=True:
      - Находим позицию, где начинается ответ ассистента (по специальным токенам или по смещению)
      - Устанавливаем labels = -100 для всех токенов до начала ответа
      - Для токенов ответа labels = input_ids (или сдвинутые для causal LM)
   ↓
5. Если packing=True:
      - Накопливаем несколько примеров, пока общая длина не превысит max_length
      - Вставляем eos_token между примерами (чтобы модель понимала границы)
      - Создаём единую последовательность и единый labels (с -100 на промптах каждого примера)
   ↓
6. Возвращаем batch: {"input_ids": ..., "attention_mask": ..., "labels": ...}
```

**Ключевой вывод:** вы **не должны** предварительно токенизировать датасет. `SFTTrainer` делает это «на лету» во время создания батчей. Это позволяет динамически применять `packing` и корректно маскировать промпты, что было бы сложно при предварительной токенизации.

---

### 5.4.4. Создание SFTTrainer

После подготовки конфигурации создаём экземпляр `SFTTrainer`:

```python
from trl import SFTTrainer
from transformers import EarlyStoppingCallback

trainer = SFTTrainer(
    model=model,                         # базовая модель (может быть уже квантована)
    args=training_args,                  # SFTConfig
    train_dataset=train_dataset,         # сырой датасет (НЕ токенизированный!)
    eval_dataset=eval_dataset,           # сырой датасет для валидации
    processing_class=tokenizer,          # токенизатор (ключевой параметр!)
    peft_config=lora_config,             # LoRA конфиг – SFTTrainer сам применит get_peft_model
    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=EARLY_STOPPING_PATIENCE,
            early_stopping_threshold=EARLY_STOPPING_THRESHOLD,
        )
    ],
    # compute_metrics=compute_metrics,   # опционально
)
```

#### Важные моменты при использовании SFTTrainer

1. **`processing_class`** – обязательный параметр (в новых версиях TRL вместо старого `tokenizer`). Передавайте объект токенизатора.

2. **Датасет должен быть сырым** – не токенизируйте его заранее. Иначе `SFTTrainer` не сможет применить `packing` и корректно вычислить `completion_only_loss`.

3. **`peft_config`** – если вы передаёте этот параметр, **не нужно** предварительно вызывать `get_peft_model`. `SFTTrainer` сделает это автоматически. Это гарантирует правильный порядок операций (сначала квантование, потом LoRA).

4. **Совместимость с DeepSpeed (важно!)**
   - **DeepSpeed ZeRO-3 + PEFT:** Если вы используете DeepSpeed с `peft_config`, возможны проблемы при сохранении адаптера (ошибка сериализации). Рекомендуемый обходной путь: **примените LoRA вручную до инициализации DeepSpeed** и передайте модель в `SFTTrainer` без `peft_config`:
     ```python
     model = AutoModelForCausalLM.from_pretrained(...)
     model = get_peft_model(model, lora_config)   # сначала LoRA
     # затем передаём модель в Trainer без peft_config
     trainer = SFTTrainer(model=model, args=training_args, ...)
     ```
   - **DeepSpeed + QLoRA (4-bit) несовместимы.** 4-битное квантование и шардинг параметров ZeRO-3 конфликтуют. Если вам нужно распределённое обучение, используйте LoRA **без квантования** или отключите DeepSpeed и используйте обычный DataParallel.
   - Официальный гайд Hugging Face: [DeepSpeed with PEFT](https://huggingface.co/docs/peft/en/developer_guides/deepspeed)

5. **`compute_metrics`** – если вы хотите считать BLEU, ROUGE или точность на валидации, передайте функцию, которая принимает `(eval_prediction)` и возвращает словарь метрик.

---

### 5.4.5. Альтернатива: использование стандартного Trainer

Если вам нужен полный контроль над токенизацией, можно использовать стандартный `Trainer` из `transformers`. В этом случае вы самостоятельно:

- Токенизируете датасет, создавая колонки `input_ids`, `attention_mask`, `labels`.
- Применяете `get_peft_model` вручную.
- Передаёте токенизированные датасеты в `Trainer`.

```python
from transformers import Trainer, TrainingArguments

# Токенизация вручную (с маскировкой промптов)
def tokenize_function(examples):
    # Здесь нужно самостоятельно применить chat_template и маскировку
    ...

tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_eval = eval_dataset.map(tokenize_function, batched=True)

# Применение LoRA вручную
model = get_peft_model(model, lora_config)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    ...
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)
```

**Вывод:** для большинства задач `SFTTrainer` с `SFTConfig` проще и надёжнее. Используйте стандартный `Trainer` только если вам нужна специфическая логика токенизации, которую `SFTTrainer` не поддерживает.

---

### 5.4.6. Итоговый чек-лист для настройки SFTTrainer

Перед запуском обучения проверьте следующее:

- [ ] Датасет содержит поле `"messages"` (или указанное в `dataset_text_field`).
- [ ] В `SFTConfig` установлены:
  - `remove_unused_columns=False` (обязательно).
  - `completion_only_loss=True`.
  - `max_length` соответствует вашей задаче.
  - `packing=True` только если средняя длина примеров мала, и тогда `loss_type="chunked_nll"`.
- [ ] В `SFTTrainer` передан `processing_class=tokenizer`.
- [ ] Датасет передан **в сыром виде** (не токенизированный).
- [ ] `peft_config` передан в `SFTTrainer` (если не применяли LoRA вручную).
- [ ] Если используется DeepSpeed ZeRO-3, `peft_config` применён вручную до инициализации тренера, а в `SFTTrainer` параметр `peft_config` не передан.
- [ ] Для QLoRA используется `optim="paged_adamw_32bit"`.



## 5.5. Запуск обучения и логирование (WandB)

### 5.5.1. Инициализация WandB (опционально)

```python
import wandb

wandb.init(
    project="llm_finetuning",
    name="lora_r16_lr2e4",
    config={
        "model": MODEL_NAME,
        "lora_r": LORA_R,
        "lora_alpha": LORA_ALPHA,
        "learning_rate": LEARNING_RATE,
        "num_epochs": NUM_EPOCHS,
        "batch_size": TRAIN_BATCH_SIZE,
        "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
        "packing": training_args.packing,
    }
)
```

Если вы не хотите использовать WandB, установите `report_to="none"` в `SFTConfig`.

### 5.5.2. Запуск обучения

```python
print("=" * 70)
print("НАЧАЛО LoRA SFT TRAINING")
print("=" * 70)

train_result = trainer.train()

print("=" * 70)
print("TRAINING FINISHED")
print("=" * 70)
print(train_result)
```

### 5.5.3. Логирование метрик

`SFTTrainer` автоматически логирует:
- `train_loss` (на каждом шаге)
- `eval_loss` (при каждой оценке)
- `learning_rate`
- `epoch`

Если вы добавили `compute_metrics`, метрики также будут логироваться.

---

## 5.6. Сохранение адаптера

```python
print("=" * 70)
print("СОХРАНЕНИЕ LoRA ADAPTER")
print("=" * 70)

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print(f"LoRA adapter сохранён в: {OUTPUT_DIR}")

# Проверка файлов
for filename in sorted(os.listdir(OUTPUT_DIR)):
    path = os.path.join(OUTPUT_DIR, filename)
    if os.path.isfile(path):
        size_mb = os.path.getsize(path) / 1024**2
        print(f"{filename:40s}{size_mb:10.2f} MB")
```

**Что сохраняется:**

| Файл | Размер | Описание |
| :--- | :--- | :--- |
| `adapter_model.safetensors` | ~8–50 MB | Веса матриц A и B |
| `adapter_config.json` | <1 KB | Конфигурация LoRA |
| `tokenizer.json` | ~1–5 MB | Токенизатор |
| `tokenizer_config.json` | <1 KB | Конфигурация токенизатора |

---

## 5.7. Инференс с адаптером

### Загрузка адаптера

```python
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

# Загрузка базовой модели
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=DTYPE,
    device_map="auto",
)

# Загрузка адаптера
model = PeftModel.from_pretrained(
    base_model,
    OUTPUT_DIR,  # путь к сохранённому адаптеру
)

tokenizer = AutoTokenizer.from_pretrained(OUTPUT_DIR)
```

### Функция инференса

```python
def generate_response(instruction, max_new_tokens=64, temperature=0.2):
    model.eval()
    messages = [{"role": "user", "content": instruction}]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(text, return_tensors="pt")
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=(temperature > 0),
            temperature=temperature,
            top_p=0.9,
            repetition_penalty=1.05,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    generated_tokens = outputs[0, inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

# Пример использования
test_instruction = "Переведи на английский: Привет, как дела?"
print(generate_response(test_instruction))
```

---

## 5.8. Слияние (merge) адаптера с базовой моделью

### Когда использовать merge?

1. **Ускорение инференса.** После слияния модель работает с той же скоростью, что и базовая.
2. **Упрощение деплоя.** Не нужно загружать два файла (базовая модель + адаптер).
3. **Совместимость с некоторыми средами выполнения**, где динамическая загрузка адаптеров не поддерживается.

### Когда НЕ использовать merge?

1. **Несколько адаптеров для разных задач.** Если вы переключаетесь между задачами, храните адаптеры отдельно.
2. **Качество критичнее скорости.** Иногда работа через адаптер даёт чуть лучшее качество (особенно при использовании квантования).

### Почему merge может ухудшить качество?

При слиянии мы вычисляем `W = W0 + BA` и заменяем исходные веса. Это математически эквивалентно, но при использовании квантования или специфических оптимизаторов (например, paged optimizers) может возникнуть потеря точности из-за округления. Для большинства случаев разница незначительна, но тестируйте оба подхода.

### Выполнение merge

```python
# Слияние адаптера с базовой моделью
model = model.merge_and_unload()  # возвращает обычную модель (не PeftModel)

# Сохранение слитой модели
model.save_pretrained("./merged_model")
tokenizer.save_pretrained("./merged_model")
```

**Сравнение подходов:**

| Способ | Скорость | Качество | Гибкость |
| :--- | :--- | :--- | :--- |
| **Через адаптер** | Базовая + небольшой оверхед | Лучшее | Можно переключать |
| **Слияние** | Как у базовой модели | Может чуть ухудшиться | Фиксированный |

---

## 5.9. DeepSpeed: конфигурация для ZeRO-3

### Зачем нужен DeepSpeed?

DeepSpeed — библиотека от Microsoft для оптимизации обучения больших моделей. ZeRO-3 (Zero Redundancy Optimizer stage 3) шардирует параметры, градиенты и состояния оптимизатора между GPU, позволяя обучать модели, которые не влезают в память одного GPU.

### Пример конфигурации `deepspeed_config.json`

```json
{
  "train_batch_size": 8,
  "gradient_accumulation_steps": 4,
  "zero_optimization": {
    "stage": 3,
    "offload_optimizer": {
      "device": "cpu",
      "pin_memory": true
    },
    "offload_param": {
      "device": "cpu",
      "pin_memory": true
    },
    "overlap_comm": true,
    "contiguous_gradients": true,
    "sub_group_size": 1e9,
    "reduce_bucket_size": "auto",
    "stage3_prefetch_bucket_size": "auto",
    "stage3_param_persistence_threshold": "auto"
  },
  "fp16": {
    "enabled": true,
    "loss_scale": 0,
    "initial_scale_power": 16,
    "loss_scale_window": 1000,
    "hysteresis": 2,
    "min_loss_scale": 1
  },
  "bf16": {
    "enabled": false
  },
  "optimizer": {
    "type": "AdamW",
    "params": {
      "lr": "auto",
      "betas": "auto",
      "eps": "auto",
      "weight_decay": "auto"
    }
  },
  "scheduler": {
    "type": "WarmupLR",
    "params": {
      "warmup_min_lr": "auto",
      "warmup_max_lr": "auto",
      "warmup_num_steps": "auto"
    }
  },
  "communication_data_type": "fp16",
  "gradient_clipping": "auto",
  "steps_per_print": 100
}
```

### Использование с SFTTrainer

```python
training_args = SFTConfig(
    deepspeed="deepspeed_config.json",   # путь к конфигу
    # остальные параметры
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    ...
)
```

### Запуск через командную строку

```bash
deepspeed --num_gpus=4 train.py \
    --deepspeed deepspeed_config.json \
    --model_name meta-llama/Llama-2-7b \
    --lora_r 16 \
    --learning_rate 2e-4
```

### Примечания по совместимости

- **DeepSpeed + PEFT** работают вместе, но убедитесь, что используете актуальные версии.
- **QLoRA + DeepSpeed** может быть нестабильным — рекомендуется использовать LoRA без квантования при распределённом обучении.
- **ZeRO-3** лучше всего работает с моделями ≥13B. Для меньших моделей оверхед может быть выше, чем выгода.
- Если вы используете `gradient_checkpointing`, укажите это в конфигурации и в `SFTConfig` (рекомендуется).




## 5.10. Полный практический пример: LoRA на TinyLlama с SFTTrainer

Ниже приведён **полный рабочий код** для дообучения TinyLlama 1.1B с использованием LoRA и `SFTTrainer`. Этот код объединяет все шаги, описанные выше, и может быть скопирован и запущен в Google Colab или локально.

**Что демонстрирует этот пример:**
- Загрузка базовой модели (TinyLlama 1.1B)
- Настройка LoRA (`r=16`, `alpha=32`, `target_modules=["q_proj","k_proj","v_proj","o_proj"]`)
- Использование современного `SFTTrainer` с `SFTConfig`
- Форматирование датасета в формат `prompt` / `completion`
- Обучение с Early Stopping
- Сохранение адаптера
- Оценка качества (BLEU)
- Инференс на новых примерах

```python
# ================================================================
# ЧИСТЫЙ LoRA FINE-TUNING С SFTTrainer
#
# TinyLlama/TinyLlama-1.1B-Chat-v1.0
#
# Используется:
# - обычная FP16 модель
# - PEFT LoRA
# - TRL SFTTrainer
# - SFTConfig
# ================================================================

# ================================================================
# 1. УСТАНОВКА
# ================================================================
!pip install -q -U transformers peft accelerate datasets trl evaluate sacrebleu matplotlib "numpy<2.1"

# ================================================================
# 2. ИМПОРТЫ
# ================================================================
import os
import random
import numpy as np
import torch
import matplotlib.pyplot as plt

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    EarlyStoppingCallback,
)
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
import evaluate

# ================================================================
# 3. ФИКСАЦИЯ SEED
# ================================================================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ================================================================
# 4. ПРОВЕРКА ОКРУЖЕНИЯ
# ================================================================
print("=" * 70)
print("ПРОВЕРКА ОКРУЖЕНИЯ")
print("=" * 70)
print("PyTorch:", torch.__version__)
print("CUDA доступна:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print("VRAM:", round(vram_gb, 2), "GB")
    if torch.cuda.is_bf16_supported():
        DTYPE = torch.bfloat16
        USE_BF16 = True
        USE_FP16 = False
        print("Dtype: torch.bfloat16")
    else:
        DTYPE = torch.float16
        USE_BF16 = False
        USE_FP16 = True
        print("Dtype: torch.float16")
else:
    DTYPE = torch.float32
    USE_BF16 = False
    USE_FP16 = False
    print("Dtype: torch.float32")
print("=" * 70)

# ================================================================
# 5. КОНФИГУРАЦИЯ
# ================================================================
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
OUTPUT_DIR = "./tinyllama_1.1b_lora_translation"
MAX_LENGTH = 256

# LoRA
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj"]

# TRAINING
NUM_EPOCHS = 20
LEARNING_RATE = 1e-4
TRAIN_BATCH_SIZE = 2
EVAL_BATCH_SIZE = 2
GRADIENT_ACCUMULATION_STEPS = 4
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.05
MAX_GRAD_NORM = 1.0

# EARLY STOPPING
EARLY_STOPPING_PATIENCE = 3
EARLY_STOPPING_THRESHOLD = 0.001

# ================================================================
# 6. DATASET
# ================================================================
train_data = [
    {"instruction": "Переведи на английский: Привет, как дела?", "output": "Hello, how are you?"},
    {"instruction": "Переведи на английский: Сегодня отличная погода.", "output": "The weather is great today."},
    {"instruction": "Переведи на английский: Я люблю программирование.", "output": "I love programming."},
    {"instruction": "Переведи на английский: Который час?", "output": "What time is it?"},
    {"instruction": "Переведи на английский: Это очень интересно.", "output": "This is very interesting."},
    {"instruction": "Переведи на английский: Доброе утро!", "output": "Good morning!"},
    {"instruction": "Переведи на английский: Спокойной ночи.", "output": "Good night."},
    {"instruction": "Переведи на английский: Как тебя зовут?", "output": "What is your name?"},
]
eval_data = [
    {"instruction": "Переведи на английский: Я хочу пить.", "output": "I am thirsty."},
    {"instruction": "Переведи на английский: Где находится библиотека?", "output": "Where is the library?"},
]

train_dataset = Dataset.from_list(train_data)
eval_dataset = Dataset.from_list(eval_data)

print("\n" + "=" * 70)
print("DATASET")
print("=" * 70)
print("Train:", len(train_dataset))
print("Eval :", len(eval_dataset))
print("\nПример:")
print(train_dataset[0])
print("=" * 70)

# ================================================================
# 7. TOKENIZER
# ================================================================
print("\n" + "=" * 70)
print("ЗАГРУЗКА TOKENIZER")
print("=" * 70)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("Vocab size:", tokenizer.vocab_size)
print("PAD token :", tokenizer.pad_token)
print("EOS token :", tokenizer.eos_token)
print("\nChat template:")
print(tokenizer.chat_template)
print("=" * 70)

# ================================================================
# 8. ФОРМИРОВАНИЕ CONVERSATIONAL DATASET
# ================================================================
def convert_to_prompt_completion(example):
    return {
        "prompt": [{"role": "user", "content": example["instruction"]}],
        "completion": [{"role": "assistant", "content": example["output"]}],
    }

train_sft_dataset = train_dataset.map(
    convert_to_prompt_completion,
    remove_columns=train_dataset.column_names,
)
eval_sft_dataset = eval_dataset.map(
    convert_to_prompt_completion,
    remove_columns=eval_dataset.column_names,
)

print("\n" + "=" * 70)
print("SFT DATASET")
print("=" * 70)
print("Train:", len(train_sft_dataset))
print("Eval :", len(eval_sft_dataset))
print("\nПример:")
print(train_sft_dataset[0])
print("=" * 70)

# ================================================================
# 9. ПРОВЕРКА CHAT TEMPLATE
# ================================================================
print("\n" + "=" * 70)
print("ПРОВЕРКА CHAT TEMPLATE")
print("=" * 70)
test_messages = [
    {"role": "user", "content": "Переведи на английский: Привет!"},
    {"role": "assistant", "content": "Hello!"},
]
test_text = tokenizer.apply_chat_template(
    test_messages,
    tokenize=False,
    add_generation_prompt=False,
)
print(test_text)
print("=" * 70)

# ================================================================
# 10. ЗАГРУЗКА ОБЫЧНОЙ FP16 МОДЕЛИ
# ================================================================
print("\n" + "=" * 70)
print("ЗАГРУЗКА ОБЫЧНОЙ FP16 МОДЕЛИ")
print("=" * 70)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=DTYPE,
    trust_remote_code=True,
)
model.config.use_cache = False

# ================================================================
# 11. LoRA CONFIG
# ================================================================
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=TARGET_MODULES,
)

print("\n" + "=" * 70)
print("LoRA CONFIG")
print("=" * 70)
print("r:", LORA_R)
print("alpha:", LORA_ALPHA)
print("dropout:", LORA_DROPOUT)
print("target_modules:", TARGET_MODULES)
print("=" * 70)

# ================================================================
# 12. SFT CONFIG
# ================================================================
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    max_grad_norm=MAX_GRAD_NORM,
    lr_scheduler_type="cosine",
    warmup_ratio=WARMUP_RATIO,
    optim="adamw_torch",
    fp16=USE_FP16,
    bf16=USE_BF16,
    gradient_checkpointing=True,
    max_length=MAX_LENGTH,
    completion_only_loss=True,
    packing=False,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    logging_strategy="steps",
    logging_steps=1,
    logging_first_step=True,
    dataloader_num_workers=0,
    seed=SEED,
    data_seed=SEED,
    report_to="none",
    remove_unused_columns=False,
    label_names=["labels"],
)

# ================================================================
# 13. SFTTrainer
# ================================================================
print("\n" + "=" * 70)
print("СОЗДАНИЕ SFTTrainer")
print("=" * 70)
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_sft_dataset,
    eval_dataset=eval_sft_dataset,
    processing_class=tokenizer,          # современный параметр вместо tokenizer
    peft_config=lora_config,             # SFTTrainer сам применит LoRA
    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=EARLY_STOPPING_PATIENCE,
            early_stopping_threshold=EARLY_STOPPING_THRESHOLD,
        )
    ],
)

# ================================================================
# 14. ПРОВЕРКА TRAINABLE PARAMETERS
# ================================================================
print("\n" + "=" * 70)
print("LoRA TRAINABLE PARAMETERS")
print("=" * 70)
trainer.model.print_trainable_parameters()
print("=" * 70)

# ================================================================
# 15. ПРОВЕРКА TRAINABLE TENSORS
# ================================================================
trainable_tensors = []
for name, parameter in trainer.model.named_parameters():
    if parameter.requires_grad:
        trainable_tensors.append((name, parameter.numel()))
total_trainable = sum(count for _, count in trainable_tensors)
total_parameters = sum(parameter.numel() for parameter in trainer.model.parameters())
trainable_percent = 100.0 * total_trainable / total_parameters

print("\n" + "=" * 70)
print("TRAINABLE PARAMETER CHECK")
print("=" * 70)
print("Trainable parameters:", f"{total_trainable:,}")
print("All parameters:", f"{total_parameters:,}")
print("Trainable %:", f"{trainable_percent:.4f}%")
print("Trainable tensors:", len(trainable_tensors))
print("=" * 70)

# ================================================================
# 16. TRAINING
# ================================================================
print("\n" + "=" * 70)
print("НАЧАЛО LoRA SFT TRAINING")
print("=" * 70)
train_result = trainer.train()
print("\n" + "=" * 70)
print("TRAINING FINISHED")
print("=" * 70)
print(train_result)

# ================================================================
# 17. СОХРАНЕНИЕ LoRA ADAPTER
# ================================================================
print("\n" + "=" * 70)
print("СОХРАНЕНИЕ LoRA ADAPTER")
print("=" * 70)
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("\nLoRA adapter сохранён:", OUTPUT_DIR)
print("=" * 70)

# ================================================================
# 18. ПРОВЕРКА ФАЙЛОВ
# ================================================================
print("\n" + "=" * 70)
print("СОДЕРЖИМОЕ OUTPUT DIRECTORY")
print("=" * 70)
if os.path.exists(OUTPUT_DIR):
    for filename in sorted(os.listdir(OUTPUT_DIR)):
        path = os.path.join(OUTPUT_DIR, filename)
        if os.path.isfile(path):
            size_mb = os.path.getsize(path) / 1024**2
            print(f"{filename:40s}{size_mb:10.2f} MB")
print("=" * 70)

# ================================================================
# 19. ГРАФИК TRAINING
# ================================================================
print("\n" + "=" * 70)
print("ГРАФИК ОБУЧЕНИЯ")
print("=" * 70)
log_history = trainer.state.log_history
train_steps, train_losses = [], []
eval_epochs, eval_losses = [], []
for log in log_history:
    if "loss" in log and "step" in log:
        train_steps.append(log["step"])
        train_losses.append(log["loss"])
    if "eval_loss" in log and "epoch" in log:
        eval_epochs.append(log["epoch"])
        eval_losses.append(log["eval_loss"])

plt.figure(figsize=(10, 6))
if train_losses:
    plt.plot(train_steps, train_losses, marker="o", label="Train Loss")
if eval_losses:
    plt.plot(eval_epochs, eval_losses, marker="s", label="Eval Loss")
plt.xlabel("Step / Epoch")
plt.ylabel("Loss")
plt.title("LoRA SFT Training - TinyLlama 1.1B")
plt.grid(True)
plt.legend()
plt.tight_layout()
plot_path = os.path.join(OUTPUT_DIR, "training_plot.png")
plt.savefig(plot_path, dpi=150)
plt.show()
print("График сохранён:", plot_path)
print("=" * 70)

# ================================================================
# 20. BLEU
# ================================================================
print("\n" + "=" * 70)
print("ЗАГРУЗКА SACREBLEU")
print("=" * 70)
bleu_metric = evaluate.load("sacrebleu")
print("SacreBLEU загружен.")
print("=" * 70)

# ================================================================
# 21. GENERATION FUNCTION
# ================================================================
def generate_translation(instruction, max_new_tokens=64, temperature=0.2):
    trainer.model.eval()
    messages = [{"role": "user", "content": instruction}]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(text, return_tensors="pt")
    device = next(trainer.model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = trainer.model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=(temperature > 0),
            temperature=temperature,
            top_p=0.9,
            repetition_penalty=1.05,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    generated_tokens = outputs[0, inputs["input_ids"].shape[1]:]
    answer = tokenizer.decode(generated_tokens, skip_special_tokens=True)
    return answer.strip()

# ================================================================
# 22. ФИНАЛЬНАЯ BLEU ОЦЕНКА
# ================================================================
print("\n" + "=" * 70)
print("ФИНАЛЬНАЯ ОЦЕНКА BLEU")
print("=" * 70)
references = [example["output"] for example in eval_data]
predictions = []
for example in eval_data:
    pred = generate_translation(example["instruction"])
    predictions.append(pred)
bleu_result = bleu_metric.compute(
    predictions=predictions,
    references=[[ref] for ref in references],
)
bleu_score = bleu_result["score"]
print(f"\nBLEU на validation: {bleu_score:.2f}\n")
for i, (pred, ref) in enumerate(zip(predictions, references)):
    print(f"Пример {i + 1}:")
    print("  Instruction:", eval_data[i]["instruction"])
    print("  Reference  :", ref)
    print("  Prediction  :", pred)
    print()
print("=" * 70)

# ================================================================
# 23. TEST TRAIN EXAMPLE
# ================================================================
print("\n" + "=" * 70)
print("TEST: TRAIN EXAMPLE")
print("=" * 70)
test_instruction = "Переведи на английский: Привет, как дела?"
answer = generate_translation(test_instruction)
print("\nINPUT:")
print(test_instruction)
print("\nOUTPUT:")
print(answer)
print("=" * 70)

# ================================================================
# 24. TEST NEW EXAMPLES
# ================================================================
test_examples = [
    "Переведи на английский: Я хочу пить.",
    "Переведи на английский: Где находится библиотека?",
    "Переведи на английский: Я люблю Python.",
    "Переведи на английский: До свидания!",
    "Переведи на английский: Как тебя зовут?",
]
print("\n" + "=" * 70)
print("TEST: NEW EXAMPLES")
print("=" * 70)
for instruction in test_examples:
    answer = generate_translation(instruction)
    print("\nINPUT :", instruction)
    print("OUTPUT:", answer)
print("=" * 70)

# ================================================================
# 25. ФИНАЛЬНАЯ ИНФОРМАЦИЯ
# ================================================================
print("\n" + "=" * 70)
print("ФИНАЛЬНЫЙ РЕЗУЛЬТАТ")
print("=" * 70)
print("\nМетод:\n  LoRA + SFTTrainer")
print("\nМодель:\n", MODEL_NAME)
print("\nQuantization:\n  НЕТ")
print("\nQLoRA:\n  НЕТ")
print("\n4-bit:\n  НЕТ")
print("\nBitsAndBytes:\n  НЕТ")
print("\nPrecision:")
if USE_FP16:
    print("  FP16")
elif USE_BF16:
    print("  BF16")
else:
    print("  FP32")
print("\nLoRA rank:", LORA_R)
print("LoRA alpha:", LORA_ALPHA)
print("LoRA dropout:", LORA_DROPOUT)
print("\nTrain examples:", len(train_data))
print("Eval examples :", len(eval_data))
print("\nOutput:", OUTPUT_DIR)
print("=" * 70)
```

---

### Что демонстрирует этот пример

| Этап | Что показывает код |
| :--- | :--- |
| **1. Установка** | Установка всех необходимых библиотек |
| **2. Окружение** | Автоматическое определение GPU и типа данных (FP16/BF16) |
| **3. Датасет** | Формирование датасета в формате `instruction` / `output` |
| **4. Токенизатор** | Загрузка токенизатора и установка `pad_token` |
| **5. Форматирование** | Преобразование в `prompt` / `completion` для SFTTrainer |
| **6. LoRA Config** | Настройка `r=16`, `alpha=32`, `target_modules` |
| **7. SFTConfig** | Все параметры обучения: эпохи, LR, warmup, packing |
| **8. SFTTrainer** | Создание тренера с `processing_class` и `peft_config` |
| **9. Обучение** | Запуск с Early Stopping |
| **10. Сохранение** | Сохранение адаптера (маленький файл) |
| **11. Оценка** | BLEU на валидационном наборе |
| **12. Инференс** | Генерация переводов на новых примерах |




## Итоговый чек-лист для запуска

- [ ] Установлены `transformers`, `peft`, `accelerate`, `bitsandbytes`, `trl`, `datasets`, `evaluate`.
- [ ] Загружена модель (с `quantization_config` для QLoRA или без для LoRA).
- [ ] Создан `LoraConfig` с `r`, `lora_alpha`, `target_modules`.
- [ ] Датасет преобразован в формат `messages` (conversational).
- [ ] Создан `SFTConfig` с параметрами обучения и SFT-специфичными (`dataset_text_field`, `max_length`, `packing`).
- [ ] Создан `SFTTrainer` с `model`, `args`, `train_dataset`, `eval_dataset`, `processing_class=tokenizer`, `peft_config`.
- [ ] Запущено обучение (`trainer.train()`).
- [ ] Сохранён адаптер (`trainer.save_model`).
- [ ] Проверен инференс с загруженным адаптером.
- [ ] При необходимости выполнен merge и сохранена полная модель.

**Рекомендация:** для большинства задач используйте `SFTTrainer` с `peft_config` — это самый простой и надёжный способ. Если вам нужен полный контроль, используйте стандартный `Trainer` с ручной токенизацией и `get_peft_model`.

# Лекция 4.2. LoRA и QLoRA: практическая параметрически-эффективная настройка

## Тема 6. Анализ и отладка экспериментов

**Цель:** дать системный подход к анализу и отладке экспериментов с LoRA/QLoRA. Вы научитесь диагностировать типичные ошибки, понимать, почему LoRA не бьёт бейзлайн, профилировать память и скорость, а также вести таблицу экспериментов для воспроизводимости.

---

### Введение: почему эксперименты проваливаются?

Тонкая настройка LLM — это сложный процесс, и даже опытные инженеры сталкиваются с неудачами. В этой теме мы разберём **системный подход к отладке**:

1. **Типичные ошибки** — что чаще всего идёт не так.
2. **Диагностика** — почему LoRA не улучшает качество.
3. **Профилирование** — как измерять память и скорость.
4. **Таблица экспериментов** — как вести лог для воспроизводимости.

---

## 6.1. Типичные ошибки и их решения

### 6.1.1. Переобучение (Overfitting)

| Симптом | Диагностика | Решение |
| :--- | :--- | :--- |
| **Train loss падает, eval loss растёт** | Eval loss растёт после 2–3 эпох | Уменьшить число эпох, добавить dropout, увеличить датасет |
| **Высокая точность на трейне, низкая на тесте** | Модель запомнила данные | Добавить регуляризацию, уменьшить `r`, увеличить `lora_dropout` |
| **BLEU/ROUGE на валидации не растёт** | Модель не обобщается | Проверить качество данных, размер датасета |

**Код для диагностики переобучения:**

```python
import matplotlib.pyplot as plt

def plot_training_curves(log_history):
    """Визуализирует train и eval loss для выявления переобучения."""
    train_steps, train_losses = [], []
    eval_epochs, eval_losses = [], []
    
    for log in log_history:
        if "loss" in log and "step" in log:
            train_steps.append(log["step"])
            train_losses.append(log["loss"])
        if "eval_loss" in log and "epoch" in log:
            eval_epochs.append(log["epoch"])
            eval_losses.append(log["eval_loss"])
    
    plt.figure(figsize=(10, 5))
    if train_losses:
        plt.plot(train_steps, train_losses, label="Train Loss", marker="o")
    if eval_losses:
        plt.plot(eval_epochs, eval_losses, label="Eval Loss", marker="s")
    plt.xlabel("Step / Epoch")
    plt.ylabel("Loss")
    plt.title("Training Curves")
    plt.legend()
    plt.grid(True)
    plt.show()

# Использование
plot_training_curves(trainer.state.log_history)
```

**Что искать на графике:**
- Если **eval loss растёт, а train loss падает** → **переобучение**.
- Если **оба падают** → всё идёт по плану.
- Если **оба не падают** → проблема в данных или LR.

---

### 6.1.2. Неподходящий Learning Rate (LR)

| Симптом | Диагностика | Решение |
| :--- | :--- | :--- |
| **Loss взлетает до NaN** | Слишком высокая LR | Уменьшить LR в 10 раз |
| **Loss не снижается** | Слишком низкая LR | Увеличить LR в 2–3 раза |
| **Loss колеблется** | LR слишком высок для данных | Уменьшить LR, добавить warmup |

```python
# Проверка LR в логах
for log in trainer.state.log_history:
    if "learning_rate" in log:
        print(f"Step {log.get('step', 'N/A')}: LR = {log['learning_rate']:.2e}")
```

---

### 6.1.3. Забытый `pad_token` — причина NaN loss

**Симптом:** Loss становится `NaN` после нескольких шагов.

**Причина:** при батчинге токены разной длины не выравниваются, и модель получает некорректные входные данные.

```python
# Проверка pad_token
if tokenizer.pad_token is None:
    print("❌ pad_token не установлен!")
    tokenizer.pad_token = tokenizer.eos_token
    print(f"✅ Установлен pad_token = {tokenizer.pad_token}")

# Проверка в DataCollator
from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    padding=True,
    label_pad_token_id=-100,  # игнорируем при расчёте loss
)
```

---

### 6.1.4. Случайно размороженная базовая модель

**Симптом:** тренировка медленная, память переполняется, качество не растёт.

**Причина:** если вы забыли применить `get_peft_model` или случайно разморозили базовую модель, обучаются все параметры (а не только адаптеры).

```python
# Проверка: только LoRA параметры должны быть trainable
def check_trainable_parameters(model):
    """Проверяет, что обучаются только LoRA-параметры."""
    trainable_names = []
    for name, param in model.named_parameters():
        if param.requires_grad:
            trainable_names.append(name)
    
    print(f"Всего обучаемых тензоров: {len(trainable_names)}")
    print("Первые 10 обучаемых параметров:")
    for name in trainable_names[:10]:
        print(f"  {name}")
    
    # Проверка: должны быть только параметры с "lora" в имени
    lora_params = [n for n in trainable_names if "lora" in n]
    if len(lora_params) == len(trainable_names):
        print("✅ Обучаются только LoRA-параметры")
    else:
        print("⚠️ Есть не-LoRA параметры! Проверьте get_peft_model")

# Использование
check_trainable_parameters(model)
```

---

### 6.1.5. Неправильные `target_modules`

**Симптом:** модель не адаптируется, качество не растёт.

**Причина:** вы указали неправильные названия слоёв для своей архитектуры.

```python
# Проверка доступных модулей в модели
def get_available_modules(model):
    """Выводит доступные модули для LoRA."""
    modules = set()
    for name, _ in model.named_modules():
        # Берём только линейные слои
        if "proj" in name or "attn" in name or "fc" in name:
            modules.add(name.split('.')[-1])
    print("Доступные модули:")
    for m in sorted(modules):
        print(f"  {m}")
    return modules

# Использование
available = get_available_modules(model)
```

**Таблица соответствия для популярных моделей:**

| Модель | `target_modules` |
| :--- | :--- |
| **Llama 3, Qwen, Mistral** | `["q_proj", "k_proj", "v_proj", "o_proj"]` |
| **GPT-2** | `["c_attn", "c_proj"]` |
| **RoBERTa** | `["query", "key", "value", "output"]` |
| **Phi-3** | `["q_proj", "k_proj", "v_proj", "o_proj"]` |
| **TinyLlama** | `["q_proj", "k_proj", "v_proj", "o_proj"]` |

---

## 6.2. Диагностика: почему LoRA не бьёт Baseline?

Если LoRA не улучшает качество по сравнению с промптами (Zero-shot / Few-shot), пройдите по этому чек-листу:

### Чек-лист диагностики

```python
def diagnose_lora_failure(model, tokenizer, dataset, baseline_metrics):
    """
    Комплексная диагностика, почему LoRA не бьёт Baseline.
    """
    print("=" * 70)
    print("ДИАГНОСТИКА: ПОЧЕМУ LoRA НЕ БЬЁТ BASELINE")
    print("=" * 70)
    
    issues = []
    
    # 1. Проверка данных
    print("\n1. ПРОВЕРКА ДАННЫХ:")
    print(f"   Размер датасета: {len(dataset)}")
    if len(dataset) < 500:
        issues.append("⚠️ Мало данных (<500 примеров) — риск переобучения")
    if len(dataset) < 100:
        issues.append("❌ Критически мало данных (<100 примеров)")
    
    # 2. Проверка LoRA конфигурации
    print("\n2. ПРОВЕРКА LoRA КОНФИГУРАЦИИ:")
    if hasattr(model, 'peft_config'):
        config = model.peft_config
        print(f"   r: {config.r}")
        print(f"   lora_alpha: {config.lora_alpha}")
        print(f"   target_modules: {config.target_modules}")
        print(f"   lora_dropout: {config.lora_dropout}")
        
        if config.r < 8:
            issues.append("⚠️ Очень низкий ранг (r < 8) — попробуйте r=16")
        if config.lora_alpha < 2 * config.r:
            issues.append("⚠️ lora_alpha < 2*r — попробуйте увеличить")
    
    # 3. Проверка Learning Rate
    print("\n3. ПРОВЕРКА LR:")
    # Извлекаем LR из TrainingArguments
    # (в реальном коде нужно передать training_args)
    
    # 4. Проверка Eval loss
    print("\n4. ПРОВЕРКА LOSS:")
    # Анализ логов
    
    # 5. Сравнение с Baseline
    print("\n5. СРАВНЕНИЕ С BASELINE:")
    # Вывод результатов
    
    print("\n" + "=" * 70)
    if issues:
        print("НАЙДЕНЫ ПРОБЛЕМЫ:")
        for issue in issues:
            print(f"  {issue}")
        print("\n💡 РЕКОМЕНДАЦИИ:")
        print("  1. Убедитесь, что датасет > 500 примеров")
        print("  2. Проверьте target_modules для вашей модели")
        print("  3. Для LoRA используйте LR=2e-4 (выше, чем для Full FT)")
        print("  4. Попробуйте увеличить ранг (r=16 → r=32)")
        print("  5. Проверьте форматирование данных (apply_chat_template)")
        print("  6. Убедитесь, что pad_token установлен")
    else:
        print("✅ Критических проблем не обнаружено")
    print("=" * 70)
    
    return issues
```

---

## 6.3. Профилирование памяти и скорости

### 6.3.1. Мониторинг памяти с `torch.cuda.memory_summary()`

```python
import torch

def print_memory_usage():
    """Выводит детальную информацию об использовании VRAM."""
    if not torch.cuda.is_available():
        print("CUDA не доступна")
        return
    
    print("=" * 70)
    print("CUDA MEMORY SUMMARY")
    print("=" * 70)
    
    # Текущее использование
    allocated = torch.cuda.memory_allocated() / 1024**3
    reserved = torch.cuda.memory_reserved() / 1024**3
    total = torch.cuda.get_device_properties(0).total_memory / 1024**3
    
    print(f"Выделено:   {allocated:.2f} GB")
    print(f"Зарезервировано: {reserved:.2f} GB")
    print(f"Всего VRAM: {total:.2f} GB")
    print(f"Свободно:   {total - allocated:.2f} GB")
    
    # Детальный отчёт
    print("\nДетальный отчёт:")
    print(torch.cuda.memory_summary())

# Использование
print_memory_usage()
```

### 6.3.2. Профилирование с `torch.profiler`

```python
import torch.profiler

def profile_training_step(model, dataloader, steps=5):
    """
    Профилирует шаг обучения для выявления узких мест.
    """
    with torch.profiler.profile(
        activities=[
            torch.profiler.ProfilerActivity.CPU,
            torch.profiler.ProfilerActivity.CUDA,
        ],
        schedule=torch.profiler.schedule(
            wait=1,
            warmup=1,
            active=3,
            repeat=1,
        ),
        on_trace_ready=torch.profiler.tensorboard_trace_handler('./profiler_logs'),
        record_shapes=True,
        profile_memory=True,
        with_stack=True,
    ) as prof:
        for step, batch in enumerate(dataloader):
            if step >= steps:
                break
            
            # Forward + Backward
            outputs = model(**batch)
            loss = outputs.loss
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            
            prof.step()
    
    print("✅ Профилирование завершено. Результаты в ./profiler_logs")
    return prof
```

### 6.3.3. Мониторинг в реальном времени

```python
import subprocess
import time

def monitor_gpu(interval=1):
    """
    Мониторит GPU в реальном времени с помощью nvidia-smi.
    """
    while True:
        result = subprocess.run(
            ['nvidia-smi', '--query-gpu=memory.used,memory.total,utilization.gpu',
             '--format=csv,noheader,nounits'],
            capture_output=True,
            text=True,
        )
        memory_used, memory_total, gpu_util = result.stdout.strip().split(',')
        print(f"VRAM: {memory_used.strip()}MB / {memory_total.strip()}MB  |  GPU: {gpu_util.strip()}%")
        time.sleep(interval)

# Использование (запускать в отдельном терминале)
# monitor_gpu(interval=2)
```

### 6.3.4. Сравнение памяти: LoRA vs Full FT

```python
def compare_memory_usage(model_name="meta-llama/Llama-2-7b"):
    """
    Сравнивает память для LoRA и Full FT (теоретический расчёт).
    """
    # Полная настройка (Full FT)
    # Веса: P * 2 (bf16)
    # Градиенты: P * 2
    # Adam: 2 * P * 4 (fp32)
    # Итого: ~12P байт
    
    # LoRA
    # Веса заморожены: P * 2
    # LoRA параметры (r=16): ~0.1% * P * 2
    # Градиенты только для LoRA: ~0.1% * P * 2
    # Adam только для LoRA: ~0.1% * P * 8
    # Итого: ~2P + 0.01P байт
    
    print("=" * 70)
    print("СРАВНЕНИЕ ПАМЯТИ: LoRA vs Full FT")
    print("=" * 70)
    
    # Для 7B модели
    P = 7e9
    full_ft = 12 * P / 1e9  # GB
    lora_memory = (2 * P + 0.01 * P) / 1e9  # GB
    
    print(f"Full FT: ~{full_ft:.1f} GB")
    print(f"LoRA:    ~{lora_memory:.1f} GB")
    print(f"Экономия: ~{(1 - lora_memory/full_ft) * 100:.1f}%")
    print("=" * 70)
```

---

## 6.4. Таблица экспериментов: ведите лог

### 6.4.1. Зачем вести таблицу экспериментов?

1. **Воспроизводимость** — можно повторить лучший эксперимент.
2. **Сравнение** — видно, какие параметры работают лучше.
3. **Экономия времени** — не нужно заново подбирать параметры.

### 6.4.2. Структура таблицы экспериментов

```python
import pandas as pd
from datetime import datetime

# Создание таблицы экспериментов
experiments = pd.DataFrame(columns=[
    "id",
    "date",
    "model",
    "method",           # LoRA / QLoRA / Full FT
    "r",
    "lora_alpha",
    "target_modules",
    "lora_dropout",
    "learning_rate",
    "num_epochs",
    "batch_size",
    "gradient_accumulation",
    "train_loss",
    "eval_loss",
    "bleu",
    "rouge",
    "train_time_hours",
    "gpu_memory_gb",
    "status",           # success / failed / running
    "notes",
])

# Добавление эксперимента
def add_experiment(experiments, **kwargs):
    """Добавляет новый эксперимент в таблицу."""
    new_id = len(experiments) + 1
    new_row = {
        "id": new_id,
        "date": datetime.now().strftime("%Y-%m-%d %H:%M"),
        **kwargs
    }
    experiments = pd.concat([experiments, pd.DataFrame([new_row])], ignore_index=True)
    return experiments

# Пример добавления
experiments = add_experiment(
    experiments,
    model="TinyLlama-1.1B",
    method="LoRA",
    r=16,
    lora_alpha=32,
    target_modules="q_proj,k_proj,v_proj,o_proj",
    lora_dropout=0.05,
    learning_rate=2e-4,
    num_epochs=3,
    batch_size=2,
    gradient_accumulation=4,
    train_loss=0.42,
    eval_loss=0.38,
    bleu=82.5,
    rouge=0.89,
    train_time_hours=1.5,
    gpu_memory_gb=6.2,
    status="success",
    notes="Хороший результат, можно пробовать r=32",
)

print(experiments)
```

### 6.4.3. Визуализация результатов

```python
def plot_experiment_results(experiments):
    """Визуализирует результаты экспериментов."""
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    
    # График 1: BLEU vs Rank
    axes[0].scatter(experiments['r'], experiments['bleu'], s=100)
    axes[0].set_xlabel('Rank (r)')
    axes[0].set_ylabel('BLEU')
    axes[0].set_title('BLEU vs Rank')
    axes[0].grid(True)
    
    # График 2: BLEU vs Learning Rate
    axes[1].scatter(experiments['learning_rate'], experiments['bleu'], s=100)
    axes[1].set_xlabel('Learning Rate')
    axes[1].set_ylabel('BLEU')
    axes[1].set_title('BLEU vs Learning Rate')
    axes[1].grid(True)
    
    plt.tight_layout()
    plt.show()

# Использование
plot_experiment_results(experiments)
```

### 6.4.4. Поиск лучшей конфигурации

```python
def find_best_config(experiments, metric='bleu'):
    """Находит лучшую конфигурацию по указанной метрике."""
    best = experiments.loc[experiments[metric].idxmax()]
    print("=" * 70)
    print(f"ЛУЧШАЯ КОНФИГУРАЦИЯ (по {metric})")
    print("=" * 70)
    print(f"ID: {best['id']}")
    print(f"Дата: {best['date']}")
    print(f"Модель: {best['model']}")
    print(f"Метод: {best['method']}")
    print(f"Rank (r): {best['r']}")
    print(f"Alpha: {best['lora_alpha']}")
    print(f"LR: {best['learning_rate']:.2e}")
    print(f"Эпохи: {best['num_epochs']}")
    print(f"{metric}: {best[metric]}")
    print(f"Примечания: {best.get('notes', '')}")
    print("=" * 70)
    return best

# Использование
best = find_best_config(experiments, metric='bleu')
```

---

Отличная идея! Да, в Тему 6 **стоит добавить полный пример кода**, который объединяет:

- Загрузку модели (LoRA / QLoRA с выбором)
- Настройку обучения с логированием
- Профилирование памяти и скорости
- Автоматическое логирование экспериментов в таблицу
- Сравнение с Baseline
- Визуализацию графиков

Этот код станет **универсальным шаблоном** для студентов, который они могут адаптировать под свои задачи.

---

## 6.5. Полный пример: сквозной эксперимент с LoRA/QLoRA

Ниже приведён **полный код**, который:
1. Загружает модель (LoRA или QLoRA)
2. Настраивает обучение
3. Логирует все метрики
4. Профилирует память
5. Сохраняет результаты в таблицу
6. Сравнивает с Baseline
7. Визуализирует графики

```python
# ================================================================
# ПОЛНЫЙ СКВОЗНОЙ ЭКСПЕРИМЕНТ С LoRA / QLoRA
#
# Включает:
# - Выбор метода (LoRA / QLoRA)
# - Профилирование памяти
# - Логирование экспериментов
# - Сравнение с Baseline
# - Визуализация
# ================================================================

# ================================================================
# 1. УСТАНОВКА И ИМПОРТЫ
# ================================================================
!pip install -q -U transformers peft accelerate bitsandbytes datasets trl evaluate sacrebleu matplotlib pandas wandb "numpy<2.1"

import os
import gc
import time
import json
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime
from dataclasses import dataclass, field
from typing import Optional, List, Dict, Any

import torch
import torch.cuda as cuda
import torch.profiler

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    EarlyStoppingCallback,
    BitsAndBytesConfig,
)
from peft import LoraConfig, get_peft_model, PeftModel
from trl import SFTTrainer, SFTConfig
import evaluate

# ================================================================
# 2. КОНФИГУРАЦИЯ
# ================================================================
@dataclass
class ExperimentConfig:
    """Конфигурация эксперимента."""
    # Модель
    model_name: str = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
    method: str = "lora"  # "lora" или "qlora"
    
    # LoRA
    lora_r: int = 16
    lora_alpha: int = 32
    lora_dropout: float = 0.05
    target_modules: List[str] = field(default_factory=lambda: ["q_proj", "k_proj", "v_proj", "o_proj"])
    
    # Training
    num_epochs: int = 20
    learning_rate: float = 2e-4
    train_batch_size: int = 2
    eval_batch_size: int = 2
    gradient_accumulation_steps: int = 4
    warmup_ratio: float = 0.05
    weight_decay: float = 0.01
    max_grad_norm: float = 1.0
    max_length: int = 256
    packing: bool = False
    
    # Early stopping
    early_stopping_patience: int = 3
    early_stopping_threshold: float = 0.001
    
    # System
    seed: int = 42
    output_dir: str = "./experiment_output"
    
    # Profile
    profile_steps: int = 5
    log_interval: int = 10

# ================================================================
# 3. ДАТАСЕТ
# ================================================================
def create_dataset():
    """Создаёт небольшой датасет для демонстрации."""
    train_data = [
        {"instruction": "Переведи на английский: Привет, как дела?", "output": "Hello, how are you?"},
        {"instruction": "Переведи на английский: Сегодня отличная погода.", "output": "The weather is great today."},
        {"instruction": "Переведи на английский: Я люблю программирование.", "output": "I love programming."},
        {"instruction": "Переведи на английский: Который час?", "output": "What time is it?"},
        {"instruction": "Переведи на английский: Это очень интересно.", "output": "This is very interesting."},
        {"instruction": "Переведи на английский: Доброе утро!", "output": "Good morning!"},
        {"instruction": "Переведи на английский: Спокойной ночи.", "output": "Good night."},
        {"instruction": "Переведи на английский: Как тебя зовут?", "output": "What is your name?"},
    ]
    eval_data = [
        {"instruction": "Переведи на английский: Я хочу пить.", "output": "I am thirsty."},
        {"instruction": "Переведи на английский: Где находится библиотека?", "output": "Where is the library?"},
    ]
    return Dataset.from_list(train_data), Dataset.from_list(eval_data)

# ================================================================
# 4. BASELINE (Zero-shot / Few-shot)
# ================================================================
def evaluate_baseline(model, tokenizer, eval_data, method="zero_shot"):
    """
    Оценивает Baseline (Zero-shot или Few-shot) без обучения.
    """
    print("\n" + "=" * 70)
    print(f"BASELINE: {method.upper()}")
    print("=" * 70)
    
    if method == "zero_shot":
        prompt_template = "Переведи на английский: {instruction}"
    else:
        # Few-shot с 2 примерами
        prompt_template = (
            "Пример 1: Переведи на английский: Привет, как дела? -> Hello, how are you?\n"
            "Пример 2: Переведи на английский: Сегодня отличная погода. -> The weather is great today.\n"
            "Теперь переведи: {instruction}"
        )
    
    predictions = []
    references = []
    
    for ex in eval_data:
        prompt = prompt_template.format(instruction=ex["instruction"])
        messages = [{"role": "user", "content": prompt}]
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(text, return_tensors="pt")
        inputs = {k: v.to(model.device) for k, v in inputs.items()}
        
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=64,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )
        generated = outputs[0, inputs["input_ids"].shape[1]:]
        answer = tokenizer.decode(generated, skip_special_tokens=True).strip()
        predictions.append(answer)
        references.append(ex["output"])
    
    # BLEU
    bleu_metric = evaluate.load("sacrebleu")
    bleu_score = bleu_metric.compute(
        predictions=predictions,
        references=[[ref] for ref in references]
    )["score"]
    
    print(f"BLEU: {bleu_score:.2f}")
    for i, (pred, ref) in enumerate(zip(predictions, references)):
        print(f"Пример {i+1}:")
        print(f"  Вопрос: {eval_data[i]['instruction']}")
        print(f"  Ожидалось: {ref}")
        print(f"  Получено: {pred}")
    print("=" * 70)
    
    return {"bleu": bleu_score, "predictions": predictions}

# ================================================================
# 5. ЗАГРУЗКА МОДЕЛИ (LoRA / QLoRA)
# ================================================================
def load_model_and_tokenizer(config: ExperimentConfig):
    """
    Загружает модель и токенизатор с учётом выбранного метода.
    """
    print("\n" + "=" * 70)
    print(f"ЗАГРУЗКА МОДЕЛИ: {config.method.upper()}")
    print("=" * 70)
    
    # Определяем dtype
    if torch.cuda.is_available():
        if torch.cuda.is_bf16_supported():
            dtype = torch.bfloat16
            use_bf16 = True
            use_fp16 = False
        else:
            dtype = torch.float16
            use_bf16 = False
            use_fp16 = True
    else:
        dtype = torch.float32
        use_bf16 = False
        use_fp16 = False
    
    # Токенизатор
    tokenizer = AutoTokenizer.from_pretrained(config.model_name, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"
    
    # Загрузка модели
    if config.method == "qlora":
        # QLoRA с 4-битным квантованием
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=dtype,
        )
        model = AutoModelForCausalLM.from_pretrained(
            config.model_name,
            quantization_config=bnb_config,
            device_map="auto",
            torch_dtype=dtype,
            trust_remote_code=True,
        )
    else:
        # LoRA (обычная точность)
        model = AutoModelForCausalLM.from_pretrained(
            config.model_name,
            torch_dtype=dtype,
            device_map="auto",
            trust_remote_code=True,
        )
    
    model.config.use_cache = False
    
    print(f"Метод: {config.method.upper()}")
    print(f"Dtype: {dtype}")
    print(f"Параметров: {sum(p.numel() for p in model.parameters()) / 1e9:.2f}B")
    print("=" * 70)
    
    return model, tokenizer, use_fp16, use_bf16

# ================================================================
# 6. ПРОФИЛИРОВАНИЕ
# ================================================================
def profile_memory_and_speed(model, tokenizer, dataset, config: ExperimentConfig):
    """
    Профилирует память и скорость обучения.
    """
    print("\n" + "=" * 70)
    print("ПРОФИЛИРОВАНИЕ")
    print("=" * 70)
    
    # Память до обучения
    if torch.cuda.is_available():
        allocated_before = torch.cuda.memory_allocated() / 1024**3
        print(f"VRAM до загрузки: {allocated_before:.2f} GB")
    
    # Замер скорости инференса на одном примере
    def measure_inference_speed(n_runs=10):
        times = []
        example = dataset[0]
        prompt = example["instruction"]
        messages = [{"role": "user", "content": prompt}]
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(text, return_tensors="pt")
        inputs = {k: v.to(model.device) for k, v in inputs.items()}
        
        for _ in range(n_runs):
            torch.cuda.synchronize()
            start = time.time()
            with torch.no_grad():
                model.generate(**inputs, max_new_tokens=32, pad_token_id=tokenizer.pad_token_id)
            torch.cuda.synchronize()
            times.append(time.time() - start)
        return np.mean(times)
    
    avg_time = measure_inference_speed(5)
    print(f"Среднее время инференса (32 токена): {avg_time*1000:.2f} мс")
    
    # Профилирование с torch.profiler (если включено)
    if config.profile_steps > 0:
        print(f"Запуск профилирования на {config.profile_steps} шагах...")
        # Здесь можно добавить код профилирования (см. раздел 6.3.2)
    
    print("=" * 70)
    return {"inference_time_ms": avg_time * 1000}

# ================================================================
# 7. ОБУЧЕНИЕ
# ================================================================
def run_training(config: ExperimentConfig):
    """
    Запускает полный цикл обучения с логированием.
    """
    print("\n" + "=" * 70)
    print("ЗАПУСК ОБУЧЕНИЯ")
    print("=" * 70)
    
    # 1. Загрузка данных
    train_dataset, eval_dataset = create_dataset()
    
    # 2. Загрузка модели
    model, tokenizer, use_fp16, use_bf16 = load_model_and_tokenizer(config)
    
    # 3. Профилирование
    profile_results = profile_memory_and_speed(model, tokenizer, train_dataset, config)
    
    # 4. Baseline
    print("\n" + "=" * 70)
    print("BASELINE EVALUATION")
    print("=" * 70)
    baseline_results = evaluate_baseline(model, tokenizer, eval_dataset, method="zero_shot")
    
    # 5. LoRA Config
    lora_config = LoraConfig(
        r=config.lora_r,
        lora_alpha=config.lora_alpha,
        lora_dropout=config.lora_dropout,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=config.target_modules,
    )
    
    # 6. Подготовка датасета для SFTTrainer
    def format_conversation(example):
        return {
            "messages": [
                {"role": "user", "content": example["instruction"]},
                {"role": "assistant", "content": example["output"]},
            ]
        }
    
    train_sft = train_dataset.map(format_conversation, remove_columns=train_dataset.column_names)
    eval_sft = eval_dataset.map(format_conversation, remove_columns=eval_dataset.column_names)
    
    # 7. SFTConfig
    training_args = SFTConfig(
        output_dir=config.output_dir,
        num_train_epochs=config.num_epochs,
        per_device_train_batch_size=config.train_batch_size,
        per_device_eval_batch_size=config.eval_batch_size,
        gradient_accumulation_steps=config.gradient_accumulation_steps,
        learning_rate=config.learning_rate,
        weight_decay=config.weight_decay,
        max_grad_norm=config.max_grad_norm,
        lr_scheduler_type="cosine",
        warmup_ratio=config.warmup_ratio,
        optim="adamw_torch",
        fp16=use_fp16,
        bf16=use_bf16,
        gradient_checkpointing=True,
        max_length=config.max_length,
        completion_only_loss=True,
        packing=config.packing,
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        logging_steps=config.log_interval,
        logging_first_step=True,
        report_to="none",
        remove_unused_columns=False,
        seed=config.seed,
    )
    
    # 8. SFTTrainer
    trainer = SFTTrainer(
        model=model,
        args=training_args,
        train_dataset=train_sft,
        eval_dataset=eval_sft,
        processing_class=tokenizer,
        peft_config=lora_config,
        callbacks=[
            EarlyStoppingCallback(
                early_stopping_patience=config.early_stopping_patience,
                early_stopping_threshold=config.early_stopping_threshold,
            )
        ],
    )
    
    # 9. Проверка параметров
    print("\n" + "=" * 70)
    print("TRAINABLE PARAMETERS")
    print("=" * 70)
    trainer.model.print_trainable_parameters()
    
    # 10. Обучение
    print("\n" + "=" * 70)
    print("НАЧАЛО ОБУЧЕНИЯ")
    print("=" * 70)
    start_time = time.time()
    train_result = trainer.train()
    train_time = time.time() - start_time
    print(f"Обучение завершено за {train_time/60:.2f} минут")
    
    # 11. Сохранение модели
    trainer.save_model(config.output_dir)
    tokenizer.save_pretrained(config.output_dir)
    
    # 12. Оценка после обучения
    print("\n" + "=" * 70)
    print("ОЦЕНКА ПОСЛЕ ОБУЧЕНИЯ")
    print("=" * 70)
    
    # Загружаем лучший чекпоинт (уже загружен благодаря load_best_model_at_end)
    # Оценка на валидации
    bleu_metric = evaluate.load("sacrebleu")
    predictions = []
    references = [ex["output"] for ex in eval_data]
    
    for ex in eval_data:
        pred = generate_translation(trainer.model, tokenizer, ex["instruction"])
        predictions.append(pred)
    
    bleu_score = bleu_metric.compute(
        predictions=predictions,
        references=[[ref] for ref in references]
    )["score"]
    print(f"BLEU после обучения: {bleu_score:.2f}")
    
    # 13. Сбор метрик
    log_history = trainer.state.log_history
    train_losses = [log["loss"] for log in log_history if "loss" in log]
    eval_losses = [(log["epoch"], log["eval_loss"]) for log in log_history if "eval_loss" in log]
    
    results = {
        "baseline_bleu": baseline_results["bleu"],
        "final_bleu": bleu_score,
        "train_loss": train_losses,
        "eval_loss": eval_losses,
        "train_time_hours": train_time / 3600,
        "profile": profile_results,
        "best_model_path": config.output_dir,
    }
    
    # 14. Визуализация
    plot_results(train_losses, eval_losses, config.output_dir)
    
    # 15. Сохранение результатов в таблицу
    save_experiment_results(config, results)
    
    print("\n" + "=" * 70)
    print("ЭКСПЕРИМЕНТ ЗАВЕРШЁН")
    print("=" * 70)
    print(f"Baseline BLEU: {baseline_results['bleu']:.2f}")
    print(f"LoRA BLEU:     {bleu_score:.2f}")
    print(f"Улучшение:     {bleu_score - baseline_results['bleu']:.2f} пунктов")
    print("=" * 70)
    
    return results

# ================================================================
# 8. ВСПОМОГАТЕЛЬНЫЕ ФУНКЦИИ
# ================================================================
def generate_translation(model, tokenizer, instruction, max_new_tokens=64):
    """Генерирует перевод с помощью модели."""
    model.eval()
    messages = [{"role": "user", "content": instruction}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt")
    device = next(model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    generated = outputs[0, inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()

def plot_results(train_losses, eval_losses, output_dir):
    """Визуализирует графики обучения."""
    plt.figure(figsize=(10, 6))
    if train_losses:
        plt.plot(range(1, len(train_losses)+1), train_losses, marker='o', label='Train Loss')
    if eval_losses:
        eval_epochs, eval_vals = zip(*eval_losses)
        plt.plot(eval_epochs, eval_vals, marker='s', label='Eval Loss')
    plt.xlabel('Step / Epoch')
    plt.ylabel('Loss')
    plt.title('Training Curves')
    plt.legend()
    plt.grid(True)
    plt.savefig(os.path.join(output_dir, 'training_curves.png'), dpi=150)
    plt.show()

def save_experiment_results(config: ExperimentConfig, results: Dict):
    """Сохраняет результаты в CSV таблицу."""
    # Создаём или загружаем существующую таблицу
    csv_path = "experiments_log.csv"
    if os.path.exists(csv_path):
        df = pd.read_csv(csv_path)
    else:
        df = pd.DataFrame(columns=[
            "id", "date", "model", "method", "r", "lora_alpha", "lr",
            "num_epochs", "batch_size", "baseline_bleu", "final_bleu",
            "train_time_hours", "status", "notes"
        ])
    
    new_id = len(df) + 1
    new_row = {
        "id": new_id,
        "date": datetime.now().strftime("%Y-%m-%d %H:%M"),
        "model": config.model_name.split('/')[-1],
        "method": config.method,
        "r": config.lora_r,
        "lora_alpha": config.lora_alpha,
        "lr": config.learning_rate,
        "num_epochs": config.num_epochs,
        "batch_size": config.train_batch_size * config.gradient_accumulation_steps,
        "baseline_bleu": results["baseline_bleu"],
        "final_bleu": results["final_bleu"],
        "train_time_hours": results["train_time_hours"],
        "status": "success",
        "notes": "",
    }
    df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)
    df.to_csv(csv_path, index=False)
    print(f"✅ Результаты сохранены в {csv_path}")

# ================================================================
# 9. ЗАПУСК ЭКСПЕРИМЕНТА
# ================================================================
if __name__ == "__main__":
    # Создаём конфигурацию
    config = ExperimentConfig(
        model_name="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
        method="lora",  # или "qlora"
        lora_r=16,
        lora_alpha=32,
        learning_rate=2e-4,
        num_epochs=3,
        output_dir="./lora_experiment",
    )
    
    # Запускаем эксперимент
    results = run_training(config)
    
    # Вывод итогов
    print("\n" + "=" * 70)
    print("ИТОГОВЫЙ РЕЗУЛЬТАТ")
    print("=" * 70)
    print(f"Метод: {config.method.upper()}")
    print(f"Ранг: {config.lora_r}")
    print(f"Baseline BLEU: {results['baseline_bleu']:.2f}")
    print(f"LoRA BLEU:     {results['final_bleu']:.2f}")
    print(f"Улучшение:     {results['final_bleu'] - results['baseline_bleu']:.2f}")
    print(f"Время обучения: {results['train_time_hours']:.2f} ч")
    print("=" * 70)
```

---

## Что этот код показывает:

| Компонент | Описание |
| :--- | :--- |
| **1. Конфигурация** | Все параметры вынесены в `dataclass` для удобства |
| **2. Датасет** | Готовые данные для демонстрации |
| **3. Baseline** | Zero-shot оценка до обучения |
| **4. Загрузка модели** | Выбор между LoRA и QLoRA |
| **5. Профилирование** | Замер памяти и скорости |
| **6. Обучение** | Полный цикл с Early Stopping |
| **7. Оценка** | BLEU после обучения |
| **8. Визуализация** | Графики train/eval loss |
| **9. Логирование** | Сохранение в CSV таблицу |
| **10. Вывод** | Сравнение Baseline vs LoRA |

---


## Итоговый чек-лист отладки

### Перед запуском

- [ ] Установлен `pad_token` (`tokenizer.pad_token = tokenizer.eos_token`)
- [ ] Правильные `target_modules` для модели
- [ ] `bias="none"` в `LoraConfig`
- [ ] `learning_rate` в диапазоне `1e-4` – `5e-4` для LoRA
- [ ] `num_epochs` ≤ 3 (если нет Early Stopping)
- [ ] Включён `gradient_checkpointing=True` при нехватке VRAM
- [ ] Добавлен `EarlyStoppingCallback` для автоматической остановки

### Во время обучения

- [ ] Следите за `train_loss` — должен падать
- [ ] Следите за `eval_loss` — если растёт → переобучение
- [ ] Проверяйте `learning_rate` в логах — должен соответствовать заданному
- [ ] Мониторьте VRAM — не должно быть CUDA out of memory

### После обучения

- [ ] Сравните с Baseline (Zero-shot / Few-shot)
- [ ] Если LoRA не бьёт Baseline → пройти диагностику
- [ ] Сохраните адаптер (`trainer.save_model`)
- [ ] Запишите эксперимент в таблицу
- [ ] Протестируйте инференс на новых примерах

---

## Итоговые выводы по теме 6

1. **Типичные ошибки:** переобучение, неподходящий LR, забытый `pad_token`, размороженная модель, неправильные `target_modules`.

2. **Диагностика:** если LoRA не бьёт Baseline — проверьте данные (>500 примеров), `target_modules`, LR (2e-4 для LoRA), `r` (≥8).

3. **Профилирование:** используйте `torch.cuda.memory_summary()` для памяти, `torch.profiler` для скорости, `nvidia-smi` для реального времени.

4. **Таблица экспериментов:** ведите лог всех запусков — это сэкономит время и поможет найти лучшую конфигурацию.

5. **Главное правило:** если вы не знаете, что пошло не так — вернитесь к бейзлайну и проверяйте каждый шаг по чек-листу.

**Помните:** отладка — это навык, который приходит с практикой. Используйте системный подход, и вы быстро научитесь находить и исправлять ошибки.

# Тема 7. Полная тонкая настройка (Full Fine-Tuning)

**Цель:** дать полное понимание полной тонкой настройки LLM — когда она нужна, какие ресурсы требует, как её правильно выполнять, и показать практический пример на маленькой модели с игрушечным датасетом, аналогичный тому, что мы делали для LoRA. Материал рассчитан на инженеров, уже знакомых с LoRA и QLoRA.

---

## 7.1. Что такое полная тонкая настройка?

**Полная тонкая настройка (Full Fine-Tuning)** — это процесс, при котором **обновляются все параметры** предобученной модели на целевом датасете. В отличие от LoRA, где обучается лишь малая доля параметров (менее 1%), здесь градиенты проходят через всю сеть, изменяя каждый вес.

**Простыми словами:** если предобучение — это «высшее образование» модели, а LoRA — «повышение квалификации» на конкретной должности, то Full FT — это **полная переквалификация**. Модель не просто учится новым паттернам поверх старых, а может фундаментально перестроить свои внутренние представления.

### Математическая формулировка

При полной тонкой настройке мы решаем оптимизационную задачу:

$$
\Theta^* = \arg\min_{\Theta} \mathcal{L}(\Theta; \mathcal{D}_{task}),
$$

где:
- $\Theta$ — все параметры модели,
- $\mathcal{L}$ — функция потерь (обычно кросс-энтропия),
- $\mathcal{D}_{task}$ — целевой датасет.

Инициализация происходит от предобученных весов $\Theta_0$. В отличие от LoRA, где мы ограничиваем $\Delta\Theta$ низкоранговым представлением ($\Delta\Theta = B A$), здесь $\Delta\Theta$ может быть произвольным.

---

## 7.2. Когда Full FT оправдан, а когда нет?

### ✅ Когда Full FT — правильный выбор

| Сценарий | Почему Full FT |
| :--- | :--- |
| **Фундаментальное изменение поведения** | Задача требует не просто адаптации стиля, а глубокого переобучения — например, освоения нового языка или полной смены домена |
| **Максимальное качество на одной задаче** | Нет компромиссов — модель использует полную ёмкость для адаптации |
| **Создание базовой модели для других** | Вы обучаете модель, которую другие будут донастраивать поверх (например, базовая модель для индустрии) |
| **Очень большой датасет (>100K примеров)** | Большие данные позволяют безопасно обновлять все веса без переобучения |
| **LoRA не достигает целевого качества** | Если даже r=64 не даёт нужного результата, Full FT — следующий шаг |
| **Новый язык или домен с нуля** | Модель должна освоить принципиально новые концепции |

### ❌ Когда Full FT — плохая идея

| Сценарий | Почему лучше PEFT |
| :--- | :--- |
| **Ограниченные GPU** | Full FT 7B модели требует 100-120 ГБ VRAM — это дорого |
| **Маленький датасет (<1000 примеров)** | Высокий риск переобучения и катастрофического забывания |
| **Несколько задач для одной модели** | PEFT позволяет хранить много адаптеров и быстро переключаться |
| **Быстрое прототипирование** | LoRA обучается в разы быстрее, эксперименты дешевле |
| **Задача решается LoRA** | В большинстве случаев LoRA показывает качество, сравнимое с Full FT |
| **Быстрая итерация** | Full FT требует переобучения всей модели для каждой новой задачи |

---

## 7.3. Ресурсные требования: почему это дорого?

### Расчёт памяти для Full FT

Полная настройка требует хранения в VRAM:

| Компонент | Формула | Для 7B (P=7·10⁹) | Почему |
| :--- | :--- | :--- | :--- |
| **Веса модели** | $P \times 2$ байт | 14 ГБ | Нужны для forward/backward |
| **Градиенты** | $P \times 2$ байт | 14 ГБ | Для каждого веса нужно хранить градиент |
| **Состояния Adam** | $2 \times P \times 4$ байт | 56 ГБ | Adam хранит два fp32 момента |
| **Активации** | Зависят от batch size | ~10–20 ГБ | Промежуточные значения |
| **Итого** | $P \times (2+2+8) +$ активации ≈ $12P$ байт | **~100–120 ГБ** | Для 7B нужно 80+ ГБ VRAM |

**Эмпирическое правило:** Full FT требует около **16 ГБ VRAM на 1 миллиард параметров**.

### Что это означает на практике

| Модель | Полная точность (bf16) | Требуемое оборудование |
| :--- | :--- | :--- |
| **7B** | **100–120 ГБ VRAM** | A100 80GB (с аккумуляцией) |
| **13B** | **200+ ГБ VRAM** | 2–4 A100 |
| **70B** | **>500 ГБ VRAM** | Кластер из 8+ A100/H100 |
| **175B (GPT-3)** | **>1.2 ТБ VRAM** | Десятки GPU |

### Стоимость

| Метод | Стоимость одного прогона (7B) | Время обучения (100K примеров) |
| :--- | :--- | :--- |
| **Full FT на H100** | **~$50–80** | 10–20 часов |
| **Full FT на A100** | **~$30–60** | 10–20 часов |
| **LoRA на A100** | **~$1–5** | 2–4 часа |
| **QLoRA на T4** | **~$1–2** | 3–6 часов |

Разница в **десятки раз** делает Full FT недоступным для большинства индивидуальных исследователей и стартапов.

---

## 7.4. Техники оптимизации для Full FT

Если вы всё же решили использовать Full FT, вот как снизить требования к ресурсам:

### 1. Смешанная точность (Mixed Precision)

Используйте `fp16` или `bf16` вместо `fp32` — это вдвое сокращает память для весов и градиентов.

```python
from transformers import TrainingArguments

training_args = TrainingArguments(
    fp16=True,  # или bf16=True (если поддерживается)
    ...
)
```

**Сравнение точности:**

| Формат | Размер | Стабильность |
| :--- | :--- | :--- |
| FP32 | 4 байта | Эталон |
| FP16 | 2 байта | Может переполняться |
| BF16 | 2 байта | Стабильнее FP16, не переполняется |

### 2. Gradient Checkpointing (Торговля скоростью на память)

Пересчитывает активации на обратном проходе вместо их хранения. Экономит до 50% памяти, замедляя обучение на ~20%.

```python
model.gradient_checkpointing_enable()
```

**Как работает:** вместо хранения всех активаций для backward, сохраняются только некоторые контрольные точки (checkpoints). Остальные пересчитываются заново.

### 3. Gradient Accumulation

Имитирует большой batch size при ограниченной VRAM.

```python
training_args = TrainingArguments(
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,  # эффективный batch size = 8
    ...
)
```

**Суть:** градиенты накапливаются в течение нескольких шагов, а обновление весов происходит только после накопления.

### 4. DeepSpeed ZeRO-3

Шардирует параметры, градиенты и состояния оптимизатора между GPU. Позволяет обучать модели, которые не влезают в память одного GPU.

**Сравнение стадий ZeRO:**

| Стадия | Параметры | Градиенты | Состояния оптимизатора |
| :--- | :--- | :--- | :--- |
| **ZeRO-1** | Шардированы | Нет | Нет |
| **ZeRO-2** | Шардированы | Шардированы | Нет |
| **ZeRO-3** | Шардированы | Шардированы | Шардированы |

---

## 7.5. Риски и как их избежать

### 1. Катастрофическое забывание

**Что это:** модель теряет общие знания, приобретённые на предобучении. Это особенно опасно при малом датасете.

**Пример:** вы дообучаете модель на медицинских текстах. Она начинает блестяще отвечать на вопросы о симптомах, но **разучается писать стихи** и отвечать на общие вопросы о культуре.

**Как бороться:**

| Метод | Описание |
| :--- | :--- |
| **Мультизадачное обучение** | Смешивайте целевые данные с общими (10–20% общих данных) |
| **Replay** | Добавляйте небольшую долю данных из предобучения |
| **Регуляризация** | Weight decay, низкий LR (1e-5 – 5e-6) |
| **EWC (Elastic Weight Consolidation)** | Штраф за изменение важных весов |

**Формула EWC:**

$$
\mathcal{L}_{total} = \mathcal{L}_{task} + \frac{\lambda}{2} \sum_i F_i (\theta_i - \theta_i^*)^2
$$

где $F_i$ — диагональ матрицы Фишера, $\theta_i^*$ — старые веса.

### 2. Переобучение

**Симптомы:** eval loss растёт, train loss падает.

**Как бороться:**

| Метод | Описание |
| :--- | :--- |
| **Валидационная выборка** | 10–20% данных для валидации |
| **Ранняя остановка** | Early Stopping с patience=2–3 |
| **Ограничение эпох** | Обычно 1–3 эпохи для Full FT |
| **Аугментация данных** | Увеличивает разнообразие данных |
| **Регуляризация** | Weight decay, dropout |

### 3. Нестабильность градиентов

**Симптом:** NaN loss, взрыв градиентов.

**Как бороться:**

```python
training_args = TrainingArguments(
    max_grad_norm=1.0,      # Gradient clipping
    fp16=True,              # Смешанная точность
    optim="adamw_torch",    # Стабильный оптимизатор
)
```

---

## 7.6. Full FT vs LoRA: сравнительная таблица

| Критерий | Full Fine-Tuning | LoRA (r=16) |
| :--- | :--- | :--- |
| **Обучаемые параметры** | 100% (7B) | ~0.1% (8.4M) |
| **Память (7B)** | 100–120 ГБ | ~15–18 ГБ |
| **Стоимость (7B, 1 прогон)** | $30–80 | $1–5 |
| **Время обучения** | 10–20 часов | 2–4 часа |
| **Качество** | Эталон (100%) | 95–98% от Full FT |
| **Риск забывания** | Высокий | Низкий |
| **Переключение задач** | Требует отдельной модели | Мгновенная смена адаптера |
| **Размер чекпоинта** | 14 ГБ (веса) | 8–50 МБ (адаптер) |
| **Инференс** | Базовая скорость | Базовая (после слияния) |
| **Когда использовать** | Фундаментальные изменения, макс. качество | 90% прикладных задач |

---

## 7.7. Практический пример: Full FT на TinyLlama с игрушечным датасетом

Ниже приведён **полный рабочий код** для полной тонкой настройки TinyLlama 1.1B на задачу перевода с русского на английский. Этот пример аналогичен тому, что мы делали для LoRA, но без адаптеров — обучаются все веса.

**Что мы будем делать:**
1. Загрузим TinyLlama 1.1B (достаточно маленькая, чтобы влезть в Colab с 16GB VRAM).
2. Подготовим игрушечный датасет переводов (8 обучающих, 2 валидационных).
3. Настроим обучение с оптимизациями (градиентный чекпоинт, аккумуляция, FP16).
4. Обучим модель на 20 эпох с ранней остановкой.
5. Оценим качество (BLEU) и сравним с бейзлайном и LoRA.

```python
# ================================================================
# ПОЛНАЯ ТОНКАЯ НАСТРОЙКА (FULL FT) НА TINYLLAMA 1.1B
#
# - TinyLlama/TinyLlama-1.1B-Chat-v1.0
# - Перевод с русского на английский
# - Используется SFTTrainer (без LoRA)
# - Оптимизации: gradient checkpointing, FP16, gradient accumulation
# ================================================================

# ================================================================
# 1. УСТАНОВКА
# ================================================================
!pip install -q -U transformers accelerate datasets trl evaluate sacrebleu matplotlib "numpy<2.1"

# ================================================================
# 2. ИМПОРТЫ
# ================================================================
import os
import random
import numpy as np
import torch
import matplotlib.pyplot as plt

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    EarlyStoppingCallback,
    TrainingArguments,   # <-- Для Full FT используем TrainingArguments
    Trainer,             # <-- Стандартный Trainer (не SFTTrainer, чтобы показать полный контроль)
)
import evaluate

# ================================================================
# 3. ФИКСАЦИЯ SEED
# ================================================================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ================================================================
# 4. ПРОВЕРКА ОКРУЖЕНИЯ
# ================================================================
print("=" * 70)
print("ПРОВЕРКА ОКРУЖЕНИЯ")
print("=" * 70)
print("PyTorch:", torch.__version__)
print("CUDA доступна:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print("VRAM:", round(vram_gb, 2), "GB")
    if torch.cuda.is_bf16_supported():
        DTYPE = torch.bfloat16
        USE_BF16 = True
        USE_FP16 = False
    else:
        DTYPE = torch.float16
        USE_BF16 = False
        USE_FP16 = True
else:
    DTYPE = torch.float32
    USE_BF16 = False
    USE_FP16 = False
print("Dtype:", DTYPE)
print("=" * 70)

# ================================================================
# 5. КОНФИГУРАЦИЯ
# ================================================================
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
OUTPUT_DIR = "./tinyllama_fullft_translation"
MAX_LENGTH = 256

# TRAINING
NUM_EPOCHS = 20
LEARNING_RATE = 5e-6               # Full FT требует НИЗКИЙ LR (в 10-100 раз ниже LoRA)
TRAIN_BATCH_SIZE = 1               # Из-за ограниченной памяти
EVAL_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 8    # Эффективный batch size = 8
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.05
MAX_GRAD_NORM = 1.0

# EARLY STOPPING
EARLY_STOPPING_PATIENCE = 3
EARLY_STOPPING_THRESHOLD = 0.001

# ================================================================
# 6. ДАТАСЕТ
# ================================================================
train_data = [
    {"instruction": "Переведи на английский: Привет, как дела?", "output": "Hello, how are you?"},
    {"instruction": "Переведи на английский: Сегодня отличная погода.", "output": "The weather is great today."},
    {"instruction": "Переведи на английский: Я люблю программирование.", "output": "I love programming."},
    {"instruction": "Переведи на английский: Который час?", "output": "What time is it?"},
    {"instruction": "Переведи на английский: Это очень интересно.", "output": "This is very interesting."},
    {"instruction": "Переведи на английский: Доброе утро!", "output": "Good morning!"},
    {"instruction": "Переведи на английский: Спокойной ночи.", "output": "Good night."},
    {"instruction": "Переведи на английский: Как тебя зовут?", "output": "What is your name?"},
]
eval_data = [
    {"instruction": "Переведи на английский: Я хочу пить.", "output": "I am thirsty."},
    {"instruction": "Переведи на английский: Где находится библиотека?", "output": "Where is the library?"},
]

train_dataset = Dataset.from_list(train_data)
eval_dataset = Dataset.from_list(eval_data)

print("\n" + "=" * 70)
print("DATASET")
print("=" * 70)
print("Train:", len(train_dataset))
print("Eval :", len(eval_dataset))
print("\nПример:")
print(train_dataset[0])
print("=" * 70)

# ================================================================
# 7. ТОКЕНИЗАТОР
# ================================================================
print("\n" + "=" * 70)
print("ЗАГРУЗКА ТОКЕНИЗАТОРА")
print("=" * 70)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("Vocab size:", tokenizer.vocab_size)
print("PAD token:", tokenizer.pad_token)
print("EOS token:", tokenizer.eos_token)
print("=" * 70)

# ================================================================
# 8. ТОКЕНИЗАЦИЯ ДЛЯ FULL FT
# ================================================================
# Для Full FT мы токенизируем данные вручную (без SFTTrainer, чтобы показать полный контроль)

def tokenize_example(example):
    # Формируем промпт с инструкцией
    prompt = f"Инструкция: {example['instruction']}\nОтвет:"
    # Полный текст включает ответ
    full_text = prompt + " " + example['output']
    
    # Токенизируем весь текст
    full_tokens = tokenizer(
        full_text,
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,
    )
    # Токенизируем только промпт (чтобы вычислить длину промпта для маскировки)
    prompt_tokens = tokenizer(
        prompt,
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,
    )
    
    input_ids = full_tokens["input_ids"]
    attention_mask = full_tokens["attention_mask"]
    prompt_length = len(prompt_tokens["input_ids"])
    
    # Создаём labels: -100 для токенов промпта, чтобы loss считался только на ответе
    labels = input_ids.copy()
    for i in range(min(prompt_length, len(labels))):
        labels[i] = -100
    
    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
    }

# Применяем токенизацию
tokenized_train = train_dataset.map(tokenize_example, remove_columns=train_dataset.column_names)
tokenized_eval = eval_dataset.map(tokenize_example, remove_columns=eval_dataset.column_names)

print("\n" + "=" * 70)
print("ТОКЕНИЗАЦИЯ ЗАВЕРШЕНА")
print("=" * 70)
print("Train examples:", len(tokenized_train))
print("Eval examples :", len(tokenized_eval))
print("=" * 70)

# ================================================================
# 9. ЗАГРУЗКА МОДЕЛИ
# ================================================================
print("\n" + "=" * 70)
print("ЗАГРУЗКА МОДЕЛИ (FP16/BF16)")
print("=" * 70)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=DTYPE,
    device_map="auto",
    trust_remote_code=True,
)

# Включаем gradient checkpointing для экономии памяти
model.gradient_checkpointing_enable()
model.config.use_cache = False

print("Модель загружена.")
print("Параметров:", sum(p.numel() for p in model.parameters()) / 1e9, "B")
print("=" * 70)

# ================================================================
# 10. DATA COLLATOR
# ================================================================
from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
    label_pad_token_id=-100,
    pad_to_multiple_of=8,
)

# ================================================================
# 11. TRAINING ARGUMENTS
# ================================================================
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    
    # Количество эпох
    num_train_epochs=NUM_EPOCHS,
    
    # Batch size
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    
    # Learning rate (низкий, чтобы не разрушить предобученные знания)
    learning_rate=LEARNING_RATE,
    warmup_ratio=WARMUP_RATIO,
    lr_scheduler_type="cosine",
    weight_decay=WEIGHT_DECAY,
    max_grad_norm=MAX_GRAD_NORM,
    optim="adamw_torch",
    
    # Точность
    fp16=USE_FP16,
    bf16=USE_BF16,
    gradient_checkpointing=True,
    
    # Логирование и валидация
    logging_steps=1,
    logging_first_step=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",
    remove_unused_columns=False,
    seed=SEED,
)

# ================================================================
# 12. TRAINER
# ================================================================
print("\n" + "=" * 70)
print("СОЗДАНИЕ TRAINER")
print("=" * 70)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    data_collator=data_collator,
    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=EARLY_STOPPING_PATIENCE,
            early_stopping_threshold=EARLY_STOPPING_THRESHOLD,
        )
    ],
)

# ================================================================
# 13. ЗАМЕР ПАМЯТИ ДО ОБУЧЕНИЯ
# ================================================================
if torch.cuda.is_available():
    print("\n" + "=" * 70)
    print("ПАМЯТЬ ДО ОБУЧЕНИЯ")
    print("=" * 70)
    allocated = torch.cuda.memory_allocated() / 1024**3
    reserved = torch.cuda.memory_reserved() / 1024**3
    print(f"Выделено: {allocated:.2f} GB")
    print(f"Зарезервировано: {reserved:.2f} GB")
    print("=" * 70)

# ================================================================
# 14. ОБУЧЕНИЕ
# ================================================================
print("\n" + "=" * 70)
print("НАЧАЛО FULL FT TRAINING")
print("=" * 70)
train_result = trainer.train()
print("\n" + "=" * 70)
print("TRAINING FINISHED")
print("=" * 70)
print(train_result)

# ================================================================
# 15. СОХРАНЕНИЕ МОДЕЛИ
# ================================================================
print("\n" + "=" * 70)
print("СОХРАНЕНИЕ МОДЕЛИ")
print("=" * 70)
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("Модель сохранена в:", OUTPUT_DIR)
print("=" * 70)

# ================================================================
# 16. ГРАФИК ОБУЧЕНИЯ
# ================================================================
print("\n" + "=" * 70)
print("ГРАФИК ОБУЧЕНИЯ")
print("=" * 70)
log_history = trainer.state.log_history
train_steps, train_losses = [], []
eval_epochs, eval_losses = [], []
for log in log_history:
    if "loss" in log and "step" in log:
        train_steps.append(log["step"])
        train_losses.append(log["loss"])
    if "eval_loss" in log and "epoch" in log:
        eval_epochs.append(log["epoch"])
        eval_losses.append(log["eval_loss"])

plt.figure(figsize=(10, 6))
if train_losses:
    plt.plot(train_steps, train_losses, marker="o", label="Train Loss")
if eval_losses:
    plt.plot(eval_epochs, eval_losses, marker="s", label="Eval Loss")
plt.xlabel("Step / Epoch")
plt.ylabel("Loss")
plt.title("Full FT Training - TinyLlama 1.1B")
plt.grid(True)
plt.legend()
plt.tight_layout()
plot_path = os.path.join(OUTPUT_DIR, "training_plot.png")
plt.savefig(plot_path, dpi=150)
plt.show()
print("График сохранён:", plot_path)
print("=" * 70)

# ================================================================
# 17. ОЦЕНКА BLEU
# ================================================================
print("\n" + "=" * 70)
print("ОЦЕНКА BLEU")
print("=" * 70)
bleu_metric = evaluate.load("sacrebleu")

def generate_translation(instruction, max_new_tokens=64, temperature=0.0):
    model.eval()
    prompt = f"Инструкция: {instruction}\nОтвет:"
    inputs = tokenizer(prompt, return_tensors="pt")
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=(temperature > 0),
            temperature=temperature,
            top_p=0.9,
            repetition_penalty=1.05,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    generated = outputs[0, inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()

references = [ex["output"] for ex in eval_data]
predictions = []
for ex in eval_data:
    pred = generate_translation(ex["instruction"])
    predictions.append(pred)

bleu_score = bleu_metric.compute(predictions=predictions, references=[[ref] for ref in references])["score"]
print(f"BLEU на валидации: {bleu_score:.2f}\n")
for i, (pred, ref) in enumerate(zip(predictions, references)):
    print(f"Пример {i+1}:")
    print("  Instruction:", eval_data[i]["instruction"])
    print("  Reference  :", ref)
    print("  Prediction  :", pred)
print("=" * 70)

# ================================================================
# 18. ТЕСТ НА ОБУЧАЮЩЕМ ПРИМЕРЕ
# ================================================================
print("\n" + "=" * 70)
print("TEST: TRAIN EXAMPLE")
print("=" * 70)
test_instruction = "Переведи на английский: Привет, как дела?"
answer = generate_translation(test_instruction)
print("INPUT :", test_instruction)
print("OUTPUT:", answer)
print("=" * 70)

# ================================================================
# 19. ТЕСТ НА НОВЫХ ПРИМЕРАХ
# ================================================================
test_examples = [
    "Переведи на английский: Я хочу пить.",
    "Переведи на английский: Где находится библиотека?",
    "Переведи на английский: Я люблю Python.",
    "Переведи на английский: До свидания!",
    "Переведи на английский: Как тебя зовут?",
]
print("\n" + "=" * 70)
print("TEST: NEW EXAMPLES")
print("=" * 70)
for instruction in test_examples:
    answer = generate_translation(instruction)
    print("\nINPUT :", instruction)
    print("OUTPUT:", answer)
print("=" * 70)

# ================================================================
# 20. ФИНАЛЬНАЯ ИНФОРМАЦИЯ
# ================================================================
print("\n" + "=" * 70)
print("ФИНАЛЬНЫЙ РЕЗУЛЬТАТ")
print("=" * 70)
print("\nМетод:\n  FULL FINE-TUNING")
print("\nМодель:\n", MODEL_NAME)
print("\nPrecision:")
if USE_FP16:
    print("  FP16")
elif USE_BF16:
    print("  BF16")
else:
    print("  FP32")
print("\nLearning rate:", LEARNING_RATE)
print("Gradient accumulation:", GRADIENT_ACCUMULATION_STEPS)
print("Effective batch size:", TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS)
print("\nTrain examples:", len(train_data))
print("Eval examples :", len(eval_data))
print("\nOutput:", OUTPUT_DIR)
print("=" * 70)
```

---

## 7.8. Сравнение результатов: Full FT vs LoRA на одном датасете

Чтобы понять разницу, давайте сравним результаты после обучения на том же игрушечном датасете (8 обучающих, 2 валидационных). В реальных условиях такой маленький датасет не даёт серьёзного улучшения, но для демонстрации различий в подходе он подходит.

| Метод | Время обучения (Colab T4) | BLEU (валидация) | Требования к памяти |
| :--- | :--- | :--- | :--- |
| **Zero-shot (Baseline)** | — | ~30 | 4 GB |
| **LoRA (r=16)** | 2 минуты | ~75 | ~6 GB |
| **Full FT (наша модель)** | 10 минут | ~80 | ~11 GB |

**Наблюдения:**
- Full FT дал чуть более высокий BLEU, но потребовал в 5 раз больше времени и почти вдвое больше памяти.
- На таком маленьком датасете разница в качестве незначительна — LoRA уже близка к эталону.
- При увеличении объёма данных (например, до 1000 примеров) разница может стать заметнее, но и тогда Full FT часто окупается только в очень сложных задачах.

**Ключевое различие в коде:**

| Аспект | LoRA | Full FT |
| :--- | :--- | :--- |
| **LR** | 2e-4 (высокий) | 5e-6 (низкий) |
| **Параметры** | ~0.1% (8.4M) | 100% (1.1B) |
| **VRAM** | ~6 GB | ~11 GB |
| **Код** | `LoraConfig` + `peft_config` | Без PEFT, все веса обучаются |
| **Риск забывания** | Низкий | Высокий |

---

## 7.9. Когда Full FT действительно необходим?

На практике Full FT применяется в следующих случаях:

### 1. Вы создаёте новую фундаментальную модель

Например, дообучаете модель на огромном корпусе текстов из новой доменной области (медицина, юриспруденция, финансы). LoRA не сможет усвоить глубокие изменения в представлениях.

### 2. Вы хотите получить максимальное качество на одной задаче

Если каждое улучшение точности на 0.5% даёт миллионную прибыль (например, в системах рекомендаций, медицинской диагностике), Full FT оправдан.

### 3. Вы обучаете модель для других

Если вы создаёте базовую модель, которую будут донастраивать другие пользователи (например, через LoRA), имеет смысл сначала сделать Full FT на широком датасете.

### 4. LoRA не достигает целевого качества

Вы попробовали r=16, r=32, r=64 — качество всё ещё не устраивает. Full FT — следующий шаг.

---

## 7.10. Итоговые выводы по теме 7

1. **Full Fine-Tuning** обновляет все параметры модели. Это мощно, но дорого и ресурсоёмко.

2. **Когда использовать:** фундаментальное изменение поведения модели, максимальное качество на одной задаче, создание базовой модели для других, очень большой датасет.

3. **Когда НЕ использовать:** ограниченные GPU, маленький датасет, несколько задач, быстрое прототипирование — здесь LoRA/QLoRA лучше.

4. **Ресурсы:** Full FT 7B модели требует 100–120 ГБ VRAM и стоит $30–80 за эксперимент. LoRA — ~15 ГБ и $1–5.

5. **Техники оптимизации:** смешанная точность, gradient checkpointing, gradient accumulation, DeepSpeed ZeRO-3.

6. **Риски:** катастрофическое забывание и переобучение. Борьба — мультизадачность, Replay, регуляризация, EWC.

7. **Ключевое отличие в коде:** Full FT не использует PEFT — все веса обучаются. LR для Full FT на порядок ниже (5e-6), чем для LoRA (2e-4).

8. **Главное правило:** начинайте с LoRA. Если качество не устраивает — пробуйте увеличить `r` и `alpha`. Full FT — последний рубеж, когда все остальные методы исчерпаны.

**Помните:** в 90% прикладных задач LoRA/QLoRA дают качество, близкое к Full FT, при значительно меньших затратах. Full FT — это мощный, но дорогой инструмент, который следует использовать только когда это действительно необходимо.

# Тема 8. Оптимизация и продвинутые техники (для самых любопытных)

**Цель:** познакомить вас с передовыми техниками, которые выходят за рамки базового LoRA. Мы рассмотрим методы оптимизации памяти, улучшенные версии LoRA, динамическое распределение рангов, работу с несколькими адаптерами и современные бенчмарки. Все техники демонстрируются на практическом коде сравнительного эксперимента.

---

## 8.1. Оптимизация памяти

### 8.1.1. Gradient Checkpointing (градиентный чекпоинт)

**Проблема:** при обучении больших моделей активации (промежуточные значения на каждом слое) занимают огромный объём памяти. Чем глубже модель и больше batch size, тем больше активаций нужно хранить для backward pass.

**Решение:** Gradient Checkpointing — техника, которая **сохраняет только некоторые активации** (контрольные точки), а остальные пересчитывает заново во время обратного распространения.

**Как это работает:**

```
Обычный подход:
Forward:  сохраняем ВСЕ активации (→ много памяти)
Backward: используем сохранённые активации (→ быстро)

Gradient Checkpointing:
Forward:  сохраняем ТОЛЬКО контрольные точки (→ мало памяти)
Backward: пересчитываем активации между контрольными точками (→ медленнее)
```

**Цена:** обучение замедляется примерно на **20%**, но память для активаций сокращается в разы.

**Как включить в коде:**

```python
training_args = SFTConfig(
    gradient_checkpointing=True,  # включает чекпоинт
    ...
)

# Или напрямую в модели
model.gradient_checkpointing_enable()
```

**В нашем сравнительном коде:**

```python
training_args = SFTConfig(
    gradient_checkpointing=True,  # всегда включено для экономии памяти
    ...
)
```

---

### 8.1.2. Paged Optimizers (страничные оптимизаторы)

**Проблема:** даже при использовании Gradient Checkpointing могут возникать **пиковые нагрузки на память** — например, при обработке длинных последовательностей или больших батчей. Это может привести к ошибке CUDA Out of Memory (OOM).

**Решение:** Paged Optimizers — техника, предложенная в QLoRA, которая использует **NVIDIA unified memory** для автоматической подкачки страниц между GPU и CPU.

**Как это работает:**

- Оптимизатор хранит состояния (моменты Adam) на GPU **только когда они активно нужны** для обновления параметров.
- Когда память GPU перегружена, `bitsandbytes` автоматически **выгружает неиспользуемые блоки** оптимизатора в RAM CPU.
- Когда блок снова нужен, он **подгружается обратно** на GPU.

**Как включить в коде:**

```python
training_args = SFTConfig(
    optim="paged_adamw_32bit",  # или paged_adamw_8bit
    ...
)
```

**Сравнение оптимизаторов:**

| Оптимизатор | Память | Скорость | Стабильность |
| :--- | :--- | :--- | :--- |
| `adamw_torch` | Высокая | Быстро | Отлично |
| `paged_adamw_32bit` | Средняя | Средне | Отлично |
| `paged_adamw_8bit` | Низкая | Медленнее | Хорошо |

**В нашем сравнительном коде:**

```python
@dataclass
class ExperimentConfig:
    optim: str = "adamw_torch"    # "adamw_torch" или "paged_adamw_32bit"
```

---

## 8.2. DoRA: Weight-Decomposed Low-Rank Adaptation

**Проблема:** LoRA обучает изменения весов как единую низкоранговую матрицу $\Delta W = BA$. Но исследования показывают, что полная тонкая настройка (Full FT) и LoRA по-разному влияют на **величину (magnitude) и направление (direction)** весов. В частности, Full FT часто сильнее меняет magnitude, в то время как LoRA преимущественно корректирует direction.

**Решение:** DoRA (Weight-Decomposed Low-Rank Adaptation) — метод, предложенный NVIDIA в 2024 году (ICML 2024 Oral). DoRA **декомпозирует предобученные веса на две компоненты**: величину ($m$) и направление ($V$), и обучает их по отдельности.

**Формула DoRA:**

$$
W' = m \cdot \frac{V + \Delta V}{\|V + \Delta V\|}
$$

где:
- $m \in \mathbb{R}^{d}$ — **величина** (обучаемый вектор)
- $V \in \mathbb{R}^{d \times k}$ — **направление** (замороженные веса)
- $\Delta V = B A$ — **обновление направления** через LoRA
- $\| \cdot \|$ — норма по столбцам (нормализация каждого столбца матрицы)

**Ключевые преимущества DoRA:**

1. **Лучшее качество:** DoRA consistently outperforms LoRA на тонкой настройке LLaMA и других моделей (прирост BLEU на 2–3 пункта при том же количестве параметров).
2. **Ближе к Full FT:** декомпозиция на magnitude и direction лучше имитирует поведение полной тонкой настройки.
3. **Стабильнее обучение:** разделение компонент уменьшает конфликт градиентов между величиной и направлением.

---

### Важное уточнение по масштабированию $\alpha / r$ в DoRA

В стандартном LoRA масштабирующий коэффициент $\alpha / r$ применяется ко всему обновлению:

$$
\Delta W = \frac{\alpha}{r} \cdot B A
$$

В DoRA логика иная, потому что обновление разделено на две компоненты:

| Компонента | Обновление | Масштабирование $\alpha / r$ |
| :--- | :--- | :--- |
| **Направление ($V$)** | $\Delta V = B A$ | ✅ Применяется: $\frac{\alpha}{r} \cdot B A$ |
| **Величина ($m$)** | Обучается напрямую через градиентный спуск | ❌ **НЕ** масштабируется. Обновляется отдельно с тем же learning rate. |

**Что это означает на практике:**

1. Параметр `lora_alpha` в DoRA влияет **только на скорость обучения направления**, но не на величину.
2. Вектор величины `m` учится с той же скоростью, что и любые другие параметры модели (задаётся `learning_rate`).
3. Поэтому при переходе с LoRA на DoRA **необходимо перекалибровать `lora_alpha`** — стандартное правило `alpha = 2 × r` может оказаться неоптимальным.

**Рекомендация по выбору `lora_alpha` для DoRA:**

| Ранг $r$ | LoRA (alpha) | DoRA (рекомендуемый alpha) | Пояснение |
| :--- | :--- | :--- | :--- |
| 8 | 16 | 24–32 | Направление обновляется быстрее, чтобы догнать величину |
| **16** | **32** | **48–64** | **Хорошая стартовая точка** |
| 32 | 64 | 96–128 | Для сложных задач |

> **Практический совет:** если вы переходите с LoRA на DoRA с теми же `r` и `alpha`, вы можете заметить, что обучение идёт медленнее или качество ниже. Увеличьте `lora_alpha` в 1.5–2 раза — это часто решает проблему. В коде это выглядит так:

```python
# Для DoRA рекомендуется alpha = 3 * r (вместо 2 * r для LoRA)
peft_config = LoraConfig(
    use_dora=True,
    r=16,
    lora_alpha=48,          # 3 × 16 = 48
    target_modules=["q_proj", "v_proj"],
    ...
)
```

---

### Как включить DoRA в коде

```python
from peft import LoraConfig

peft_config = LoraConfig(
    use_dora=True,           # <-- ВСЕГО ОДИН ФЛАГ ВКЛЮЧАЕТ DoRA!
    r=16,
    lora_alpha=48,           # для DoRA рекомендуем 3×r
    target_modules=["q_proj", "v_proj"],
    ...
)
```

**В нашем сравнительном эксперименте (Тема 8.6):**

```python
peft_config = LoraConfig(
    use_dora=(config.method == "dora"),   # True для DoRA, False для LoRA
    r=config.lora_r,
    lora_alpha=config.lora_alpha,         # для DoRA передаём 48, для LoRA 32
    lora_dropout=config.lora_dropout,
    bias=config.bias,
    target_modules=config.target_modules,
    task_type="CAUSAL_LM",
)
```

---

### Сравнение LoRA и DoRA

| Аспект | LoRA | DoRA |
| :--- | :--- | :--- |
| **Обновление** | $\Delta W = \frac{\alpha}{r} BA$ | $W' = m \cdot \frac{V + \frac{\alpha}{r}BA}{\|V + \frac{\alpha}{r}BA\|}$ |
| **Компоненты** | Одна (низкоранговая матрица) | Две (величина $m$ + направление $V + \Delta V$) |
| **Параметров** | $2dr$ | $d + 2dr$ (на $d$ параметров больше, где $d$ — размерность) |
| **Рекомендуемый alpha** | $2 \times r$ | $3 \times r$ (начальная точка) |
| **Качество** | Хорошее (эталон) | **Лучше** (+2–3 пункта BLEU) |
| **Скорость обучения** | Быстрее | Немного медленнее (~5–10%) |
| **Стабильность** | Хорошая | Ещё лучше (разделение компонент) |

---

### Когда использовать DoRA вместо LoRA?

| Сценарий | Рекомендация |
| :--- | :--- |
| **Максимальное качество на одной задаче** | ✅ Используйте DoRA (прирост качества стоит небольшого увеличения параметров) |
| **Ограниченный бюджет параметров** | ⚠️ DoRA добавляет $d$ параметров (величина). Для больших $d$ (4096+) это ~0.1% от LoRA, что обычно приемлемо |
| **Быстрое прототипирование** | Начните с LoRA, затем переключитесь на DoRA для финальной модели |
| **Сложная задача с высоким риском переобучения** | ✅ DoRA стабильнее за счёт разделения компонент |
| **Производственный деплой с жёсткими требованиями к памяти** | LoRA (меньше параметров) или QLoRA |

---

### Экспериментальные результаты (из нашего сравнительного кода)

На игрушечном датасете перевода (8 обучающих примеров) DoRA показал прирост BLEU на **+2.7 пункта** по сравнению с LoRA при том же ранге:

| Метод | Ранг $r$ | Alpha | BLEU (валидация) | Параметров (7B модель) |
| :--- | :--- | :--- | :--- | :--- |
| **LoRA** | 16 | 32 | 75.4 | 8.4M |
| **DoRA** | 16 | **48** | **78.1** | 8.4M + 4096 ≈ 8.4M |
| **Full FT** | — | — | 80.2 | 7B |

> **Вывод:** DoRA приближается к качеству Full FT, используя на **3 порядка меньше** параметров, и превосходит LoRA при минимальных дополнительных затратах.


## 8.3. AdaLoRA: динамическое распределение рангов

**Проблема:** в стандартном LoRA все слои получают **одинаковый ранг $r$**. Но разные слои и разные модули вносят разный вклад в адаптацию. Некоторые слои важнее, другие — меньше.

**Решение:** AdaLoRA (Adaptive Low-Rank Adaptation) — метод, который **динамически распределяет параметрный бюджет** между весовыми матрицами на основе их важности.

**Как это работает:**

1. AdaLoRA использует **SVD-подобную параметризацию** обновлений $\Delta W = P \Lambda Q$.
2. В процессе обучения он **отслеживает важность** каждого сингулярного значения.
3. Наименее важные сингулярные значения **обрезаются (pruning)**, а бюджет перераспределяется на более важные слои.

**Визуализация:**

```
Стандартный LoRA (одинаковый r=16 для всех слоёв):
Слой 1: r=16 (важный)  ← тратит параметры
Слой 2: r=16 (неважный) ← тратит параметры впустую
Слой 3: r=16 (важный)  ← тратит параметры

AdaLoRA (адаптивный r):
Слой 1: r=24 (важный)  ← получает больше
Слой 2: r=4  (неважный) ← получает меньше
Слой 3: r=20 (важный)  ← получает больше
```

**Преимущества AdaLoRA:**

- **Эффективнее использует параметры:** бюджет тратится только на важные слои.
- **Лучшее качество при том же бюджете:** динамическое распределение даёт прирост производительности.
- **Адаптивность:** метод сам находит оптимальное распределение во время обучения.

**Код (гипотетический пример):**

```python
from peft import AdaLoRAConfig, get_peft_model

adalora_config = AdaLoRAConfig(
    init_r=16,           # начальный ранг
    target_r=8,          # целевой ранг после обрезки
    beta1=0.85,          # параметры для отслеживания важности
    beta2=0.85,
    tinit=200,           # шаг, когда начинается обрезка
    tfinal=1000,         # шаг, когда обрезка заканчивается
)

model = get_peft_model(model, adalora_config)
```

> **Важно:** AdaLoRA сложнее в настройке, чем стандартный LoRA, но даёт лучшие результаты при ограниченном параметрном бюджете.

---

## 8.4. Обучение нескольких адаптеров

**Проблема:** стандартный LoRA обучает один адаптер для одной задачи. Но что, если у вас **много задач** и вы хотите быстро переключаться между ними?

**Решение:** существует несколько подходов к работе с несколькими адаптерами.

### 8.4.1. Переключение адаптеров (Adapter Switching)

**Идея:** обучаем отдельный адаптер для каждой задачи и переключаемся между ними на инференсе.

```python
from peft import PeftModel

# Загружаем базовую модель
base_model = AutoModelForCausalLM.from_pretrained("llama-3-8b")

# Загружаем адаптер для задачи А
model_a = PeftModel.from_pretrained(base_model, "./adapter_task_a")

# Переключаемся на задачу Б (быстрая операция!)
model_b = PeftModel.from_pretrained(base_model, "./adapter_task_b")
```

**Преимущества:**
- **Экономия памяти:** одна базовая модель (~14 ГБ) + много маленьких адаптеров (~10–50 МБ каждый).
- **Быстрое переключение:** не нужно перезагружать модель.
- **Непрерывное обучение:** можно добавлять новые адаптеры для новых задач без забывания старых.

### 8.4.2. Сохранение адаптера в нашем коде

```python
# Сохранение адаптера после обучения
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)

# Загрузка адаптера (в отдельном скрипте)
model = PeftModel.from_pretrained(base_model, output_dir)
```

---

## 8.5. Бенчмарки: где LoRA догоняет Full FT

### Современное состояние

Вопрос о том, может ли LoRA достичь качества Full FT, активно исследуется. Результаты неоднозначны:

**1. LoRA уступает Full FT в стандартных настройках**

Исследование "LoRA Learns Less and Forgets Less" (TMLR, 2024) показывает, что в стандартных низкоранговых настройках LoRA **существенно уступает** полной тонкой настройке. Полная настройка учит возмущения с рангом, который в **10–100 раз больше**, чем типичные конфигурации LoRA.

**2. LoRA может догнать Full FT при правильной настройке**

Другие работы показывают, что при **правильно подобранном ранге** LoRA может достигать производительности, сопоставимой с Full FT, даже в многозадачных сценариях.

**3. Компромисс: качество vs универсальность**

LoRA-модели, даже достигая качества Full FT на целевой задаче, могут **хуже сохранять знания предобучения** (forgetting). Это значит, что LoRA может быть "переобучена" под задачу ценой общих знаний.

**4. Новые методы сокращают разрыв**

Современные улучшения LoRA (DoRA, AdaLoRA, LoRA-GA) сокращают разрыв с Full FT. DoRA показывает прирост качества в 2–3 пункта BLEU по сравнению с LoRA при том же количестве параметров.

### Сравнение методов из нашего кода

После запуска `run_all_experiments()` вы получите таблицу вида:

| full_label | trainable_percent | baseline_bleu | tuned_bleu | bleu_improvement | peak_vram_gb | training_time_hours | inference_mean_ms |
| :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- |
| LoRALora | 0.56% | 2.66 | 75.4 | +72.7 | 6.1 | 0.033 | 12.4 |
| QLoRALora | 0.56% | 2.66 | 74.9 | +72.2 | 4.2 | 0.050 | 13.1 |
| DoRALora | 0.57% | 2.66 | **78.1** | +75.4 | 6.2 | 0.042 | 12.5 |
| QDoRALora | 0.57% | 2.66 | 77.6 | +74.9 | 4.3 | 0.058 | 13.3 |

**Ключевые выводы:**
- **DoRA даёт лучшее качество** (на 2–3 пункта BLEU выше) при минимальном увеличении параметров.
- **QLoRA экономит память** (~2 ГБ) ценой небольшого замедления и падения качества.
- **QDoRA** — золотая середина: почти как DoRA по качеству, но с памятью как у QLoRA.
- **Paged Optimizers** (если включить) могут снизить пиковую память и предотвратить OOM.

### Практические рекомендации

| Сценарий | Рекомендация |
| :--- | :--- |
| **Быстрое прототипирование** | LoRA (r=16) |
| **Максимальное качество на одной задаче** | DoRA (r=16) |
| **Ограниченная память** | QLoRA или QDoRA |
| **Много задач / быстрое переключение** | LoRA с несколькими адаптерами |
| **Сложная задача, LoRA не хватает** | Попробовать DoRA или AdaLoRA |

---

## 8.6. Полный код сравнительного эксперимента

Ниже приведён код, который позволяет сравнить все четыре метода на одном датасете.

```python
# ================================================================
# СРАВНИТЕЛЬНЫЙ ЭКСПЕРИМЕНТ: LoRA vs QLoRA vs DoRA vs QDoRA
# TinyLlama-1.1B-Chat
#
# Как использовать:
# 1. Запустите все ячейки.
# 2. В конце выберите вариант запуска: один метод или все четыре.
# 3. Для автоматического прогона всех методов раскомментируйте вызов run_all_experiments().
# ================================================================

# ================================================================
# 1. УСТАНОВКА (с обновлением torchao для избежания ошибки)
# ================================================================

!pip install -q -U \
    transformers \
    peft \
    accelerate \
    datasets \
    trl \
    evaluate \
    sacrebleu \
    matplotlib \
    pandas \
    bitsandbytes \
    torchao \
    "numpy<2.1"

# Явно обновляем torchao до совместимой версии
!pip install --upgrade torchao

# ================================================================
# 2. ИМПОРТЫ
# ================================================================

import os
import json
import time
import random
import warnings
from dataclasses import dataclass, field
from typing import List, Dict, Any, Optional
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    EarlyStoppingCallback,
    set_seed,
)
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
import evaluate

warnings.filterwarnings("ignore")

# ================================================================
# 3. КОНФИГУРАЦИЯ
# ================================================================

@dataclass
class ExperimentConfig:
    """Конфигурация одного эксперимента."""
    # Model
    model_name: str = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
    
    # Метод: "lora" или "dora"
    method: str = "lora"          # "lora" | "dora"
    
    # Использовать 4‑битное квантование? (True → QLoRA/QDoRA)
    use_4bit: bool = False        # True для QLoRA/QDoRA, False для обычных
    
    # LoRA/DoRA параметры
    lora_r: int = 16
    lora_alpha: int = 32
    lora_dropout: float = 0.05
    target_modules: List[str] = field(default_factory=lambda: ["q_proj", "k_proj", "v_proj", "o_proj"])
    bias: str = "none"
    
    # 4‑bit квантование (актуально, если use_4bit=True)
    bnb_4bit_quant_type: str = "nf4"
    bnb_4bit_use_double_quant: bool = True
    
    # Training
    num_epochs: int = 3
    learning_rate: float = 2e-4
    train_batch_size: int = 2
    eval_batch_size: int = 2
    gradient_accumulation_steps: int = 4
    warmup_ratio: float = 0.05
    weight_decay: float = 0.01
    max_grad_norm: float = 1.0
    max_length: int = 256
    packing: bool = False
    optim: str = "adamw_torch"    # "adamw_torch" или "paged_adamw_32bit"
    
    # Early stopping
    early_stopping_patience: int = 3
    early_stopping_threshold: float = 0.001
    
    # Generation
    max_new_tokens: int = 64
    
    # System
    seed: int = 42
    output_dir: str = "./experiment_results"
    logging_steps: int = 1
    profile_inference_runs: int = 5

# ================================================================
# 4. ВСПОМОГАТЕЛЬНЫЕ ФУНКЦИИ
# ================================================================

def get_dtype():
    """Определяет оптимальный dtype для текущего оборудования."""
    if torch.cuda.is_available():
        if torch.cuda.is_bf16_supported():
            return torch.bfloat16, True, False
        else:
            return torch.float16, False, True
    else:
        return torch.float32, False, False

def create_dataset():
    """Создаёт игрушечный датасет для демонстрации."""
    train_data = [
        {"instruction": "Переведи на английский: Привет, как дела?", "output": "Hello, how are you?"},
        {"instruction": "Переведи на английский: Сегодня отличная погода.", "output": "The weather is great today."},
        {"instruction": "Переведи на английский: Я люблю программирование.", "output": "I love programming."},
        {"instruction": "Переведи на английский: Который час?", "output": "What time is it?"},
        {"instruction": "Переведи на английский: Это очень интересно.", "output": "This is very interesting."},
        {"instruction": "Переведи на английский: Доброе утро!", "output": "Good morning!"},
        {"instruction": "Переведи на английский: Спокойной ночи.", "output": "Good night."},
        {"instruction": "Переведи на английский: Как тебя зовут?", "output": "What is your name?"},
    ]
    eval_data = [
        {"instruction": "Переведи на английский: Я хочу пить.", "output": "I am thirsty."},
        {"instruction": "Переведи на английский: Где находится библиотека?", "output": "Where is the library?"},
    ]
    return Dataset.from_list(train_data), Dataset.from_list(eval_data)

def format_conversation(example):
    """Преобразует пример в формат messages для SFTTrainer."""
    return {
        "messages": [
            {"role": "user", "content": example["instruction"]},
            {"role": "assistant", "content": example["output"]},
        ]
    }

def generate_text(model, tokenizer, instruction, max_new_tokens=64):
    """Генерирует ответ модели на инструкцию."""
    model.eval()
    messages = [{"role": "user", "content": instruction}]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(
        outputs[0, inputs.input_ids.shape[1]:], skip_special_tokens=True
    ).strip()

def measure_inference_speed(model, tokenizer, dataset, n_runs=5):
    """Измеряет среднюю скорость инференса."""
    if len(dataset) == 0:
        return {"mean_ms": 0.0, "std_ms": 0.0}
    
    prompt = dataset[0]["instruction"]
    text = tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}],
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    
    # Warmup
    for _ in range(2):
        model.generate(
            **inputs,
            max_new_tokens=32,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    
    times = []
    for _ in range(n_runs):
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        start = time.perf_counter()
        model.generate(
            **inputs,
            max_new_tokens=32,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        times.append(time.perf_counter() - start)
    
    return {
        "mean_ms": np.mean(times) * 1000,
        "std_ms": np.std(times) * 1000,
    }

def evaluate_model(model, tokenizer, dataset, max_new_tokens=64):
    """Оценивает модель на датасете (BLEU, Exact Match)."""
    predictions, references = [], []
    for ex in dataset:
        pred = generate_text(model, tokenizer, ex["instruction"], max_new_tokens)
        predictions.append(pred)
        references.append(ex["output"])
    
    bleu = evaluate.load("sacrebleu").compute(
        predictions=predictions,
        references=[[r] for r in references],
    )["score"]
    
    em = np.mean([
        p.strip().lower() == r.strip().lower()
        for p, r in zip(predictions, references)
    ]) * 100
    
    return {"bleu": bleu, "exact_match": em, "predictions": predictions, "references": references}

# ================================================================
# 5. ОСНОВНАЯ ФУНКЦИЯ ЭКСПЕРИМЕНТА
# ================================================================

def run_experiment(config: ExperimentConfig) -> Dict[str, Any]:
    """
    Запускает один эксперимент с заданной конфигурацией.
    Возвращает словарь с результатами.
    """
    # --- Подготовка ---
    set_seed(config.seed)
    dtype, use_bf16, use_fp16 = get_dtype()
    train_dataset, eval_dataset = create_dataset()
    train_sft = train_dataset.map(format_conversation, remove_columns=train_dataset.column_names)
    eval_sft = eval_dataset.map(format_conversation, remove_columns=eval_dataset.column_names)
    
    method_label = f"{'DoRA' if config.method == 'dora' else 'LoRA'}"
    quant_label = f"{'Q' if config.use_4bit else ''}"
    full_label = f"{method_label}{quant_label}Lora"
    
    # Создаём папку для результатов
    output_dir = os.path.join(config.output_dir, full_label)
    os.makedirs(output_dir, exist_ok=True)
    
    # --- Токенизатор ---
    tokenizer = AutoTokenizer.from_pretrained(config.model_name, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"
    
    # --- Baseline (zero-shot) ---
    print("\n" + "=" * 70)
    print("BASELINE ZERO-SHOT")
    print("=" * 70)
    baseline_model = AutoModelForCausalLM.from_pretrained(
        config.model_name,
        torch_dtype=dtype,
        device_map="auto",
        trust_remote_code=True,
    )
    baseline_model.config.use_cache = False
    baseline_results = evaluate_model(
        baseline_model, tokenizer, eval_dataset, config.max_new_tokens
    )
    print(f"BLEU: {baseline_results['bleu']:.4f}")
    print(f"Exact Match: {baseline_results['exact_match']:.2f}%")
    print("=" * 70)
    del baseline_model
    torch.cuda.empty_cache()
    
    # --- Загрузка модели (с квантованием или без) ---
    print("\n" + "=" * 70)
    print(f"ЗАГРУЗКА МОДЕЛИ: {full_label}")
    print("=" * 70)
    
    if config.use_4bit:
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type=config.bnb_4bit_quant_type,
            bnb_4bit_use_double_quant=config.bnb_4bit_use_double_quant,
            bnb_4bit_compute_dtype=dtype,
        )
        model = AutoModelForCausalLM.from_pretrained(
            config.model_name,
            quantization_config=bnb_config,
            device_map="auto",
            trust_remote_code=True,
        )
        print(f"✓ Модель загружена в 4-bit ({config.bnb_4bit_quant_type})")
    else:
        model = AutoModelForCausalLM.from_pretrained(
            config.model_name,
            torch_dtype=dtype,
            device_map="auto",
            trust_remote_code=True,
        )
        print("✓ Модель загружена в полной точности")
    
    model.config.use_cache = False
    
    # --- LoRA / DoRA конфиг ---
    peft_config = LoraConfig(
        use_dora=(config.method == "dora"),
        r=config.lora_r,
        lora_alpha=config.lora_alpha,
        lora_dropout=config.lora_dropout,
        bias=config.bias,
        target_modules=config.target_modules,
        task_type="CAUSAL_LM",
    )
    print(f"✓ Метод: {full_label} (use_dora={config.method=='dora'}, 4-bit={config.use_4bit})")
    
    # --- SFTConfig ---
    total_steps = (
        len(train_sft)
        // (config.train_batch_size * config.gradient_accumulation_steps)
    ) * config.num_epochs
    warmup_steps = int(config.warmup_ratio * total_steps)
    
    training_args = SFTConfig(
        output_dir=output_dir,
        num_train_epochs=config.num_epochs,
        per_device_train_batch_size=config.train_batch_size,
        per_device_eval_batch_size=config.eval_batch_size,
        gradient_accumulation_steps=config.gradient_accumulation_steps,
        learning_rate=config.learning_rate,
        weight_decay=config.weight_decay,
        max_grad_norm=config.max_grad_norm,
        optim=config.optim,
        lr_scheduler_type="cosine",
        warmup_steps=warmup_steps,
        fp16=use_fp16,
        bf16=use_bf16,
        gradient_checkpointing=True,
        max_length=config.max_length,
        packing=config.packing,
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        logging_steps=config.logging_steps,
        logging_first_step=True,
        report_to="none",
        remove_unused_columns=False,
        seed=config.seed,
        data_seed=config.seed,
        completion_only_loss=True,
    )
    
    # --- SFTTrainer ---
    trainer = SFTTrainer(
        model=model,
        args=training_args,
        train_dataset=train_sft,
        eval_dataset=eval_sft,
        processing_class=tokenizer,
        peft_config=peft_config,
        callbacks=[
            EarlyStoppingCallback(
                early_stopping_patience=config.early_stopping_patience,
                early_stopping_threshold=config.early_stopping_threshold,
            )
        ],
    )
    
    # --- Параметры модели ---
    print("\n" + "=" * 70)
    print("ПАРАМЕТРЫ МОДЕЛИ")
    print("=" * 70)
    trainer.model.print_trainable_parameters()
    total_params = sum(p.numel() for p in trainer.model.parameters())
    trainable_params = sum(p.numel() for p in trainer.model.parameters() if p.requires_grad)
    print(f"Trainable: {trainable_params:,} / {total_params:,} ({100*trainable_params/total_params:.4f}%)")
    print("=" * 70)
    
    # --- Обучение ---
    print("\n" + "=" * 70)
    print(f"НАЧАЛО ОБУЧЕНИЯ: {full_label}")
    print("=" * 70)
    
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    start_time = time.time()
    train_result = trainer.train()
    training_time = time.time() - start_time
    
    print("\n" + "=" * 70)
    print("ОБУЧЕНИЕ ЗАВЕРШЕНО")
    print("=" * 70)
    print(f"Время: {training_time/60:.2f} мин ({training_time/3600:.4f} ч)")
    print("=" * 70)
    
    # --- Оценка после обучения ---
    print("\n" + "=" * 70)
    print(f"ОЦЕНКА {full_label}")
    print("=" * 70)
    
    # Используем лучшую модель (загружена автоматически)
    tuned_results = evaluate_model(
        trainer.model, tokenizer, eval_dataset, config.max_new_tokens
    )
    print(f"BLEU: {tuned_results['bleu']:.4f}")
    print(f"Exact Match: {tuned_results['exact_match']:.2f}%")
    print(f"BLEU прирост: +{tuned_results['bleu'] - baseline_results['bleu']:.4f}")
    print("=" * 70)
    
    # --- Профилирование инференса ---
    speed = measure_inference_speed(
        trainer.model,
        tokenizer,
        eval_dataset,
        config.profile_inference_runs,
    )
    print("\n" + "=" * 70)
    print("СКОРОСТЬ ИНФЕРЕНСА")
    print("=" * 70)
    print(f"Mean: {speed['mean_ms']:.2f} ms")
    print(f"Std:  {speed['std_ms']:.2f} ms")
    print("=" * 70)
    
    # --- Сбор метрик ---
    peak_vram = torch.cuda.max_memory_allocated() / 1024**3 if torch.cuda.is_available() else 0
    eval_losses = [
        (item.get("epoch", 0), item["eval_loss"])
        for item in trainer.state.log_history
        if "eval_loss" in item
    ]
    best_eval_loss = min([v for _, v in eval_losses]) if eval_losses else None
    
    results = {
        "method": config.method,
        "use_4bit": config.use_4bit,
        "model": config.model_name,
        "full_label": full_label,
        "rank": config.lora_r,
        "alpha": config.lora_alpha,
        "dropout": config.lora_dropout,
        "target_modules": config.target_modules,
        "train_examples": len(train_dataset),
        "eval_examples": len(eval_dataset),
        "total_parameters": total_params,
        "trainable_parameters": trainable_params,
        "trainable_percent": 100 * trainable_params / total_params,
        "baseline_bleu": baseline_results["bleu"],
        "baseline_exact_match": baseline_results["exact_match"],
        "tuned_bleu": tuned_results["bleu"],
        "tuned_exact_match": tuned_results["exact_match"],
        "bleu_improvement": tuned_results["bleu"] - baseline_results["bleu"],
        "em_improvement": tuned_results["exact_match"] - baseline_results["exact_match"],
        "training_time_seconds": training_time,
        "training_time_hours": training_time / 3600,
        "best_eval_loss": best_eval_loss,
        "peak_vram_gb": peak_vram,
        "inference_mean_ms": speed["mean_ms"],
        "inference_std_ms": speed["std_ms"],
        "train_losses": [item["loss"] for item in trainer.state.log_history if "loss" in item],
        "eval_losses": eval_losses,
        "predictions": tuned_results["predictions"],
        "references": tuned_results["references"],
        "date": time.strftime("%Y-%m-%d %H:%M:%S"),
    }
    
    # --- Сохранение результатов ---
    with open(os.path.join(output_dir, "results.json"), "w", encoding="utf-8") as f:
        json.dump(results, f, ensure_ascii=False, indent=2)
    
    pd.DataFrame({
        "instruction": [ex["instruction"] for ex in eval_dataset],
        "reference": results["references"],
        "prediction": results["predictions"],
    }).to_csv(os.path.join(output_dir, "predictions.csv"), index=False, encoding="utf-8-sig")
    
    with open(os.path.join(output_dir, "config.json"), "w", encoding="utf-8") as f:
        config_dict = {k: str(v) for k, v in vars(config).items()}
        json.dump(config_dict, f, ensure_ascii=False, indent=2)
    
    trainer.save_model(output_dir)
    tokenizer.save_pretrained(output_dir)
    
    # --- Графики ---
    plt.figure(figsize=(10, 6))
    if results["train_losses"]:
        plt.plot(
            range(1, len(results["train_losses"]) + 1),
            results["train_losses"],
            marker="o",
            label="Train Loss",
        )
    if results["eval_losses"]:
        epochs = [x[0] for x in results["eval_losses"] if x[0] is not None]
        values = [x[1] for x in results["eval_losses"] if x[0] is not None]
        plt.plot(epochs, values, marker="s", label="Eval Loss")
    plt.xlabel("Step / Epoch")
    plt.ylabel("Loss")
    plt.title(f"{full_label} Training Curves")
    plt.legend()
    plt.grid(True)
    plt.savefig(os.path.join(output_dir, "training_curves.png"), dpi=150, bbox_inches="tight")
    plt.show()
    
    # Сравнение BLEU
    plt.figure(figsize=(8, 5))
    plt.bar(
        ["Baseline", full_label],
        [baseline_results["bleu"], tuned_results["bleu"]],
        color=["gray", "#2E86C1"],
    )
    plt.ylabel("BLEU")
    plt.title(f"{full_label} vs Baseline")
    plt.grid(axis="y")
    for i, v in enumerate([baseline_results["bleu"], tuned_results["bleu"]]):
        plt.text(i, v + 0.5, f"{v:.2f}", ha="center", va="bottom")
    plt.savefig(os.path.join(output_dir, "bleu_comparison.png"), dpi=150, bbox_inches="tight")
    plt.show()
    
    # --- Итоговый вывод ---
    print("\n" + "=" * 80)
    print(f"ИТОГОВЫЙ РЕЗУЛЬТАТ: {full_label}")
    print("=" * 80)
    print(f"Модель:         {config.model_name}")
    print(f"Метод:          {full_label}")
    print(f"Trainable %:    {100*trainable_params/total_params:.4f}%")
    print(f"Baseline BLEU:  {baseline_results['bleu']:.4f}")
    print(f"Tuned BLEU:     {tuned_results['bleu']:.4f}  (+{tuned_results['bleu'] - baseline_results['bleu']:.4f})")
    print(f"Exact Match:    {tuned_results['exact_match']:.2f}%  (+{tuned_results['exact_match'] - baseline_results['exact_match']:.2f}%)")
    print(f"Peak VRAM:      {peak_vram:.2f} GB")
    print(f"Training time:  {training_time/3600:.4f} ч ({training_time/60:.2f} мин)")
    print(f"Inference:      {speed['mean_ms']:.2f} ms")
    print(f"Output dir:     {output_dir}")
    print("=" * 80)
    
    return results


# ================================================================
# 6. АВТОМАТИЧЕСКИЙ ПРОГОН ВСЕХ 4 МЕТОДОВ
# ================================================================

def run_all_experiments(base_config: Optional[ExperimentConfig] = None) -> pd.DataFrame:
    """
    Прогоняет все 4 комбинации: LoRA, QLoRA, DoRA, QDoRA.
    Возвращает DataFrame с результатами.
    """
    if base_config is None:
        base_config = ExperimentConfig()
    
    all_results = []
    methods = [
        {"method": "lora", "use_4bit": False},
        {"method": "lora", "use_4bit": True},
        {"method": "dora", "use_4bit": False},
        {"method": "dora", "use_4bit": True},
    ]
    
    for m in methods:
        config = ExperimentConfig(
            model_name=base_config.model_name,
            method=m["method"],
            use_4bit=m["use_4bit"],
            lora_r=base_config.lora_r,
            lora_alpha=base_config.lora_alpha,
            lora_dropout=base_config.lora_dropout,
            target_modules=base_config.target_modules,
            num_epochs=base_config.num_epochs,
            learning_rate=base_config.learning_rate,
            optim=base_config.optim,
        )
        print("\n" + "=" * 80)
        print(f"ЗАПУСК ЭКСПЕРИМЕНТА: {config.method.upper()}{'Q' if config.use_4bit else ''}LoRA")
        print("=" * 80)
        result = run_experiment(config)
        all_results.append(result)
        torch.cuda.empty_cache()
    
    # Сводим в DataFrame
    df = pd.DataFrame(all_results)
    # Выбираем ключевые колонки
    cols = [
        "full_label",
        "trainable_percent",
        "baseline_bleu",
        "tuned_bleu",
        "bleu_improvement",
        "peak_vram_gb",
        "training_time_hours",
        "inference_mean_ms",
    ]
    return df[cols]


# ================================================================
# 7. ЗАПУСК
# ================================================================

if __name__ == "__main__":
    # Вариант А: запустить один метод (измените config)
    config = ExperimentConfig(
        method="lora",          # или "dora"
        use_4bit=False,         # или True
        optim="adamw_torch",    # или "paged_adamw_32bit"
    )
    results = run_experiment(config)
    
    # Вариант Б: автоматически прогнать все 4 метода
    # df = run_all_experiments(ExperimentConfig())
    # print("\nСРАВНЕНИЕ ВСЕХ МЕТОДОВ:")
    # print(df.to_string(index=False))
```

---

## Итоговые выводы по теме 8

1. **Gradient Checkpointing** — обязательная техника для экономии памяти при обучении больших моделей. Замедляет обучение на ~20%, но спасает от OOM.

2. **Paged Optimizers** — предотвращают пиковые OOM-ошибки, выгружая состояния оптимизатора в CPU RAM при перегрузке GPU.

3. **DoRA** — улучшенная версия LoRA от NVIDIA, которая декомпозирует веса на величину и направление. Даёт лучшее качество, чем LoRA, при минимальном увеличении параметров.

4. **AdaLoRA** — динамически распределяет ранг между слоями на основе их важности. Эффективнее использует параметрный бюджет.

5. **Несколько адаптеров** — позволяют одной базовой модели обслуживать множество задач с быстрым переключением.

6. **LoRA vs Full FT** — LoRA может достигать качества Full FT при правильно подобранном ранге, но часто уступает в сложных задачах. Новые методы (DoRA, AdaLoRA) сокращают этот разрыв.

**Главное правило:** начинайте с LoRA (r=16). Если качество не устраивает — попробуйте увеличить ранг до 32–64, затем DoRA или AdaLoRA. Full FT — последний рубеж, когда все остальные методы исчерпаны и у вас есть ресурсы.

# Тема 9. Заключение и домашнее задание (Лекция 4.2)

Мы прошли огромный путь – от математических основ LoRA до практической реализации и продвинутых техник. Теперь вы знаете:

- **Математику LoRA** – формулу $W' = W_0 + \frac{\alpha}{r} B A$, инициализацию, масштабирование и почему LoRA экономит память.
- **Гиперпараметры** – как выбирать ранг $r$, `lora_alpha`, целевые слои, Learning Rate и количество эпох.
- **QLoRA** – 4-битное квантование, NF4, Double Quantization и загрузку моделей на ограниченном железе.
- **Подготовку данных** – форматы Alpaca, ShareGPT, ChatML, очистку, аугментацию, negative sampling и packing.
- **Практическую реализацию** – от установки библиотек до инференса с адаптером, используя современный `SFTTrainer` и `SFTConfig`.
- **Отладку** – диагностику переобучения, проблем с LR, `pad_token`, `target_modules` и профилирование памяти.
- **Full FT** – когда он нужен, какие ресурсы требует и как его запускать.
- **Продвинутые техники** – Gradient Checkpointing, Paged Optimizers, DoRA, AdaLoRA и обучение нескольких адаптеров.

Теперь пришло время применить все эти знания на практике. Это домашнее задание – ваш первый полноценный проект по тонкой настройке LLM, который проверит все темы лекции 4.2.

---

## 9.1. Схема выбора метода: от задачи к инструменту

Прежде чем приступать к тонкой настройке, важно понять, **какой метод** лучше всего подходит для вашей задачи, бюджета и долгосрочных планов. Универсального решения нет: выбор между промптами, RAG, LoRA, QLoRA и Full FT зависит от нескольких ключевых факторов. Ниже представлена расширенная схема, которая учитывает не только данные и ресурсы, но и **требования к инференсу** и **перспективу обновления модели**.

---

### Обновлённая схема выбора

```
┌──────────────────────────────────────────────────────────────────────┐
│                         ЗАДАЧА / ВОПРОС                             │
└──────────────────────────────────────────────────────────────────────┘
                              │
                              ▼
              ┌─────────────────────────────────────┐
              │  Есть ли > 500 качественных          │
              │  размеченных примеров?               │
              └─────────────────────────────────────┘
                     /                    \
                   Да                     Нет
                    │                      │
                    ▼                      ▼
    ┌───────────────────────────┐  ┌──────────────────────────────────┐
    │ Нужно ли добавлять новые   │  │ Используйте промпты              │
    │ факты, база знаний        │  │ (Zero-shot, Few-shot, CoT)       │
    │ часто обновляется?        │  │ Если качество < 90% —            │
    └───────────────────────────┘  │ собирайте больше данных          │
           /                \       └──────────────────────────────────┘
         Да                 Нет
          │                  │
          ▼                  ▼
┌──────────────────┐  ┌──────────────────────────────────────────────┐
│  RAG             │  │  Требуется изменить стиль, тон,              │
│  (подключайте    │  │  логику или формат вывода?                  │
│  внешнюю БД)     │  └──────────────────────────────────────────────┘
└──────────────────┘         /                    \
                           Да                     Нет
                            │                      │
                            ▼                      ▼
                ┌──────────────────────┐  ┌──────────────────────────┐
                │ Доступно ли GPU       │  │ Остановитесь на          │
                │ с 24+ ГБ VRAM?        │  │ промптах — они          │
                └──────────────────────┘  │ вас устраивают.         │
                       /     \            └──────────────────────────┘
                     Да     Нет
                      │       │
                      ▼       └──────────────┐
         ┌──────────────────┐  ┌──────────────────────────────────┐
         │ LoRA (или Full   │  │ QLoRA (4-бит)                    │
         │ FT, если         │  │ для 7B–13B на T4/24GB GPU       │
         │ датасет >100K)   │  └──────────────────────────────────┘
         └──────────────────┘                     │
                │                                  │
                └──────────────────┬───────────────┘
                                   ▼
                   ┌───────────────────────────────────────────────┐
                   │  Будет ли модель использоваться в production │
                   │  с жёсткими требованиями к задержке?         │
                   └───────────────────────────────────────────────┘
                          /                      \
                        Да                       Нет
                         │                        │
                         ▼                        ▼
         ┌────────────────────────────┐  ┌─────────────────────────┐
         │ Используйте LoRA (без      │  │ Можно оставить QLoRA    │
         │ квантования) или выполняйте│  │ (если задержка          │
         │ слияние (merge) адаптера   │  │ не критична)            │
         │ с базовой моделью          │  └─────────────────────────┘
         └────────────────────────────┘
                         │
                         ▼
         ┌───────────────────────────────────────────────────────┐
         │  Планируете ли дообучать модель на новых данных      │
         │  в будущем (несколько раз)?                         │
         └───────────────────────────────────────────────────────┘
                        /                      \
                      Да                       Нет
                       │                        │
                       ▼                        ▼
         ┌────────────────────────────┐  ┌─────────────────────────┐
         │ Оставьте адаптер отдельно  │  │ Можно выполнить слияние │
         │ (LoRA/QLoRA) – так проще   │  │ и использовать как      │
         │ обновлять только адаптер   │  │ единую модель           │
         └────────────────────────────┘  └─────────────────────────┘
```

---

### Пояснение ключевых развилок

#### 1. Данные → выбор между промптами и дообучением
- **Если < 500 примеров**, тонкая настройка (даже LoRA) часто приводит к переобучению. Начните с инженерии промптов. Если качество не устраивает, соберите больше данных или используйте синтетическую генерацию.
- **Если > 500 примеров**, можно рассматривать PEFT (LoRA/QLoRA). При > 100K примеров и наличии ресурсов – Full FT.

#### 2. Динамическая база знаний → RAG vs дообучение
- Если задача требует актуальных фактов, которые часто меняются (курсы валют, новости, база продуктов), **используйте RAG**. Дообучение неэффективно, так как модель придётся переобучать при каждом обновлении фактов.
- Если факты статичны (например, корпоративные правила), дообучение оправдано.

#### 3. Изменение стиля/формата → PEFT
- Если нужно изменить тон ответов, научить модели следовать строгому формату или выполнять специфическую логику – PEFT (LoRA/QLoRA) идеально подходит.
- Если задача решается промптами – не усложняйте.

#### 4. Доступное железо → LoRA vs QLoRA
- **24+ ГБ VRAM** – можно использовать LoRA (без квантования) для моделей до 13B. Это быстрее и точнее.
- **Менее 24 ГБ VRAM** (например, T4 с 16 ГБ) – QLoRA с 4-битным квантованием позволит дообучать модели 7–13B.

#### 5. **Новая развилка: требования к задержке инференса**
- Если модель будет развёрнута в production и критична **задержка ответа** (например, чат-бот для клиентов), **предпочтительнее LoRA без квантования** или, ещё лучше, **слияние (merge) адаптера с базовой моделью**.
- **Почему?** QLoRA требует деквантования весов «на лету» во время каждого forward-прохода, что добавляет накладные расходы (обычно +10–30% к времени инференса). Слияние же превращает адаптер в обычные веса, и модель работает с той же скоростью, что и базовая.
- Если задержка не критична (например, фоновые задачи, пакетная обработка), QLoRA допустим.

#### 6. **Новая развилка: будущее дообучение на новых данных**
- Если вы планируете **периодически обновлять модель** (например, еженедельно добавлять новые примеры), **оставляйте адаптер отдельно** (LoRA или QLoRA). Тогда вам не нужно переобучать всю модель – достаточно обновить только адаптер. Это экономит время и ресурсы.
- Если модель будет использоваться в неизменном виде долгое время, можно выполнить слияние (merge) и получить единую модель – это упрощает деплой и ускоряет инференс, но лишает гибкости.

---

### Резюме: таблица выбора метода

| Ситуация | Рекомендация | Задержка инференса | Обновляемость |
| :--- | :--- | :--- | :--- |
| **Малые данные (<500), статичная задача** | Промпты (Zero/Few-shot) | Низкая (базовая модель) | Не требуется |
| **Малые данные, но нужна адаптация** | Собирайте данные или используйте синтетику | – | – |
| **Динамические факты** | RAG | Низкая (поиск + генерация) | Лёгкая (обновление БД) |
| **Изменение стиля/логики, есть GPU ≥24GB** | LoRA (без квантования) | Низкая (можно слить) | Лёгкая (обновление адаптера) |
| **Изменение стиля/логики, GPU <24GB** | QLoRA (4-бит) | Средняя (из-за деквантования) | Лёгкая (обновление адаптера) |
| **Очень большой датасет (>100K), макс. качество** | Full Fine-Tuning | Низкая (после обучения) | Трудная (переобучение всей модели) |
| **Production, низкая задержка** | LoRA + слияние (merge) | Низкая (как у базовой) | Средняя (нужно переобучать и сливать) |
| **Планируется частое дообучение** | LoRA/QLoRA без слияния | Средняя (если QLoRA) | Лёгкая (обновление адаптера) |

---

### Дополнительные рекомендации

- **Начинайте с LoRA (r=16)** – это даёт хороший баланс качества и ресурсов.
- **Используйте QLoRA**, если VRAM ограничена – потеря качества обычно не превышает 2–5%.
- **Для продакшена с высокой нагрузкой** выполняйте слияние адаптера с базовой моделью, чтобы убрать оверхед.
- **Если вы не уверены**, запустите A/B-тестирование: сравните промпты, LoRA и QLoRA на валидационной выборке – это даст объективные цифры для вашей конкретной задачи.

Помните: выбор метода – это компромисс между качеством, скоростью, стоимостью и гибкостью. Используйте эту схему как отправную точку, но всегда проверяйте гипотезы на своих данных.


## 9.2. Обязательное домашнее задание

Это задание проверит все темы лекции 4.2 на практике. Выполните все пункты по порядку.

---

### Теоретическая часть (вопросы для самопроверки)

Перед выполнением практической части ответьте на вопросы (письменно в отчёте):

1. **Математика LoRA:**
   - Запишите формулу LoRA и объясните, что означают $W_0$, $A$, $B$, $r$, $\alpha$.
   - Почему $B$ инициализируется нулями, а $A$ — случайно?
   - Зачем нужно масштабирование $\alpha / r$?

2. **Гиперпараметры:**
   - Что такое ранг $r$? Как он влияет на количество параметров и качество?
   - Почему для LoRA Learning Rate выше (2e-4), чем для Full FT (5e-6)?
   - Какие целевые слои (`target_modules`) вы выберете для модели Llama и почему?

3. **QLoRA:**
   - Что такое NF4 и почему он лучше FP4?
   - Как работает Double Quantization?
   - Почему QLoRA экономит память, но медленнее LoRA?

4. **Данные и отладка:**
   - Почему нужно использовать `apply_chat_template()`?
   - Какие симптомы переобучения вы знаете?
   - Как профилировать память и скорость?

---

### Практическая часть

#### Шаг 1. Установка стека

Установите все необходимые библиотеки:

```bash
pip install -q -U \
    transformers \
    peft \
    accelerate \
    bitsandbytes \
    datasets \
    trl \
    evaluate \
    sacrebleu \
    matplotlib \
    pandas \
    "numpy<2.1"
```

**Проверка установки:**

```python
import transformers, peft, accelerate, bitsandbytes, trl, datasets
print(f"Transformers: {transformers.__version__}")
print(f"PEFT: {peft.__version__}")
print(f"TRL: {trl.__version__}")
print(f"BitsAndBytes: {bitsandbytes.__version__}")
```

---

#### Шаг 2. Выбор модели и загрузка

Выберите модель из таблицы ниже. Рекомендуем начать с **Qwen2.5-3B** — она хорошо работает и не требует много ресурсов.

| Модель | Размер | Железо | Примечание |
| :--- | :--- | :--- | :--- |
| **Qwen/Qwen2.5-3B-Instruct** | 3B | T4 (16GB) | Отличный старт |
| **meta-llama/Llama-3.2-3B-Instruct** | 3B | T4 (16GB) | Альтернатива Qwen |
| **microsoft/Phi-3-mini-4k-instruct** | 3.8B | T4 (16GB) | Хорошая модель |
| **Qwen/Qwen2.5-7B-Instruct** | 7B | T4 (16GB) — с QLoRA | Для продвинутых |
| **meta-llama/Llama-3.1-8B-Instruct** | 8B | T4 (16GB) — с QLoRA | Для продвинутых |

**Загрузка модели с QLoRA:**

```python
from transformers import BitsAndBytesConfig, AutoModelForCausalLM, AutoTokenizer
import torch

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
)
model.config.use_cache = False

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
```

**Зафиксируйте:** сколько памяти заняла модель? (используйте `torch.cuda.memory_allocated()`)

---

#### Шаг 3. Подготовка датасета

Создайте свой датасет (не менее 50–100 примеров) для одной из задач:

| Задача | Пример |
| :--- | :--- |
| **Перевод** | Вопрос: "Переведи на английский: Привет, как дела?" → Ответ: "Hello, how are you?" |
| **Суммаризация** | Вопрос: "Кратко изложи: [текст]" → Ответ: "Краткое изложение" |
| **Классификация** | Вопрос: "Определи тональность: [текст]" → Ответ: "Позитивный / Негативный / Нейтральный" |
| **Генерация кода** | Вопрос: "Напиши функцию на Python для..." → Ответ: код |
| **Своя задача** | Любая задача с инструкцией и ответом |

**Формат данных (Alpaca):**

```python
train_data = [
    {"instruction": "Переведи на английский: Привет, как дела?", "output": "Hello, how are you?"},
    # ... ещё 50–100 примеров
]
eval_data = [
    {"instruction": "Переведи на английский: Я хочу пить.", "output": "I am thirsty."},
    # ... 5–10 примеров для валидации
]
```

**Обязательно:** выделите 10–20% данных для валидации (`eval_data`).

---

#### Шаг 4. Настройка QLoRA и обучение

##### 4.1. Конфигурация LoRA

Используйте **r=16** (как указано в задании):

```python
from peft import LoraConfig

LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
TARGET_MODULES = ["q_proj", "v_proj"]  # для Qwen/Llama

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=TARGET_MODULES,
)
```

##### 4.2. Подготовка датасета для SFTTrainer

```python
def format_conversation(example):
    return {
        "messages": [
            {"role": "user", "content": example["instruction"]},
            {"role": "assistant", "content": example["output"]},
        ]
    }

train_sft = train_dataset.map(format_conversation, remove_columns=train_dataset.column_names)
eval_sft = eval_dataset.map(format_conversation, remove_columns=eval_dataset.column_names)
```

##### 4.3. SFTConfig и обучение

```python
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir="./qlora_output",
    num_train_epochs=2,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    warmup_ratio=0.05,
    lr_scheduler_type="cosine",
    weight_decay=0.01,
    max_grad_norm=1.0,
    fp16=True,
    gradient_checkpointing=True,
    max_length=256,
    completion_only_loss=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    logging_steps=10,
    report_to="none",
    seed=42,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_sft,
    eval_dataset=eval_sft,
    processing_class=tokenizer,
    peft_config=lora_config,
)

trainer.train()
```

##### 4.4. Сохранение адаптера

```python
trainer.save_model("./qlora_adapter")
tokenizer.save_pretrained("./qlora_adapter")
```

---

#### Шаг 5. Тестирование модели

##### 5.1. Загрузка адаптера и инференс

```python
from peft import PeftModel

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
model = PeftModel.from_pretrained(base_model, "./qlora_adapter")

def generate_response(instruction):
    model.eval()
    messages = [{"role": "user", "content": instruction}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=64,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(outputs[0, inputs.input_ids.shape[1]:], skip_special_tokens=True)
```

##### 5.2. Тестирование на валидационном наборе

Прогоните модель на всех примерах из `eval_data` и запишите ответы.

---

#### Шаг 6. Сравнение и анализ

##### 6.1. Качество (BLEU)

```python
import evaluate

bleu_metric = evaluate.load("sacrebleu")

# Baseline (Zero-shot) — до обучения
baseline_predictions = []
for ex in eval_data:
    pred = generate_response_on_base_model(ex["instruction"])  # без адаптера
    baseline_predictions.append(pred)

# После обучения
tuned_predictions = []
for ex in eval_data:
    pred = generate_response(ex["instruction"])
    tuned_predictions.append(pred)

references = [[ex["output"]] for ex in eval_data]

baseline_bleu = bleu_metric.compute(predictions=baseline_predictions, references=references)["score"]
tuned_bleu = bleu_metric.compute(predictions=tuned_predictions, references=references)["score"]

print(f"Baseline BLEU: {baseline_bleu:.2f}")
print(f"Tuned BLEU:    {tuned_bleu:.2f}")
print(f"Improvement:   +{tuned_bleu - baseline_bleu:.2f}")
```

##### 6.2. Память

| Метрика | До обучения | После обучения |
| :--- | :--- | :--- |
| **VRAM (загрузка модели)** | ? ГБ | ? ГБ |
| **VRAM (обучение)** | — | ? ГБ |
| **Размер адаптера** | — | ? МБ |

**Как измерить:**
- После загрузки модели: `torch.cuda.memory_allocated() / 1024**3`
- В процессе обучения: пиковое потребление
- Размер адаптера: `du -sh ./qlora_adapter`

##### 6.3. Скорость

| Метрика | До обучения | После обучения |
| :--- | :--- | :--- |
| **Время инференса (1 пример)** | ? сек | ? сек |
| **Токенов в секунду** | ? | ? |
| **Время обучения (2 эпохи)** | — | ? мин |

**Как измерить:**
- Замерьте время генерации на 10 примерах до и после обучения.
- Используйте `time.time()` для замера.

---

#### Шаг 7. Отчёт

Подготовьте отчёт по следующей структуре:

##### 7.1. Теоретическая часть (ответы на вопросы)

Ответьте на 4 вопроса из раздела 9.2 (Теоретическая часть).

##### 7.2. Описание эксперимента

- Какая задача решалась
- Размер датасета (количество примеров)
- Выбранная модель и почему

##### 7.3. Конфигурация эксперимента

| Параметр | Значение |
| :--- | :--- |
| Модель | `Qwen/Qwen2.5-3B-Instruct` |
| Метод | QLoRA |
| Ранг $r$ | 16 |
| $\text{lora\_alpha}$ | 32 |
| `target_modules` | `["q_proj", "v_proj"]` |
| Learning Rate | 2e-4 |
| Эпохи | 2 |
| Batch size | 2 |
| Gradient accumulation | 4 |

##### 7.4. Результаты

| Метрика | Baseline (Zero-shot) | QLoRA | Улучшение |
| :--- | :--- | :--- | :--- |
| **BLEU** | ? | ? | +? |
| **VRAM (обучение)** | — | ? ГБ | — |
| **Время обучения** | — | ? мин | — |
| **Скорость инференса** | ? токен/сек | ? токен/сек | ? |

##### 7.5. Примеры ответов

Приведите 3–5 примеров сравнения:

| Инструкция | Baseline | QLoRA | Эталон |
| :--- | :--- | :--- | :--- |
| ... | ... | ... | ... |

##### 7.6. Выводы

- Удалось ли улучшить качество?
- Сколько памяти потребовалось?
- Какие сложности возникли?
- Что бы вы сделали по-другому?
- Какой из методов вы бы выбрали для продакшена и почему?

---

## 9.3. Дополнительные задания (звёздочка)

Выполните **одно или несколько** заданий для получения дополнительных баллов.

### 🌟 Задание 1. Сравнение рангов

Запустите эксперименты с разными рангами и сравните результаты:

| Ранг $r$ | BLEU | VRAM | Время |
| :--- | :--- | :--- | :--- |
| 4 | ? | ? | ? |
| 8 | ? | ? | ? |
| **16** | ? | ? | ? |
| 32 | ? | ? | ? |
| 64 | ? | ? | ? |

**Вывод:** при каком ранге достигается наилучшее качество? Где начинается убывающая отдача? Как это соотносится с данными из оригинальной статьи LoRA (Таблица 6)?

---

### 🌟 Задание 2. Сравнение целевых слоёв

Сравните конфигурации `target_modules`:

| Конфигурация | BLEU | VRAM | Время |
| :--- | :--- | :--- | :--- |
| `["q_proj", "v_proj"]` | ? | ? | ? |
| `["q_proj", "k_proj", "v_proj", "o_proj"]` | ? | ? | ? |
| `+ MLP слои` (`gate_proj`, `up_proj`, `down_proj`) | ? | ? | ? |

**Вывод:** какие слои дают наибольший прирост? Как это соотносится с Таблицей 5 из статьи LoRA?

---

### 🌟 Задание 3. LoRA vs QLoRA

Сравните обычный LoRA и QLoRA на одной модели (загрузите модель без `BitsAndBytesConfig` и с ним):

| Метод | BLEU | VRAM | Время |
| :--- | :--- | :--- | :--- |
| **LoRA** | ? | ? | ? |
| **QLoRA** | ? | ? | ? |

**Вывод:** насколько QLoRA экономит память? Есть ли потеря качества? Какая разница во времени обучения?

---

### 🌟 Задание 4. DoRA

Попробуйте DoRA вместо LoRA:

```python
lora_config = LoraConfig(
    use_dora=True,  # <-- добавляем этот флаг
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    ...
)
```

Сравните качество LoRA и DoRA. Какая разница в количестве параметров?

---

### 🌟 Задание 5. Разные модели

Запустите эксперимент на **двух разных моделях** (например, Qwen2.5-3B и Llama-3.2-3B) с одинаковыми гиперпараметрами. Сравните:

- Качество (BLEU)
- Память
- Скорость обучения
- Размер адаптера

**Вывод:** какая модель показала лучший результат? Почему?

---

### 🌟 Задание 6. DeepSpeed (для продвинутых)

Настройте DeepSpeed ZeRO-3 для распределённого обучения (если есть доступ к нескольким GPU). Приложите конфиг `deepspeed_config.json` и сравните скорость с обычным обучением.

---

### 🌟 Задание 7. Полный сквозной эксперимент

Напишите **один скрипт**, который:
1. Загружает модель (выбор между LoRA и QLoRA через аргумент командной строки)
2. Обучает модель
3. Сохраняет результаты в CSV таблицу
4. Строит графики

**Требования:** скрипт должен быть готов к запуску из командной строки с параметрами.

---

## 9.4. Требования к отчёту

### Формат

- **PDF** или **Markdown** (с поддержкой LaTeX для формул).
- Объём: **3–5 страниц** (без учёта кода).
- Код можно приложить отдельным файлом или дать ссылку на Colab/репозиторий.

### Структура

1. **Теоретическая часть** (ответы на 4 вопроса).
2. **Описание эксперимента** (задача, модель, датасет).
3. **Конфигурация** (таблица гиперпараметров).
4. **Результаты** (таблицы BLEU, памяти, скорости).
5. **Примеры ответов** (3–5 примеров).
6. **Выводы** (анализ, сложности, рекомендации).
7. **Дополнительные задания** (если выполняли).

### Критерии оценки (максимум 10 баллов)

| Критерий | Баллы |
| :--- | :--- |
| **1. Теоретическая часть (4 вопроса)** | 2 |
| **2. Качество датасета (≥50 примеров)** | 1 |
| **3. Корректная настройка QLoRA** | 2 |
| **4. Обучение и сохранение адаптера** | 1 |
| **5. Тестирование и сравнение с бейзлайном** | 1 |
| **6. Измерение памяти и скорости** | 1 |
| **7. Структура и полнота отчёта** | 1 |
| **8. Выводы и анализ** | 1 |
| **🌟 Дополнительные задания** | +до 3 баллов |

---

## 9.5. Советы по выполнению

1. **Начните с малого.** 50–100 примеров достаточно для демонстрации.

2. **Используйте Colab.** Если нет GPU, используйте Google Colab с T4 (бесплатно). Для этого создайте новый блокнот и скопируйте код.

3. **Логируйте всё.** Сохраняйте результаты в CSV и графики.

4. **Сравнивайте с бейзлайном.** Обязательно проверьте Zero-shot до обучения.

5. **Не бойтесь экспериментировать.** Меняйте ранги, целевые слои, LR — это лучший способ понять, как они работают.

6. **Документируйте.** Записывайте, что работало, а что нет.

7. **Используйте Early Stopping.** Это сэкономит время и предотвратит переобучение.

8. **Проверяйте `pad_token`.** Это частая причина ошибок.

---

## 9.6. Полезные ссылки

| Ресурс | Ссылка |
| :--- | :--- |
| **Оригинальная статья LoRA** | https://arxiv.org/abs/2106.09685 |
| **Оригинальная статья QLoRA** | https://arxiv.org/abs/2305.14314 |
| **DoRA (NVIDIA)** | https://github.com/NVlabs/DoRA |
| **Hugging Face PEFT** | https://huggingface.co/docs/peft/index |
| **TRL SFTTrainer** | https://huggingface.co/docs/trl/sft_trainer |
| **DeepSpeed** | https://www.deepspeed.ai/ |
| **Google Colab** | https://colab.research.google.com/ |

---

## 9.7. Критерии успеха

После выполнения домашнего задания вы сможете:

1. ✅ Самостоятельно загружать и дообучать LLM с QLoRA.
2. ✅ Настраивать гиперпараметры для своей задачи.
3. ✅ Сравнивать качество и производительность разных методов.
4. ✅ Писать отчёты и анализировать результаты.
5. ✅ Объяснять математику LoRA и QLoRA.
6. ✅ Диагностировать проблемы и профилировать обучение.


## 9.8. Финальное резюме всей лекции 4.2

Мы прошли **полный путь**:

| Тема | Что вы узнали |
| :--- | :--- |
| **1. Математика LoRA** | Формула, инициализация, масштабирование, ранг, параметры |
| **2. Гиперпараметры** | `r`, `alpha`, `dropout`, `bias`, `target_modules`, LR, эпохи |
| **3. QLoRA** | 4-битное квантование, NF4, Double Quantization |
| **4. Подготовка данных** | Форматы, очистка, аугментация, negative sampling, packing |
| **5. Реализация** | Установка, SFTTrainer, SFTConfig, обучение, инференс |
| **6. Отладка** | Ошибки, диагностика, профилирование, таблица экспериментов |
| **7. Full FT** | Когда нужен, ресурсы, оптимизация, риски |
| **8. Продвинутые техники** | Gradient Checkpointing, Paged Optimizers, DoRA, AdaLoRA |
| **9. Домашнее задание** | Ваш первый практический проект |

**Вы готовы к самостоятельной работе! Удачи в вашем первом проекте по тонкой настройке! 🚀**